# Arm 5 — Dense reference at K = 3

## The question

Can a 4-mode network represent this wake at all? Trains on 5,000 scattered u,v,p points. E1 reached 0.9946 but at Nmodes 3, a different architecture; this is the matched ceiling.

## What differs from Arm 1 (baseline)

    

Everything else is held at the K3 series settings: `--Tmax 9`, `--Nint 50000`, `--Nmes 5000`,
`--multigrid --Ngrid 5 --NgridTurn 200`, `--WidthLayer 25 --Nmodes 4`, `--Seed 0`, `--FreestreamBC`,
cold start, Adam kept.

## Runtime

About 9 hours on a T4. Weights are written to Drive before any evaluation runs.


---

## Why this is a retry, and what changed

The first attempt was killed by this notebook's own 10.5 h wall-clock guard at
**iterate 38,672 / 40,800 evaluations / 10.48 h**, still inside L-BFGS. Nothing usable
survived: the weight save runs after L-BFGS *and* Adam, so a mid-optimisation kill
writes no pickle, and the copy to Drive happens after that.

**This was foreseeable and was mis-ranked.** The right counterpart is E1, the project's
only dense run, which spent **9.76 h** in L-BFGS at the *smaller* truncation - 0.74 h
under the guard before adding a mode and 1.76x the parameters. Dense training is the
most expensive configuration in the set, not the cheapest.

**It was also nearly finished.** At the kill the loss was 2.943e-04, which is *below*
E1's converged 9.492e-04, and descent had almost stopped (8e-09 over four iterates).
The optimisation was not struggling; it simply needed more clock than the guard allowed.

**Three changes:**

| change | why |
|---|---|
| `--LBFGSMaxit 40000 --LBFGSMaxfun 40000` | L-BFGS exits *normally* at the cap, so the save path runs and weights are guaranteed. 40,000 sits just under where the first attempt already reached a loss better than E1's converged value. |
| `--LBFGSCheckpointIters 5000,...,40000` | periodic snapshots, so a kill can never cost the whole run again |
| `--SkipAdam`, guard 10.5 h -> 13 h | Adam cannot start once L-BFGS consumes the `--Tmax 9` budget, so skipping it makes the stopping point explicit rather than incidental; the raised guard leaves room for the save and the evaluators |

**How to report this arm:** stopped at 40,000 L-BFGS iterations rather than at its own convergence, and not schedule-matched to arms 1-3, which ran L-BFGS to convergence then Adam. State that its loss at the stopping point was already below E1's converged value, so it is a usable representability ceiling rather than an under-trained arm.


## 1. T4 gate

In [ ]:
import subprocess,os,sys,json,time,glob,shutil,queue,threading,datetime,csv,re
import numpy as np


gpu_name=subprocess.check_output(
    ['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
    text=True).strip()
print('GPU:',gpu_name)
subprocess.run(['nvidia-smi'],check=True)
if 'T4' not in gpu_name.upper():
    raise RuntimeError('Select a T4 runtime. Detected: '+gpu_name)
print('T4 confirmed.')

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULT_ROOT='/content/drive/MyDrive/ModalPINN_results'
ARM_TAG='dense_reference'
DEST=os.path.join(RESULT_ROOT,'K3_series',ARM_TAG)
os.makedirs(DEST,exist_ok=True)
print('arm        :',ARM_TAG)
print('destination:',DEST)

## 3. Legacy TF1 environment

In [ ]:
CONDA_ROOT='/content/miniconda'
LEGACY_PY='/content/miniconda/envs/modalpinn/bin/python'

if not os.path.exists(os.path.join(CONDA_ROOT,'bin','conda')):
    subprocess.run([
        'curl','-sL','-o','/tmp/miniconda.sh',
        'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh'
    ],check=True)
    subprocess.run(['bash','/tmp/miniconda.sh','-b','-p',CONDA_ROOT],check=True)

if not os.path.exists(LEGACY_PY):
    subprocess.run([
        os.path.join(CONDA_ROOT,'bin','conda'),'create','-y','-n','modalpinn',
        '-c','conda-forge','--override-channels',
        'python=3.7','cudatoolkit=10.0','cudnn=7.6.5'
    ],check=True)

subprocess.run([
    LEGACY_PY,'-m','pip','install',
    'tensorflow-gpu==1.14.0','numpy==1.17.4','scipy==1.3.2',
    'matplotlib==3.1.1','gputil','protobuf==3.11.3'
],check=True)

env=os.environ.copy()
env['MPLBACKEND']='Agg'
env['LD_LIBRARY_PATH']='/content/miniconda/envs/modalpinn/lib:'+env.get('LD_LIBRARY_PATH','')

rr=subprocess.run([
    LEGACY_PY,'-c',
    "import tensorflow as tf; print('TF',tf.__version__); "
    "s=tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(allow_soft_placement=False)); "
    "print('GPU devices:',[d.name for d in s.list_devices() if d.device_type=='GPU'])"
],env=env,capture_output=True,text=True)
print(rr.stdout)
print(rr.stderr[-2000:])
assert rr.returncode==0 and 'GPU:0' in rr.stdout

## 4. Write the source tree

In [ ]:
%%writefile Load_train_data_desync.py
# -*- coding: utf-8 -*-
"""
This file contains functionsspecific to
-Load_train_data_desync.py:
    Python file containing functions that extract and prepare data for training 
    and validation.
@author: Gaétan Raynaud
"""

# =============================================================================
# Library import
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib.pyplot as plt
from text_flow import read_flow
from reactions_process import extract_reactions

# =============================================================================
# matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


def gen_int_random_points(Npoints,geom,method = 'uniform', disp_plot = False, Delta_r_c = 0.5):
    """
    Generate spatial Navier--Stokes collocation points.

    R12/R14 random wake-biased sampler:
      30% whole-domain random
      35% formation wake random: 0 <= x < 3, |y| <= 2
      25% far wake random:       3 <= x <= 8, |y| <= 2
      10% near-cylinder random annulus: 0.5 < r <= 1.2

    R15 structured wake-biased grid:
      EXACTLY the same 30/35/25/10 source counts and regions, but each
      source component is deterministic and approximately equidistant:
        - Cartesian cell-centred grids in the three rectangular regions;
        - an area-regular staggered polar grid in the annulus.

    The multi-zone construction intentionally retains overlap between source
    components, just as the random wake-biased mixture does. There is no
    importance correction: wake/formation physics is deliberately upweighted.
    """
    print('Generating %d points for equations penalization'%(Npoints))
    print('Method = '+method)

    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom

    def uniform_box(n, xmin, xmax, ymin, ymax):
        xs = np.empty(n, dtype=np.float64)
        ys = np.empty(n, dtype=np.float64)
        filled = 0
        while filled < n:
            need = n-filled
            m = max(need*2, 64)
            xt = xmin + (xmax-xmin)*np.random.rand(m)
            yt = ymin + (ymax-ymin)*np.random.rand(m)
            ok = ((xt-x_c)**2 + (yt-y_c)**2 > r_c**2)
            xt = xt[ok]
            yt = yt[ok]
            take = min(need, len(xt))
            if take:
                xs[filled:filled+take] = xt[:take]
                ys[filled:filled+take] = yt[:take]
                filled += take
        return xs,ys

    def regular_box_exact(n, xmin, xmax, ymin, ymax):
        # Build a cell-centred Cartesian grid with dx ~ dy, reject the
        # cylinder, and (only if necessary) thin the slight oversupply with
        # evenly distributed deterministic indices. The output count is exact.
        width = float(xmax-xmin)
        height = float(ymax-ymin)
        if n <= 0 or width <= 0.0 or height <= 0.0:
            raise ValueError('Invalid regular_box_exact request.')

        # Start slightly above the ideal area count because the cylinder may
        # remove points. Increase resolution until at least n fluid points exist.
        target = int(np.ceil(1.02*n))
        nx = max(2, int(np.ceil(np.sqrt(target*width/height))))
        ny = max(2, int(np.ceil(target/float(nx))))

        while True:
            xv = xmin + (np.arange(nx)+0.5)*(width/nx)
            yv = ymin + (np.arange(ny)+0.5)*(height/ny)
            X,Y = np.meshgrid(xv,yv,indexing='xy')
            xx = X.ravel()
            yy = Y.ravel()
            ok = ((xx-x_c)**2 + (yy-y_c)**2 > r_c**2)
            xx = xx[ok]
            yy = yy[ok]
            if len(xx) >= n:
                break

            # Refine the direction with the larger current physical spacing.
            dx = width/nx
            dy = height/ny
            if dx >= dy:
                nx += 1
            else:
                ny += 1

        if len(xx) > n:
            # Even thinning across the flattened grid. Unlike random removal,
            # this preserves deterministic broad coverage and does not create
            # a large local hole.
            sel = np.floor(
                (np.arange(n)+0.5)*len(xx)/float(n)
            ).astype(np.int64)
            sel = np.minimum(sel, len(xx)-1)
            xx = xx[sel]
            yy = yy[sel]

        if len(xx) != n:
            raise RuntimeError('regular_box_exact failed to return exact count.')

        return xx.astype(np.float64),yy.astype(np.float64)

    def regular_annulus_exact(n, rin, rout):
        # Find a factorization nr*ntheta=n whose radial and circumferential
        # spacings are as similar as practical.
        best = None
        rmean = 0.5*(rin+rout)
        for nr in range(1,int(np.sqrt(n))+1):
            if n % nr != 0:
                continue
            nt = n//nr
            dr = (rout-rin)/float(nr)
            ds = 2*np.pi*rmean/float(nt)
            score = abs(np.log((dr+1e-30)/(ds+1e-30)))
            if best is None or score < best[0]:
                best = (score,nr,nt)

        _,nr,ntheta = best

        # Equal-area radial locations: r^2 is equally spaced.
        frac = (np.arange(nr)+0.5)/float(nr)
        rr = np.sqrt(rin**2 + frac*(rout**2-rin**2))

        xs=[]
        ys=[]
        dtheta=2*np.pi/float(ntheta)
        for i,rval in enumerate(rr):
            # Stagger neighboring rings by half an angular cell to avoid
            # artificial radial spokes.
            offset = 0.5*dtheta if (i % 2) else 0.0
            th = offset + np.arange(ntheta)*dtheta
            xs.append(x_c + rval*np.cos(th))
            ys.append(y_c + rval*np.sin(th))

        xx=np.concatenate(xs)
        yy=np.concatenate(ys)
        if len(xx) != n:
            raise RuntimeError('regular_annulus_exact count failure.')
        return xx.astype(np.float64),yy.astype(np.float64)

    if method in ('wake_biased','wake_biased_grid'):
        n_whole = int(round(0.30*Npoints))
        n_form  = int(round(0.35*Npoints))
        n_far   = int(round(0.25*Npoints))
        n_ann   = Npoints - n_whole - n_form - n_far

        if method == 'wake_biased':
            x0,y0 = uniform_box(n_whole, Lxmin,Lxmax,Lymin,Lymax)
            x1,y1 = uniform_box(n_form, max(0.0,Lxmin), min(3.0,Lxmax),
                                max(-2.0,Lymin), min(2.0,Lymax))
            x2,y2 = uniform_box(n_far, max(3.0,Lxmin), Lxmax,
                                max(-2.0,Lymin), min(2.0,Lymax))

            r_outer = 1.2
            theta = 2*np.pi*np.random.rand(n_ann)
            rr = np.sqrt(r_c**2 + (r_outer**2-r_c**2)*np.random.rand(n_ann))
            x3 = x_c + rr*np.cos(theta)
            y3 = y_c + rr*np.sin(theta)

        else:
            x0,y0 = regular_box_exact(
                n_whole,Lxmin,Lxmax,Lymin,Lymax)
            x1,y1 = regular_box_exact(
                n_form,max(0.0,Lxmin),min(3.0,Lxmax),
                max(-2.0,Lymin),min(2.0,Lymax))
            x2,y2 = regular_box_exact(
                n_far,max(3.0,Lxmin),Lxmax,
                max(-2.0,Lymin),min(2.0,Lymax))
            x3,y3 = regular_annulus_exact(n_ann,r_c,1.2)

        x = np.concatenate([x0,x1,x2,x3])
        y = np.concatenate([y0,y1,y2,y3])

        if len(x) != Npoints:
            raise RuntimeError('Wake-biased sampler returned wrong point count.')

        # Keep deterministic geometry for the grid case. Random mixture is
        # shuffled as before.
        if method == 'wake_biased':
            perm = np.random.permutation(Npoints)
            x = x[perm]
            y = y[perm]

        print('source mixture counts: whole=%d formation=%d far=%d annulus=%d' %
              (n_whole,n_form,n_far,n_ann))
        if method == 'wake_biased_grid':
            print('R15 structured/equidistant multi-zone grid active.')

    else:
        x = np.zeros(Npoints)
        y = np.zeros(Npoints)

        for j in range(Npoints):
            indomain = False
            while indomain == False:
                x_test = (Lxmax-Lxmin)*np.random.rand(1)[0] + Lxmin

                if method == 'y_normal':
                    y_test = np.random.normal(
                        y_c,0.5*(Lymax-Lymin),1)[0]

                elif method == '2zones' and np.random.uniform()>0.8:
                    r_random = r_c + np.random.uniform()*Delta_r_c
                    theta_random = np.random.uniform()*2*np.pi
                    x_test = x_c + r_random*np.cos(theta_random)
                    y_test = y_c + r_random*np.sin(theta_random)

                else:
                    y_test = (Lymax-Lymin)*np.random.rand(1)[0] + Lymin

                if (((x_test-x_c)**2 + (y_test-y_c)**2 > r_c**2) and
                        y_test < Lymax and y_test > Lymin):
                    x[j] = x_test
                    y[j] = y_test
                    indomain = True

    if disp_plot:
        plt.figure()
        plt.scatter(x,y,c='black',marker='.',s=1.)
        plt.xlabel('$x$')
        plt.ylabel('$y$')
        plt.title('Training points: '+method)
        plt.tight_layout()

    return x,y



def read_cut_simulation_data(filename_data,geom):
    '''
    Read simulation data results from filename_data file
    Crop data points in the fluid domain defined by Lxmin,Lxmax,Lymin,Lymax
    ----
    Return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    # Step 1 : data loading
    print('Simulation data reading...')
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(filename_data) 

    # Step 2 : Selection of points

    condition_cut_parts = np.array([np.where(nodes_X[0,:] < Lxmax,True,False), \
                  np.where(nodes_X[0,:] > Lxmin,True,False), \
                  np.where(nodes_Y[0,:] > Lymin,True,False), \
                  np.where(nodes_Y[0,:] < Lymax,True,False)])
    condition_cut = np.all(condition_cut_parts,axis=0)

    index_cut = np.argwhere(condition_cut)[:,0]

    # Step 3 : cropping

    nodes_X = nodes_X[:,index_cut]
    nodes_Y = nodes_Y[:,index_cut]
    Us = Us[:,index_cut]
    Vs = Vs[:,index_cut]
    Ps = Ps[:,index_cut]

    print('Reading and cropping simulation data ... ok')

    return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps


def cut_data_index(index_space,nodes_X, nodes_Y, Us, Vs, Ps):
    '''
    return data nodes_X, nodes_Y, Us, Vs, Ps [:,index_space]
    '''
    nodes_X = nodes_X[:,index_space]
    nodes_Y = nodes_Y[:,index_space]
    Us = Us[:,index_space]
    Vs = Vs[:,index_space]
    Ps = Ps[:,index_space]

    return nodes_X, nodes_Y, Us, Vs, Ps


def cut_simu_cylinder_only(geom,nodes_X, nodes_Y, Us, Vs, Ps, n_taps=30):
    '''
    nodes_X, nodes_Y, Us, Vs, Ps : [Ntime,Nelt] array of scalars
    feom : array containing geom info [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c]
    n_taps : number of pressure taps to select uniformly around the cylinder border
    return data from the points located on the cylinder border
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom

    eps = 1e-5
    r = np.sqrt(np.square(nodes_X[0,:]-x_c) + np.square(nodes_Y[0,:]-y_c))
    delta_r_rc = np.square(r-r_c)

    condition_cut_cyl = np.where(delta_r_rc < eps, True, False)
    index_cylinder = np.argwhere(condition_cut_cyl)[:,0]

    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_cylinder,nodes_X, nodes_Y, Us, Vs, Ps)

    # Select 30 randomly points
    # np.random.shuffle(index_cylinder)
    # index_cylinder = index_cylinder[:30]

    # Select n_taps points that are the nearest from a uniform disposition over the cylinder
    # endpoint=False so that requesting n_taps genuinely yields n_taps distinct physical
    # locations (endpoint=True would place s=0 and s=1 on the same point on the circle)
    s_lin = np.linspace(0.,1.,n_taps,endpoint=False)
    x_points = x_c + r_c*np.cos(2*np.pi*s_lin)
    y_points = y_c + r_c*np.sin(2*np.pi*s_lin)

    index_reduce = 0*x_points
    index_reduce = np.asarray([int(i) for i in index_reduce])
    for k in range(len(x_points)):
        index_reduce[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))

    print('Cylinder taps requested: %d, distinct mesh nodes found: %d' % (n_taps, len(np.unique(index_reduce))))

    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_reduce,nodes_X, nodes_Y, Us, Vs, Ps)

    return nodes_X, nodes_Y, Us, Vs, Ps

def cut_simu_pitot_only(geom,nodes_X, nodes_Y, Us, Vs, Ps):
    '''
    nodes_X, nodes_Y, Us, Vs, Ps : [Ntime,Nelt] array of scalars
    feom : array containing geom info [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c] 
    Return data from the points on pitot points locations
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    d = 2*r_c
    # Step 2 : defining the position of wanted points
    N_per_section = 10
    x_points = np.zeros(4*N_per_section)
    y_points = np.zeros(4*N_per_section)

    # line 1
    x_points[:N_per_section] = -3.*d*np.ones(N_per_section)
    y_points[:N_per_section] = np.linspace(Lymin,Lymax,N_per_section)

    # line 2 post cylinder
    x_points[N_per_section:2*N_per_section] = d*np.ones(N_per_section)
    y_points[N_per_section:2*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)

    # line 3 
    x_points[2*N_per_section:3*N_per_section] = 2*d*np.ones(N_per_section)
    y_points[2*N_per_section:3*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)

    # line 4
    x_points[3*N_per_section:4*N_per_section] = 3*d*np.ones(N_per_section)
    y_points[3*N_per_section:4*N_per_section] = np.linspace(Lymin,Lymax,N_per_section)

    # Step 3 : finding closest point in data
    index_pitot = 0*x_points
    index_pitot = np.asarray([int(i) for i in index_pitot])
    for k in range(len(x_points)):
        index_pitot[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))

    nodes_X, nodes_Y, Us, Vs, Ps = cut_data_index(index_pitot,nodes_X, nodes_Y, Us, Vs, Ps)

    return nodes_X, nodes_Y, Us, Vs, Ps


def read_cut_simulation_data_exp_point_and_cylinder(filename_data,geom,n_taps=30):
    '''
    Read simulation data results from filename_data_result
    Pick out data that are the nearest from simulated experimental measurement points
    n_taps : number of pressure taps to select uniformly around the cylinder border
    ----
    Return Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    # Step 1 : data loading and cut into studied domain [Lxmin,Lxmax]x[Lymin,Lymax]
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)

    data_pitot = cut_simu_pitot_only(geom,nodes_X, nodes_Y, Us, Vs, Ps)
    #data_pitot = [x_pitot, y_pitot, u_pitot_, v_pitot, p_pitot]

    data_cyl = cut_simu_cylinder_only(geom,nodes_X, nodes_Y, Us, Vs, Ps, n_taps=n_taps)
    #data_cyl = [x_cyl, y_cyl, u_cyl, v_cyl, p_cy]

    return times,data_cyl,data_pitot

def read_cut_simulation_data_inlet_points(filename_data,geom):
    '''
    Read simulation data results from filename_data
    Select points from a uniform sampling of location at x = Lxmin and between y = Lymin to y= Lymax on 10 points
    Return times, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet 
    where times [Nt,] list of instants
    and x,y,u,v,p are of size [Nt,10]
    '''    

    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)

    # Select 10 points that are the nearest from a uniform disposition over the inlet
    s_lin = np.linspace(0.,1.,10)
    x_points = Lxmin + 0.*s_lin
    y_points = Lymin + s_lin*(Lymax-Lymin)

    index_reduce = 0*x_points
    index_reduce = np.asarray([int(i) for i in index_reduce])
    for k in range(len(x_points)):
        index_reduce[k] = np.argmin(np.square(nodes_X[0,:] - x_points[k])+np.square(nodes_Y[0,:] - y_points[k]))

    x_inlet, y_inlet, u_inlet, v_inlet, p_inlet = cut_data_index(index_reduce,nodes_X, nodes_Y, Us, Vs, Ps)


    return times, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet


def data_flatten_cut(x,y,t,u,v,p,Nmes=0):
    '''
    x,y,t,u,v,p : [Nt,Nelts] array
    Nmes (int) : random truncature of data to Nmes. If Nmes= 0, no truncature is performed
    ----
    return x_ft,y_ft,t_ft,u_ft,v_ft,pft flattened and truncated : 1D array of size [Nmes,] or [Nt*Nelts] if Nmes = 0
    '''

    index = np.array(range(len(x[0,:])*len(x[:,0])))

    if Nmes != 0:
        np.random.shuffle(index)
        index = index[:Nmes]

    x = np.ndarray.flatten(x)[index]
    y = np.ndarray.flatten(y)[index]
    t = np.ndarray.flatten(t)[index]
    u = np.ndarray.flatten(u)[index]
    v = np.ndarray.flatten(v)[index]
    p = np.ndarray.flatten(p)[index]

    return x,y,t,u,v,p

def get_reactions(filename,timemin=-1.,timemax=1e10):
    '''
    Extract forces on cylinder from a .reactions file using extract_reactions()
    Cut time axis between timemin and timemax
    Return times, Fx, Fy
    '''
    times, Fx, Fy, Mz, flag = extract_reactions(filename)

    condition_cut_parts = np.array([np.where(times > timemin, True, False), np.where(times < timemax, True, False)])    
    condition_cut = np.all(condition_cut_parts,axis=0)

    index_cut = np.argwhere(condition_cut)[:,0]

    times = times[index_cut]
    Fx = Fx[index_cut]
    Fy = Fy[index_cut]

    return times, Fx, Fy


def addNoise(x,stdNoise):
    """
    x : 1D np array
    stdNoise float > 0. : standard deviation of Gaussian Noise
    Return x + epsilon, epsilon ~ N(0,std)
    """

    return x + np.random.normal(loc=0.0,scale=stdNoise,size=len(x))


def training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmin=0.,Tintmax=1e2,cut=True,data_selection='all',desync=False,multigrid=False,Ngrid=10,stdNoise=0.,method_int = '2zones',n_taps=30):
    '''
    cut = True : if True, cut data set to only Nmes points
    if False, keep all the values
    ---
    data_selection (str) :
        if 'all' : returns data randomly sampled in fluid domain in quantity Nmes
        if 'inlet' : returns data at 10 points uniformly separated points at inlet
        if 'cylinder_only' : returns data only from points on the border of the cylinder
        if 'cylinder_pitot' : returns data from cylinder border and on pitot points
        if 'pitot_only'  returns data only at pitot points
    desync = False : (bool) if True, add a uniformly distributed phase shift for measuremnts in pitot and cylinder at each position
    multigrid = False (bool) if True, return a list of size Ngrid, each element containing a sampling of space-time coordinates of size Nint
    Ngrid (int) length of the list of sampled space-time coordinates returned when multigrid = True
    '''
    # Part 1 : border normalised coordinate
    s_train = np.random.rand(Nbc)
    # Part 2 : int points

    if multigrid:
        x_int = []
        y_int = []
        t_int = []
        for k in range(Ngrid):
            x_int_temp,y_int_temp = gen_int_random_points(Nint,geom,method = method_int,disp_plot=False)
            x_int.append(x_int_temp)
            y_int.append(y_int_temp)
            t_int.append(Tintmin + (Tintmax-Tintmin)*np.random.rand(Nint))
    else:
        x_int,y_int = gen_int_random_points(Nint,geom,method = method_int,disp_plot=False)
        t_int = Tintmin + (Tintmax-Tintmin)*np.random.rand(Nint)

    # Part 3 : simulation data points


    if data_selection == 'all':

        Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)
        times_dedouble = np.asarray([t*np.ones(len(nodes_X[0,:])) for t in times])
        if cut:
            xmes,ymes,tmes,umes,vmes,pmes = data_flatten_cut(nodes_X,nodes_Y,times_dedouble,Us,Vs,Ps,Nmes)
        else:
            xmes,ymes,tmes,umes,vmes,pmes = data_flatten_cut(nodes_X,nodes_Y,times_dedouble,Us,Vs,Ps)

        return x_int,y_int,t_int,s_train,xmes,ymes,tmes,umes,vmes,pmes

    elif data_selection == 'inlet':

        times2, x_inlet, y_inlet, u_inlet, v_inlet, p_inlet = read_cut_simulation_data_inlet_points(filename_data,geom)
        t_inlet = np.asarray([t*np.ones(len(x_inlet[0,:])) for t in times2])
        x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet = data_flatten_cut(x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet)


        return x_int, y_int, t_int, s_train, x_inlet, y_inlet, t_inlet, u_inlet, v_inlet, p_inlet

    else:
        cut = False
        times,data_cyl,data_pitot = read_cut_simulation_data_exp_point_and_cylinder(filename_data,geom,n_taps=n_taps)

        # Cylinder data
        xmes_cyl, ymes_cyl, umes_cyl, vmes_cyl, pmes_cyl = data_cyl
        tmes_cyl = np.asarray([t*np.ones(len(xmes_cyl[0,:])) for t in times])
        xmes_cyl, ymes_cyl, tmes_cyl, umes_cyl, vmes_cyl, pmes_cyl = data_flatten_cut(xmes_cyl, ymes_cyl, tmes_cyl, umes_cyl, vmes_cyl, pmes_cyl)
        pmes_cyl = addNoise(pmes_cyl,stdNoise)

        # Pitot data
        xmes_pitot, ymes_pitot, umes_pitot, vmes_pitot, pmes_pitot = data_pitot

        Nxpitot = 40
        TDesyncMax = 6.06
        if desync:
            #Delta_phi_np_pitot = np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            Delta_t_np_pitot = np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            tmes_pitot = np.asarray([[times[t] + Delta_t_np_pitot[k] for k in range(len(xmes_pitot[t,:]))] for t in range(len(times))])
        else:
            Delta_t_np_pitot = 0.*np.random.uniform(low=0.0,high=TDesyncMax, size=Nxpitot)
            tmes_pitot = np.asarray([t*np.ones(len(xmes_pitot[0,:])) for t in times])

        xmes_pitot, ymes_pitot, tmes_pitot, umes_pitot, vmes_pitot, pmes_pitot = data_flatten_cut(xmes_pitot, ymes_pitot, tmes_pitot, umes_pitot, vmes_pitot, pmes_pitot)
        umes_pitot = addNoise(umes_pitot,stdNoise)
        vmes_pitot = addNoise(vmes_pitot,stdNoise)
        pmes_pitot = addNoise(pmes_pitot,stdNoise)

        if data_selection == 'cylinder_only':
            return x_int,y_int,t_int,s_train,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl

        elif data_selection == 'pitot_only':

            return x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,Delta_t_np_pitot

        elif data_selection == 'cylinder_pitot' :

            return x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl,Delta_t_np_pitot

        else :

            return x_int,y_int,t_int,s_train

def find_pression_static_amont():
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_cut_simulation_data(filename_data,geom)
    x_target = -4.
    y_target = -4.
    index = np.argmin(np.square(nodes_X[0,:] - x_target)+np.square(nodes_Y[0,:] - y_target))
    plt.figure()
    plt.plot(times,Ps[:,index])
    plt.xlabel('$t$')
    plt.ylabel('Pressure $p$')

In [ ]:
%%writefile ModalPINN_VortexShedding.py
# -*- coding: utf-8 -*-
"""
ModalPINN Python Code
This is the main Python file for performing flow reconstruction using ModalPINN
as described in the paper

    ModalPINN : an extension of Physics-Informed Neural Networks with enforced 
    truncated Fourier decomposition  for periodic flow reconstruction using a 
    limited number of imperfect sensors. 
    G. Raynaud, S. Houde, F. P. Gosselin (2021)

This file contains the main losses functions of the ModalPINN as well as the 
main steps of the training. Nonetheless, it calls functions from 
-Load_train_data_desync.py:
    Python file containing functions that extract and prepare data for training 
    and validation.
-NN_functions.py:
    Python file containing functions specific to
        o neural networks (construction, initialisation),
        o optimisers (calling from scipy or tf interfaces, initialisation, training steps),
        o plots.

This file is designed to be launched on a computationel cluster (initially for 
Compute Canada - Graham server) using the following batch commands:
    #!/bin/bash
    #SBATCH --gres=gpu:t4:1
    #SBATCH --nodelist=gra1337
    #SBATCH --cpus-per-task=2
    #SBATCH --mem=50G
    #SBATCH --job-name=ModalPINN
    #SBATCH --time=0-10:00

    module load python/3.7.4
    source ~/ENV/bin/activate
    python ./ModalPINN_VortexShedding.py --Tmax 9 --Nmes 5000 --Nint 50000 --multigrid --Ngrid 5 --NgridTurn 200 --WidthLayer 25 --Nmodes 3 
    deactivate

For each job launched, a folder is created in ./OutputPythonScript and is 
identified by date-time information. In this folder, the content of consol prints 
is saved in a out.txt file alongside other files (mode shapes, various plots...)
including the model itself in a pickle archive.

Please refer to the help for the arguments sent to the parser and to the readme 
for librairies requirements.


@author: Gaétan Raynaud. 
ORCID : orcid.org/0000-0002-2802-7366
email : gaetan.raynaud (at) polymtl.ca
"""

# =============================================================================
# Librairies Import
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import datetime
import os
import pickle
from shutil import copyfile
import sys
import GPUtil
import time
import argparse
import glob
import resource
from tensorflow.python.client import device_lib

# Code parts
import NN_functions as nnf
import Load_train_data_desync as ltd

def print_mem(tag):
    '''Peak resident memory so far (KB on Linux), printed on its own line at
    a few key checkpoints - added while diagnosing an OOM traced to R5's
    --K0Loss/--CV1Loss graph construction (see PROJECT_LOG.md). Cheap
    (a single getrusage syscall), left in permanently since colab-cli
    sessions have a hard 12GB memcg limit and future loss terms may hit it
    again - this makes the next one fast to diagnose instead of another
    multi-hour trial-and-error round.'''
    print('MEM %s : %d MB' % (tag, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024))

# Link to simulations data 
# In the paper, we used those from Boudina et al. (2020) that can be downloaded 
# at https://zenodo.org/record/5039610
filename_data = 'Data/fixed_cylinder_atRe100'

t0 = time.time()
# =============================================================================
# matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


# =============================================================================
# Preparing the writing of console prints in out.txt
# =============================================================================

class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush() # If you want the output to be visible immediately
    def flush(self) :
        for f in self.files:
            f.flush()

# =============================================================================
# File copy and folder creation
# Here we create a folder containing all the data of this job
# And we copy current python files to keep track of how the job was launched
# =============================================================================
r = int(np.ceil(1000*np.random.rand(1)[0])) # This random number is used in case 2 jobs are launched at the exact same time so that the newly created folders does not merge the one into the other
d = datetime.datetime.now()
pythonfile = os.path.basename(__file__)
repertoire = 'OutputPythonScript/ModalPINN_'+ d.strftime("%Y_%m_%d-%H_%M_%S") + '__' +str(r)
os.makedirs(repertoire, exist_ok=True)
copyfile(pythonfile,repertoire+'/Copy_python_script.py')
copyfile('NN_functions.py',repertoire+'/NN_functions.py')
copyfile('Load_train_data_desync.py',repertoire+'/Load_train_data_desync.py')

f = open(repertoire+'/out.txt', 'w')
original = sys.stdout
sys.stdout = Tee(sys.stdout, f)
print('File copy and stdout ok')


# Print devices available


list_devices = device_lib.list_local_devices()
print('Devices available')
print(list_devices)

# =============================================================================
# Set arguments passed through bash
# =============================================================================

parser = argparse.ArgumentParser()

parser.add_argument('--Tmax',type=float,default=None,help="Define the max time allowed for optimisation (in hours)")
parser.add_argument('--Nmodes',type=int,default=2,help="Number of modes, including zero frequency")
parser.add_argument('--Nmes',type=int,default=5000,help="Number of measurement points to provide for optimisation")
parser.add_argument('--Nint',type=int,default=50000,help="Number of computing points to provide for equation evaluation during optimisation")
parser.add_argument('--LossModes',action="store_true",default=False,help="Use of modal equations during optimisation")
parser.add_argument('--multigrid',action="store_true",default=False,help="Use of multi grid")
parser.add_argument('--Ngrid',type=int,default=1,help="Number of batch for Adam optimization")
parser.add_argument('--NgridTurn',type=int,default=1000,help="Number of iterations between each batch changement")
parser.add_argument('--Noise',type=float,default=0.,help="Define standard deviation of gaussian noise added to measurements")
parser.add_argument('--WidthLayer',type=int,default=20,help="Number of neurons per layer and per mode")
parser.add_argument('--SparseData',action="store_true",default=False,help="if activated, use simulated  measurements data for training. Else use dense data")
parser.add_argument('--DesyncSparseData',action="store_true",default=False,help="if activated (and --SparseData == True), then simulated measurements are randomly made out of synchronisation")
parser.add_argument('--TwoZonesSampling',action="store_true",default=False,help="if activated, the sampling of equation penalisation points is carried out using 2 zones (with more points near the cylinder). Else use a uniform sampling")
parser.add_argument('--WakeBiasedSampling',action="store_true",default=False,help="R12: use 30% whole-domain uniform + 35% formation wake + 25% far wake + 10% near-cylinder annulus for NS collocation.")
parser.add_argument('--WakeBiasedGridSampling',action="store_true",default=False,help="R15: same 30/35/25/10 wake-biased source allocation as --WakeBiasedSampling, but deterministic approximately equidistant Cartesian/polar grids.")
parser.add_argument('--PressureOnly',action="store_true",default=False,help="if activated (requires --SparseData), drop pitot (u,v) velocity measurements and train only on cylinder-surface pressure taps.")
parser.add_argument('--NTaps',type=int,default=30,help="Number of pressure taps sampled uniformly around the cylinder border when --SparseData is used.")
parser.add_argument('--Seed',type=int,default=0,help="Seed for numpy and TensorFlow RNGs, for reproducible comparisons across tap counts.")
parser.add_argument('--FreestreamBC',action="store_true",default=False,help="Blend the network's mean velocity mode toward the known free-stream value (u=u_in, v=0) near the inlet, upstream of the cylinder. A second, independent prior alongside the existing cylinder no-slip encoding - not used downstream/in the wake, where the real flow is not free-stream.")
parser.add_argument('--FluctuationInletBC',action="store_true",default=False,help="Damp the fluctuating velocity modes (k>=1) toward zero at the inlet, using the same ramp as --FreestreamBC. Shedding is a wake phenomenon; nothing previously stopped a spurious oscillation from leaking upstream. Velocity only - pressure fluctuations do reach the inlet.")
parser.add_argument('--BVF',action="store_true",default=False,help="Add the Lighthill boundary-vorticity-flux loss term: enforces (1/Re)*d(omega)/dn = (1/R)*dp/dtheta at the cylinder wall, using a target derived from the same pressure taps (see bvf_targets.py). Requires --BVFTargets.")
parser.add_argument('--LambdaBVF',type=float,default=1.0,help="Weight of the BVF loss term when --BVF is set.")
parser.add_argument('--BVFTargets',type=str,default=None,help="Path to the .npz file produced by bvf_targets.py, required when --BVF is set.")
parser.add_argument('--CausalWeighting',action="store_true",default=False,help="Reweight the physics-residual loss so points near x_front (starting near the cylinder) count for more early in training, expanding downstream over --CausalWarmupIters iterations. Addresses the PINN 'causality violation' pathology (Wang et al. 2022) where distant collocation points can score near-zero residual for a near-zero (wrong) field just as easily as for a correct one, removing any training-time incentive to propagate the wake downstream. No new loss term, no new derivative order - same NS residual, just reweighted.")
parser.add_argument('--CausalSteepness',type=float,default=2.0,help="Sigmoid steepness of the causal weighting ramp (see --CausalWeighting).")
parser.add_argument('--CausalStartX',type=float,default=-1.0,help="Starting x position of the causal weighting frontier (see --CausalWeighting).")
parser.add_argument('--CausalEndX',type=float,default=8.0,help="Ending x position of the causal weighting frontier - the domain's downstream edge (see --CausalWeighting).")
parser.add_argument('--CausalWarmupIters',type=int,default=3000,help="Number of L-BFGS/Adam iterations over which the causal weighting frontier sweeps from --CausalStartX to --CausalEndX (see --CausalWeighting). Deliberately an absolute iteration count, not a fraction of some assumed total iteration budget: L-BFGS's actual convergence point isn't known in advance (R3 converged at ~15452 iterations, far short of the 50000 cap), so anchoring to a fraction of maxit risks the frontier never finishing its sweep before ftol is satisfied. 3000 is comfortably smaller than that precedent, leaving most of training at full-domain weighting to consolidate.")
parser.add_argument('--K0Loss',action="store_true",default=False,help="Add the k=0 harmonic momentum residual (mean-flow / Reynolds-stress balance, from the existing modal-equation machinery) as a separate weighted loss term. A dead (zero-amplitude) wake cannot satisfy this equation, since it needs the quadratic Reynolds-stress divergence that only a live oscillating wake supplies (Noack 2003 / Mantic-Lugo 2014 mean-field coupling). Also hard-projects the k=0 (mean) mode onto its real part in out_nn_modes_uv/out_nn_modes_p (see NN_functions), removing an otherwise-unconstrained imaginary gauge direction that would poison this loss (see 'R5 measured best candidates plan.md').")
parser.add_argument('--LambdaK0',type=float,default=0.025,help="Weight of the k=0 harmonic momentum residual loss term when --K0Loss is set. Calibrated via audit_r5_losses.py --Mode checkpoint against R3's early/mid-training loss magnitude (~0.1-0.3, the iteration range where R3's wake-collapse decision actually got locked in), not its tiny converged value (1.4e-4) - a lambda calibrated against the converged loss would be ~1000x smaller and make this term negligible during the exact training window where it needs to matter.")
parser.add_argument('--CV1Loss',action="store_true",default=False,help="Add the k=1 control-volume integral momentum balance (six fixed wake boxes) as a separate weighted loss term - a first-derivative-only, integral form of the k=1 momentum equation that measured strong (and adversarially robust) sensitivity to wake collapse in the diagnostic loss study. See 'R5 measured best candidates plan.md'.")
parser.add_argument('--LambdaCV1',type=float,default=0.0075,help="Weight of the k=1 control-volume integral loss term when --CV1Loss is set. Calibrated the same way as --LambdaK0 (see there) - against R3's early/mid-training loss magnitude, not its converged value.")
parser.add_argument('--HardSym',action="store_true",default=False,help="Hard-enforce the Karman-street mode parity (u_k,p_k even/odd, v_k odd/even in y per (-1)^k) by reflection-symmetrizing each mode's output in out_nn_modes_uv/out_nn_modes_p. Roughly doubles mode-network forward evaluations (graph size/step time). Optional, go/no-go at the smoke test (see plan).")
parser.add_argument('--SkipDiagnostics',action="store_true",default=False,help="R9: skip the post-training 'Error details' tf_print block. Those diagnostics evaluate graphs never built during training (notably the modal-equation residual, Loss_int_mode_wrap, when training used the time-collocation loss); TF 1.14 builds+optimizes each on first sess.run, which measured >10 min of total silence with the street ops threaded in - the R9 smoke test's stall guard killed exactly this. Used by the smoke test (training+save is what the gate proves); the real run keeps full diagnostics, protected by the pre-diagnostics safety save.")
parser.add_argument('--LBFGSFtol',type=float,default=1e-12,help="ftol passed to declare_LBFGS (ported from R6/R8, same rationale): the scipy default (~2.22e-16, machine epsilon) made R5's L-BFGS terminate after only 4574 iterations via a line-search failure at that unusually tight tolerance, not genuine convergence. 1e-12 is the value R6-R8 ran with.")
parser.add_argument('--TrustStreet',action="store_true",default=False,help="R9: wrap the k>=1 mode networks as a bounded trust-region correction around the closed-form von Karman street prior derived from the taps alone (see street_prior.py + NN_functions.street_modes_k). q_k = S_k + (rho|S_k| + cap)*tanh-bounded network correction, so the dead-wake solution q_k=0 of R1-R8 is excluded from the search space wherever the street is alive. The k=0 (mean) modes stay free networks. Validated in R9_wake_rescue/ (testbed: far-wake E_v 1.0 -> 0.40).")
parser.add_argument('--StreetPrior',type=str,default=None,help="Path to the street_prior_Ntap<N>.npz file produced by street_prior.py (required when --TrustStreet is set).")
parser.add_argument('--TrustRho',type=float,default=0.6,help="Trust-region radius as a fraction of the local street mode amplitude |S_k| (see --TrustStreet).")
parser.add_argument('--TrustCap',type=float,default=0.12,help="Additive floor of the trust-region radius - keeps a minimal correction capacity where the street prior is small (formation region, upstream). NOTE: in this codebase's one-sided mode convention (no factor 2 in NN_time_*) this value corresponds to 0.06 in the R9_wake_rescue testbed's two-sided convention (see street_modes_k's docstring).")

# R10: targeted radial trust candidate selected by the CFD diagnostic notebook.
parser.add_argument('--V1RadialTrust',action="store_true",default=False,help="Apply a proper complex radial trust region ONLY to v mode k=1 in the downstream wake. u, p, k=0 and k>=2 remain ordinary ModalPINN outputs.")
parser.add_argument('--V1TrustRho',type=float,default=0.70,help="Radial correction radius fraction rho for downstream v1. Candidate diagnostic selected rho=0.70 at x>=3D.")
parser.add_argument('--V1TrustXStart',type=float,default=3.0,help="Streamwise start of the v1 radial trust gate in D.")
parser.add_argument('--V1TrustXWidth',type=float,default=0.30,help="Smooth tanh transition width of the v1 radial trust gate.")
parser.add_argument('--V1TrustYMax',type=float,default=2.0,help="Half-width of the central wake band for v1 radial trust.")
parser.add_argument('--V1TrustYWidth',type=float,default=0.20,help="Smooth tanh transverse transition width.")
parser.add_argument('--LBFGSMaxit',type=int,default=50000,help="Maximum L-BFGS iterations. R10 smoke uses a deliberately short cap.")
parser.add_argument('--LBFGSMaxfun',type=int,default=50000,help="Maximum L-BFGS function evaluations.")
parser.add_argument('--SkipAdam',action="store_true",default=False,help="Skip the Adam polish after L-BFGS. Used for matched short smoke comparisons.")
parser.add_argument('--ExitAfterSafetySave',action="store_true",default=False,help="Exit cleanly immediately after the pre-diagnostics safety model save. Intended for smoke/preflight runs that only need trained weights for external evaluation; avoids legacy post-processing that assumes a non-empty Adam history.")
parser.add_argument('--RestoreModel',type=str,default=None,help='Warm-start from an existing DNN...pickle checkpoint. Restored tensors remain trainable.')
parser.add_argument('--LBFGSCheckpointIters',type=str,default='',help='R14: comma-separated accepted L-BFGS iterations to checkpoint, e.g. 250,500,1000,2000,4000,6000. Checkpoint 0 and the final accepted step are always included when this option is non-empty.')


args = parser.parse_args()

if args.PressureOnly and not args.SparseData:
    raise ValueError('--PressureOnly requires --SparseData to also be set.')

if args.TrustStreet and args.StreetPrior is None:
    raise ValueError('--TrustStreet requires --StreetPrior <path to street_prior_Ntap<N>.npz produced by street_prior.py>.')

if args.V1RadialTrust and args.StreetPrior is None:
    raise ValueError('--V1RadialTrust requires --StreetPrior <street_prior_Ntap<N>.npz>.')
if args.V1RadialTrust and args.TrustStreet:
    raise ValueError('Use either --V1RadialTrust or legacy --TrustStreet, not both in the same run.')

if args.V1RadialTrust:
    if not (0.0 < args.V1TrustRho < 1.0):
        raise ValueError('--V1TrustRho must satisfy 0 < rho < 1 for the anti-collapse guarantee.')
    if args.V1TrustXWidth <= 0.0 or args.V1TrustYWidth <= 0.0:
        raise ValueError('--V1TrustXWidth and --V1TrustYWidth must be strictly positive.')

if args.BVF and args.BVFTargets is None:
    raise ValueError('--BVF requires --BVFTargets <path to the .npz file produced by bvf_targets.py>.')

if args.CV1Loss and not args.SparseData:
    raise ValueError('--CV1Loss requires --SparseData to also be set (the Vin-append anti-escape-hatch mechanism is only wired into the sparse-data collocation path).')

print('Args passed to python script')
print('Tmax '+str(args.Tmax)+' (h)')
print('Nmodes %d' % (args.Nmodes))
print('Nmes %d' % (args.Nmes))
print('Nint %d' % (args.Nint))
print('Use Loss Modes : ' + str(args.LossModes))
print('Multigrid : '+str(args.multigrid))
print('Ngrid : '+str(args.Ngrid))
print('Ngrid Turn : '+str(args.NgridTurn))
print('STD Noise : %.2e' % (args.Noise))
print('Neurons per layer and per mode : %d' % (args.WidthLayer))
print('Sparse Data : ' + str(args.SparseData))
print('Desync Sparse Data : ' + str(args.DesyncSparseData))
print('Pressure Only : ' + str(args.PressureOnly))
print('NTaps : %d' % (args.NTaps))
print('Seed : %d' % (args.Seed))
print('Freestream BC : ' + str(args.FreestreamBC))
print('Fluctuation Inlet BC : ' + str(args.FluctuationInletBC))
print('BVF : ' + str(args.BVF))
print('Lambda BVF : %.4g' % (args.LambdaBVF))
print('BVF Targets : ' + str(args.BVFTargets))
print('Causal Weighting : ' + str(args.CausalWeighting))
print('Causal Steepness : %.4g' % (args.CausalSteepness))
print('Causal Start X : %.4g' % (args.CausalStartX))
print('Causal End X : %.4g' % (args.CausalEndX))
print('Causal Warmup Iters : %d' % (args.CausalWarmupIters))
print('K0 Loss : ' + str(args.K0Loss))
print('Lambda K0 : %.4g' % (args.LambdaK0))
print('CV1 Loss : ' + str(args.CV1Loss))
print('Lambda CV1 : %.4g' % (args.LambdaCV1))
print('Hard Sym : ' + str(args.HardSym))
print('V1 Radial Trust : ' + str(args.V1RadialTrust))
print('V1 Trust rho/xstart/xwidth/ymax/ywidth : %.3g / %.3g / %.3g / %.3g / %.3g' % (args.V1TrustRho,args.V1TrustXStart,args.V1TrustXWidth,args.V1TrustYMax,args.V1TrustYWidth))
print('L-BFGS maxit/maxfun : %d / %d' % (args.LBFGSMaxit,args.LBFGSMaxfun))
print('Skip Adam : ' + str(args.SkipAdam))

_sampling_flags = [
    bool(args.TwoZonesSampling),
    bool(args.WakeBiasedSampling),
    bool(args.WakeBiasedGridSampling)
]
if sum(_sampling_flags) > 1:
    raise ValueError('Choose only one collocation sampler flag.')

if args.WakeBiasedGridSampling:
    IntSampling = 'wake_biased_grid'
elif args.WakeBiasedSampling:
    IntSampling = 'wake_biased'
elif args.TwoZonesSampling:
    IntSampling = '2zones'
else:
    IntSampling = 'uniform'

print('Sampling of V_in : '+IntSampling)
if args.WakeBiasedSampling:
    print('wake-biased RANDOM source mixture: 30% whole / 35% formation / 25% far / 10% annulus')
if args.WakeBiasedGridSampling:
    print('R15 wake-biased REGULAR GRID: 30% whole / 35% formation / 25% far / 10% polar annulus')

# =============================================================================
# Reproducibility: seed numpy and TF graph-level RNG
# =============================================================================
np.random.seed(args.Seed)
tf.compat.v1.set_random_seed(args.Seed)
print('Random seed set to %d' % (args.Seed))
print_mem('startup (before graph construction)')

repertoire_new = repertoire
if args.PressureOnly:
    repertoire_new = repertoire_new + '_Ponly_Ntap%d' % (args.NTaps)
if args.FreestreamBC:
    repertoire_new = repertoire_new + '_FSBC'
if args.FluctuationInletBC:
    repertoire_new = repertoire_new + '_FIBC'
if args.TwoZonesSampling:
    repertoire_new = repertoire_new + '_2zones'
if args.WakeBiasedSampling:
    repertoire_new = repertoire_new + '_WAKEBIAS'
if args.WakeBiasedGridSampling:
    repertoire_new = repertoire_new + '_WAKEGRID'
if args.BVF:
    repertoire_new = repertoire_new + '_BVF_lam%s' % (str(args.LambdaBVF).replace('.', 'p'))
if args.CausalWeighting:
    repertoire_new = repertoire_new + '_Causal_s%s' % (str(args.CausalSteepness).replace('.', 'p'))
if args.K0Loss:
    repertoire_new = repertoire_new + '_K0RS_lam%s' % (str(args.LambdaK0).replace('.', 'p'))
if args.CV1Loss:
    repertoire_new = repertoire_new + '_CV1_lam%s' % (str(args.LambdaCV1).replace('.', 'p'))
if args.HardSym:
    repertoire_new = repertoire_new + '_SYM'
if args.V1RadialTrust:
    _r = str(args.V1TrustRho).replace('.', 'p')
    _x = str(args.V1TrustXStart).replace('.', 'p')
    repertoire_new = repertoire_new + '_V1RAD_rho%s_x%s' % (_r, _x)
if args.RestoreModel is not None:
    repertoire_new = repertoire_new + '_WARM'
if repertoire_new != repertoire:
    os.rename(repertoire, repertoire_new)
    repertoire = repertoire_new
    print('Repertoire renamed to '+repertoire)

if args.RestoreModel is not None:
    copyfile(args.RestoreModel, repertoire+'/initial_model_used.pickle')
    print('Copied warm-start checkpoint into run folder.')

# =============================================================================
# Physical and geometrical parameters 
# =============================================================================

Re = 100.
Lxmin = -4. 
Lxmax = 8. 
Lx = Lxmax-Lxmin
Lymin = -4. 
Lymax = 4. 
Ly = Lymax-Lymin
x_c = 0. # x-Position of the centre of the cylindre
y_c = 0. # y-Position of the centre of the cylindre
r_c = 0.5 # radius
d = 2.*r_c
u_in = 1. 
rho_0 = 1.

omega_0 = 1.036 #Dimensionless frequency

geom = [Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c]


def xbc5(s):
    '''
    Compute cylinders border x coordinate as a function of curvilinear abscissa s \in [0,1]
    input : s (tf tensor, usually of shape [Nbc,1])
    return a tf tensor of the same shape as s
    '''
    return x_c + r_c*tf.cos(2*np.pi*s)
def ybc5(s):
    '''
    Compute cylinders border y coordinate as a function of curvilinear abscissa s \in [0,1]
    input : s (tf tensor, usually of shape [Nbc,1])
    return a tf tensor of the same shape as s
    '''
    return y_c + r_c*tf.sin(2*np.pi*s)

# =============================================================================
# Choix de discretisation
# =============================================================================

Nmodes = args.Nmodes

Nmes = args.Nmes # Number of measurement points in the domain in case of dense data
Nint = args.Nint # Number of points to penalize NS equations in \Omega_f
Nbc = 1000 # Number of points to sample on cylinders norder


multigrid = args.multigrid # If true, Adam optimiser will change of V_in sampling 
# every NgridTurn iterations between the Ngrid generated
Ngrid = args.Ngrid
NgridTurn = args.NgridTurn 

stdNoise = args.Noise # In case of artificially noised data, it defines the 
# standard deviation inputted in the Gaussian distribution

# List of frequencies associated with each mode shapes
# Note that it could be replaced with an arbitrary list of frequencies
# or even tf.Variables() that could be optimized during training
list_omega = np.asarray([k*omega_0 for k in range(Nmodes)]) 

# Structure of each Neural Network that approximate a mode shape
layers = [2,args.WidthLayer*Nmodes,args.WidthLayer*Nmodes,Nmodes]

# =============================================================================
# Training tracking variables
# =============================================================================

global it
global listeErrTimeSerie
global listeErrValidTimeSerie

it=0
listeErrTimeSerie = []
listeErrValidTimeSerie = []

plot_config = False

if args.Tmax==None:
    Tmax = None  #0.5*3600 #8h
else:
    Tmax = 3600*args.Tmax


# =============================================================================
# Placeholders declaration
# In TF<2, one can define placeholders and build an operation graph based on these.
# Values are provided only at the computation in a dictionary tf_dict when running
# session.run(TF quantity that depends on placeholders,feed_dict=TF dictionary containing placeholders values)
# =============================================================================

Nxpitot = 40 # Number of simulated pitot probe locations in the flow (4 sections of 10 points)
Ncyl = 30 # Number of points around the cylinder to simulated pressure probes
Ntimes = 201 # Number of timesteps in simulations data

# Placeholders for V_in (penalization of equations)
x_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_int = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Placeholders for general fitting data (especially dense data)
x_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
u_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
v_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
p_tf_mes = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Placeholder for simulated pitot probe
x_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
y_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
t_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
u_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
v_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1])
p_tf_mes_pitot = tf.compat.v1.placeholder(dtype=tf.float32,shape=[Ntimes*Nxpitot,1]) #  Not really used since only u and v are used at these locations for training


# Preparing desynchronisation of pitot probe. Especially Used if args.DesyncSparseData == True
Delta_phi_np_pitot = 0.*np.random.uniform(low=0.0,high=2*np.pi/omega_0, size=Nxpitot)

if args.DesyncSparseData:
    Delta_t_tf_pitot = tf.Variable(Delta_t_np_pitot,dtype=tf.float32,shape=[Nxpitot])
else:
    Delta_phi_tf_pitot = tf.constant(Delta_phi_np_pitot,dtype=tf.float32,shape=[Nxpitot])


t_tf_mes_pitot_unflatten = tf.reshape(t_tf_mes_pitot,[Ntimes,Nxpitot])
t_tf_mes_pitot_resync_unflatten = tf.convert_to_tensor([[ t_tf_mes_pitot_unflatten[t,k] - Delta_phi_tf_pitot[k] for k in range(Nxpitot)] for t in range(Ntimes)])
t_tf_mes_pitot_resync = tf.reshape(t_tf_mes_pitot_resync_unflatten,[Ntimes*Nxpitot,1]) 

# Cylindre data for simulated pressure probe
x_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
p_tf_mes_cyl = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Lighthill boundary-vorticity-flux enforcement grid (only fed when --BVF is
# set; declaring these unconditionally is harmless since an unused
# placeholder never needs feeding and doesn't affect any other tensor)
x_tf_bvf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
y_tf_bvf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
t_tf_bvf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
g_tf_bvf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])


# Border
s_tf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])
one_s_tf = tf.compat.v1.placeholder(dtype=tf.float32,shape=[None,1])

# Frequencies
w_tf = tf.constant(list_omega,dtype=tf.float32,shape=[Nmodes])


# =============================================================================
# Model construction
# =============================================================================

# Initialisation / trainable warm-start of weights and biases.
if args.RestoreModel is None:
    print('Initialising u/v/p networks from Xavier random weights.')
    w_u,b_u = nnf.initialize_NN(layers)
    w_v,b_v = nnf.initialize_NN(layers)
    w_p,b_p = nnf.initialize_NN(layers)
else:
    if not os.path.exists(args.RestoreModel):
        raise IOError('Restore checkpoint not found: '+args.RestoreModel)
    print('Warm-starting TRAINABLE u/v/p networks from:', args.RestoreModel)
    w_u,b_u,w_v,b_v,w_p,b_p = nnf.restore_NN(
        layers,args.RestoreModel,tf_as_constant=False)


# Known free-stream velocity, used as a prior near the inlet when
# --FreestreamBC is set (see NN_functions.f_freestream_weight/out_nn_modes_uv).
# None when the flag is off, so behaviour is unchanged by default.
freestream_target_u = u_in if args.FreestreamBC else None
freestream_target_v = 0. if args.FreestreamBC else None
# Damps the fluctuating velocity modes (k>=1) toward zero at the inlet when set
# (see NN_functions.out_nn_modes_uv). None of BVF/measurement/interior losses
# call out_nn_modes_uv/NN_time_uv directly - they all go through these four
# wrappers - so this one flag propagates everywhere automatically, including
# the end-of-run mode plots.
damp_fluct = bool(args.FluctuationInletBC)
# Hard-kills the k=0 (mean) mode's imaginary part in every u/v/p wrapper below
# (see NN_functions.out_nn_modes_uv/out_nn_modes_p) - only matters once --K0Loss
# actually reads the complex k=0 mode; a no-op change in behaviour otherwise.
kill_k0 = bool(args.K0Loss)
# Hard-enforces Karman-street mode parity in every u/v/p wrapper below (see
# NN_functions.out_nn_modes_uv/out_nn_modes_p) when --HardSym is set.
hard_sym = bool(args.HardSym)
# R9: closed-form street prior parameters for the trust ansatz (--TrustStreet).
# Plain python-float dict (baked into the TF graph as constants - the prior is
# frozen; only the correction networks train). None when the flag is off, so
# behaviour is unchanged by default. Threaded through every u/v/p wrapper
# below - like damp_fluct/hard_sym, no loss or plot code path bypasses these.
street_params = None
v1_radial_params = None
if args.TrustStreet or args.V1RadialTrust:
    _sp = np.load(args.StreetPrior)
    _loaded_street = {k: float(_sp[k]) for k in
                      ('Gamma', 'Uc', 'xf', 'r0', 'omega', 'phase',
                       'amp_scale', 'scale_p', 'ramp', 'delta')}
    if args.TrustStreet:
        street_params = _loaded_street
        print('R9 legacy TrustStreet prior:', street_params)
    if args.V1RadialTrust:
        v1_radial_params = _loaded_street
        print('R10 v1-only radial prior:', v1_radial_params)
    if abs(_loaded_street['omega'] - omega_0) > 0.02:
        print('WARNING: tap-fitted omega %.4f differs from omega_0=%.4f used '
              'by the modal ansatz' % (_loaded_street['omega'], omega_0))
trust_rho = float(args.TrustRho)
trust_cap = float(args.TrustCap)
v1_trust_rho = float(args.V1TrustRho)
v1_xstart = float(args.V1TrustXStart)
v1_xwidth = float(args.V1TrustXWidth)
v1_ymax = float(args.V1TrustYMax)
v1_ywidth = float(args.V1TrustYWidth)

def fluid_u(x,y):
    '''
    Compute mode shapes of u
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_uv(x,y,w_u,b_u,geom,freestream_target=freestream_target_u,damp_fluctuations=damp_fluct,kill_k0_imag=kill_k0,hard_sym=hard_sym,is_v=False,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap)

def fluid_u_t(x,y,t):
    '''
    Compute u at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_uv(x,y,t,w_u,b_u,geom,omega_0,freestream_target=freestream_target_u,damp_fluctuations=damp_fluct,kill_k0_imag=kill_k0,hard_sym=hard_sym,is_v=False,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap)

def fluid_v(x,y):
    '''
    Compute mode shapes of v
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_uv(x,y,w_v,b_v,geom,freestream_target=freestream_target_v,damp_fluctuations=damp_fluct,kill_k0_imag=kill_k0,hard_sym=hard_sym,is_v=True,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap,v1_radial_params=v1_radial_params,v1_trust_rho=v1_trust_rho,v1_xstart=v1_xstart,v1_xwidth=v1_xwidth,v1_ymax=v1_ymax,v1_ywidth=v1_ywidth)

def fluid_v_t(x,y,t):
    '''
    Compute v at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_uv(x,y,t,w_v,b_v,geom,omega_0,freestream_target=freestream_target_v,damp_fluctuations=damp_fluct,kill_k0_imag=kill_k0,hard_sym=hard_sym,is_v=True,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap,v1_radial_params=v1_radial_params,v1_trust_rho=v1_trust_rho,v1_xstart=v1_xstart,v1_xwidth=v1_xwidth,v1_ymax=v1_ymax,v1_ywidth=v1_ywidth)

def fluid_p(x,y):
    '''
    Compute mode shapes of p
    Input : x,y TF tensors of shape [Nint,1]
    Return TF tensor of shape [1,Nint,Nmodes] with complex values
    '''
    return nnf.out_nn_modes_p(x,y,w_p,b_p,kill_k0_imag=kill_k0,hard_sym=hard_sym,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap)

def fluid_p_t(x,y,t):
    '''
    Compute p at instant t and position x,y
    Input: x,y,t TF tensors of shape [Nint,1]
    Return TF tensor of shape [Nint,1] with real values
    '''
    return nnf.NN_time_p(x,y,t,w_p,b_p,omega_0,kill_k0_imag=kill_k0,hard_sym=hard_sym,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap)

# =============================================================================
# Forces on cylinder
# =============================================================================

def force_cylinder_flatten(t):
    '''
    t : tf.float32 tensor shape [Nt,1]  
    ----
    return
    fx_tf,fy_tf :  tf.float32 tensor of shape [Nt,] containing averaged horizontal force on cylinder at time t
    '''
    Nt = int(t.shape[0])
    Ns = 1000 # Number of points to perform the integration over the border
    s_cyl = tf.constant(np.linspace(0.,1.,Ns), dtype = tf.float32, shape = [Ns,1])*tf.transpose(1+0.*t)
    # s_cyl = tf.random.uniform([Ns,1], minval=0., maxval = 1., dtype = tf.float32)*tf.transpose(1+0.*t)
    # Reshaping Space x Times on a same dimension
    s_cyl_r = tf.reshape(s_cyl,[Nt*Ns,1])
    x_cyl_r = tf.reshape(xbc5(s_cyl_r),[Nt*Ns,1]) 
    y_cyl_r = tf.reshape(ybc5(s_cyl_r),[Nt*Ns,1])
    t_cyl = (1.+0*s_cyl)*tf.transpose(t)
    t_cyl_r = tf.reshape(t_cyl,[Nt*Ns,1])

    # Computing fluid values along the border
    u = fluid_u_t(x_cyl_r,y_cyl_r,t_cyl_r)
    v = fluid_v_t(x_cyl_r,y_cyl_r,t_cyl_r)
    p = fluid_p_t(x_cyl_r,y_cyl_r,t_cyl_r)

    # Computing differentiated quantities
    u_x = tf.gradients(u, x_cyl_r)[0]
    u_y = tf.gradients(u, y_cyl_r)[0]
    u_xx = tf.gradients(u_x, x_cyl_r)[0]
    u_yy = tf.gradients(u_y, y_cyl_r)[0]

    v_x = tf.gradients(v, x_cyl_r)[0]
    v_y = tf.gradients(v, y_cyl_r)[0]
    v_xx = tf.gradients(v_x, x_cyl_r)[0]
    v_yy = tf.gradients(v_y, y_cyl_r)[0]

    # Computing normal and tangent vectors
    nx_base = - tf.gradients(y_cyl_r, s_cyl_r)[0]
    ny_base = tf.gradients(x_cyl_r, s_cyl_r)[0]
    normalisation = tf.sqrt(tf.square(nx_base) + tf.square(ny_base))
    nx = nx_base/normalisation
    ny = ny_base/normalisation

    # Computing local forces elements
    fx_tf_local = -p*nx + 2.*(1./Re)*u_x*nx + (1./Re)*(u_y+v_x)*ny
    fy_tf_local = -p*ny + 2.*(1./Re)*v_y*ny + (1./Re)*(u_y+v_x)*nx

    # Reshape to [Ns,Nt]
    fx_tf_local_r2 = tf.reshape(fx_tf_local,[Ns,Nt])
    fy_tf_local_r2 = tf.reshape(fy_tf_local,[Ns,Nt])

    # Integrating along the border for every time step
    fx_tf = -2.*np.pi*r_c*tf.reduce_mean(fx_tf_local_r2,axis=0)
    fy_tf = -2.*np.pi*r_c*tf.reduce_mean(fy_tf_local_r2,axis=0)

    return fx_tf,fy_tf


# =============================================================================
# Definition of functions for loss
# =============================================================================

def loss_int_mode_per_k(x,y):
    '''
    Parameters
    ----------
    x,y : float 32 tensor [Nint,1]

    Returns
    -------
    Return a tf.float32 tensor of shape [1,Nint,Nmodes]: the per-mode combined
    squared residual (x-momentum + y-momentum + continuity), NOT reduced over
    the mode axis - unlike loss_int_mode (below), which sums this over k and
    is otherwise unchanged. Split out so a single mode's contribution (e.g.
    k=0, the mean-flow/Reynolds-stress balance - see --K0Loss) can be used as
    its own loss term without re-deriving or duplicating this computation.
    '''
    all_u = fluid_u(x,y)
    all_v = fluid_v(x,y)
    all_p = fluid_p(x,y)


    one = tf.transpose(0.*x + 1.)

    def customgrad(fgrad,xgrad):
        '''
        Input frgad,xgrad : tf.complex64 tensor of shape [1,Nint,N+1] and [1,Nint] resp.
        Return a tf.complex64 tensor df/dx of shape [1,Nint,N+1]
        (tf.gradients does not seem to work with complex values and with f being of order 3... But it is mainly the same thing here)
        '''
        fgrad_xgrad =  [tf.complex(tf.gradients(tf.real(fgrad[:,:,k]), xgrad, grad_ys = one)[0],tf.gradients(tf.imag(fgrad[:,:,k]), xgrad, grad_ys = one)[0]) for k in range(Nmodes)]
        return tf.transpose(tf.convert_to_tensor(fgrad_xgrad), perm=[2,1,0])

    all_u_x = customgrad(all_u,x)
    all_u_y = customgrad(all_u,y)

    all_v_x = customgrad(all_v,x)
    all_v_y = customgrad(all_v,y)

    all_p_x = customgrad(all_p,x)
    all_p_y = customgrad(all_p,y)

    all_u_xx = customgrad(all_u_x,x)
    all_u_yy = customgrad(all_u_y,y)

    all_v_xx = customgrad(all_v_x,x)
    all_v_yy = customgrad(all_v_y,y)


    # x axis momentum equation
    f_u = tf.transpose(tf.convert_to_tensor([tf.complex(0.,k*omega_0)*all_u[:,:,k] for k in range(Nmodes)]), perm=[1,2,0])
    f_u += all_p_x
    f_u += (-1./Re)*(all_u_xx + all_u_yy)

    f_u_4a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*all_u_x[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_u += tf.transpose(tf.convert_to_tensor(f_u_4a), perm = [1,2,0])

    f_u_4b = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*all_u_y[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_u += tf.transpose(tf.convert_to_tensor(f_u_4b), perm = [1,2,0])

    f_u_5a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*tf.conj(all_u_x[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5a[-1] = f_u_5a[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5a), perm=[1,2,0])

    f_u_5b = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_u[:,:,l-k])*all_u_x[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5b[-1] = f_u_5b[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5b), perm=[1,2,0])

    f_u_5c = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*tf.conj(all_u_y[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5c[-1] = f_u_5c[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5c), perm=[1,2,0])

    f_u_5d = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_v[:,:,l-k])*all_u_y[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_u_5d[-1] = f_u_5d[-2]*0.
    f_u += tf.transpose(tf.convert_to_tensor(f_u_5d), perm=[1,2,0])


    f_u_sq = nnf.square_norm(f_u)

    # y axis Momentum equation
    f_v = tf.transpose(tf.convert_to_tensor([tf.complex(0.,k*omega_0)*all_v[:,:,k] for k in range(Nmodes)]), perm=[1,2,0])
    f_v += all_p_y
    f_v += (-1./Re)*(all_v_xx + all_v_yy)

    f_v_4a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*all_v_x[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_v += tf.transpose(tf.convert_to_tensor(f_v_4a), perm = [1,2,0])

    f_v_4b = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*all_v_y[:,:,k-l] for l in range(k+1)]), axis = 0) for k in range(Nmodes)]
    f_v += tf.transpose(tf.convert_to_tensor(f_v_4b), perm = [1,2,0])

    f_v_5a = [tf.reduce_sum(tf.convert_to_tensor([all_u[:,:,l]*tf.conj(all_v_x[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5a[-1] = f_v_5a[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5a), perm=[1,2,0])

    f_v_5b = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_u[:,:,l-k])*all_v_x[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5b[-1] = f_v_5b[-2]*0.  #quand k=N, k+1 > N
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5b), perm=[1,2,0])

    f_v_5c = [tf.reduce_sum(tf.convert_to_tensor([all_v[:,:,l]*tf.conj(all_v_y[:,:,l-k]) for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5c[-1] = f_v_5c[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5c), perm=[1,2,0])

    f_v_5d = [tf.reduce_sum(tf.convert_to_tensor([tf.conj(all_v[:,:,l-k])*all_v_y[:,:,l] for l in range(k+1,Nmodes)]),axis=0) for k in range(Nmodes)]
    f_v_5d[-1] = f_v_5d[-2]*0.
    f_v += tf.transpose(tf.convert_to_tensor(f_v_5d), perm=[1,2,0])


    f_v_sq = nnf.square_norm(f_v)


    # Mass conservation equation
    div_u = all_u_x + all_v_y
    div_u_sq = nnf.square_norm(div_u)

    return div_u_sq + f_u_sq + f_v_sq


def loss_int_mode(x,y):
    '''
    Parameters
    ----------
    x,y : float 32 tensor [Nint,1]

    Returns
    -------
    Return a tf.float32 tensor of shape [Nint,1] computing squared errors on
    modal equations - bit-identical to the pre-refactor implementation, now
    just the mode-axis sum of loss_int_mode_per_k (see there).
    '''
    return tf.reduce_sum(loss_int_mode_per_k(x,y), axis=2)


# =============================================================================
# k=1 control-volume integral momentum balance (R5, --CV1Loss) - see
# "R5 measured best candidates plan.md". Integral (not pointwise) form of the
# k=1 momentum equation over six fixed wake boxes: for the true field,
# momentum in = momentum out, i.e. R_j ~ 0. A dead wake was measured (in the
# diagnostic loss study this plan is based on) to satisfy the *pointwise*
# k>=1 equations almost as well as the true field (they are nearly
# homogeneous in amplitude), but not this integral form - it stayed
# adversarially robust. Explicitly NOT implementing a k=2 version: the same
# study measured the k=2 CV balance at 0.47x sensitivity - the dead wake
# satisfies it *better* than the truth, i.e. it is actively harmful as a
# training signal. Do not add one.
# =============================================================================

def conv_mode_k(a, b, k, Nmodes_local):
    '''
    Complex convolution at harmonic k of two per-mode complex tensors a,b
    (each [1,Npts,Nmodes], the same truncated-Fourier convention used by
    loss_int_mode_per_k's f_u_4a/4b + 5a-5d terms - the direct sum over
    l in [0,k] plus the two conjugate sums over l in (k,Nmodes) that account
    for the negative-frequency image of a real signal's higher harmonics).
    No derivative and no extra factor: this is the raw (a*b)-product's k-th
    Fourier coefficient (the momentum-flux tensor for --CV1Loss), not the
    pointwise NS residual (which multiplies one factor by its gradient).
    Written generically in Nmodes_local rather than as a fixed-length formula:
    a literal k=1 flux written out for 4 harmonics (k=0..3) silently drops two
    of its terms whenever, as in every run so far including R5's own planned
    command, --Nmodes 3 means only k=0,1,2 exist (Nmodes is the network's
    actual output width, confirmed against ModalPINN_VortexShedding.py's own
    layer construction - not assumed).
    Returns a tf.complex64 tensor of shape [1,Npts].
    '''
    direct = tf.reduce_sum(tf.convert_to_tensor([a[:,:,l]*b[:,:,k-l] for l in range(k+1)]), axis=0)
    if k+1 < Nmodes_local:
        conj_a = tf.reduce_sum(tf.convert_to_tensor([a[:,:,l]*tf.conj(b[:,:,l-k]) for l in range(k+1,Nmodes_local)]), axis=0)
        conj_b = tf.reduce_sum(tf.convert_to_tensor([tf.conj(a[:,:,l-k])*b[:,:,l] for l in range(k+1,Nmodes_local)]), axis=0)
        return direct + conj_a + conj_b
    else:
        return direct

def conv_deriv_k(a, ad, k, Nmodes_local):
    '''
    TF twin of audit_r5_losses.py's conv_deriv_k_np - the derivative-
    convolution pattern used by loss_int_mode_per_k's f_u_4a/4b + 5a/5b
    terms (one raw factor a, one derivative factor ad), factored out here
    for reuse by k0_residual's targeted mode-0 computation (see there for
    why this needs to exist separately from loss_int_mode_per_k).
    Returns a tf.complex64 tensor of shape [1,Npts].
    '''
    direct = tf.reduce_sum(tf.convert_to_tensor([a[:,:,l]*ad[:,:,k-l] for l in range(k+1)]), axis=0)
    if k+1 < Nmodes_local:
        conj_a = tf.reduce_sum(tf.convert_to_tensor([a[:,:,l]*tf.conj(ad[:,:,l-k]) for l in range(k+1,Nmodes_local)]), axis=0)
        conj_b = tf.reduce_sum(tf.convert_to_tensor([tf.conj(a[:,:,l-k])*ad[:,:,l] for l in range(k+1,Nmodes_local)]), axis=0)
        return direct + conj_a + conj_b
    return direct

def k0_residual(x, y):
    '''
    Minimal, mode-0-targeted computation of the k=0 harmonic momentum
    residual - mathematically equivalent to loss_int_mode_per_k(x,y)[:,:,0],
    but deliberately NOT built by calling that function.

    loss_int_mode_per_k computes 2nd derivatives (customgrad applied twice)
    and p_x/p_y for EVERY mode (0,1,2), because the full all-modes residual
    needs that. The k=0 slice specifically only ever reads mode 0's 2nd
    derivatives and mode 0's pressure gradient (modes 1,2 only enter k=0's
    formula through their FIRST derivatives, via the Reynolds-stress
    convolution sums) - the other ~24 gradient ops loss_int_mode_per_k would
    build are pure waste for this purpose on a forward evaluation, and
    actively harmful once backpropagated: --K0Loss puts this inside the
    optimized Loss, so declare_LBFGS's tf.gradients(Loss, weights) call has
    to differentiate whatever graph this function builds a SECOND time.
    Sharing loss_int_mode_per_k's full ~60-gradient-op graph for that OOM-
    killed a gate smoke test at ~11.7GB RSS on a 12GB colab-cli session -
    confirmed (by testing with as few as 10 collocation points, no change in
    peak memory) to be a fixed graph-topology cost, not a data-volume one,
    so trimming Nmodes-per-array here (not point count) is what actually
    matters. See PROJECT_LOG.md's R5 entry for the full diagnosis.

    Returns a tf.float32 tensor of shape [1,Npts]: |f_u|^2+|f_v|^2+|div_u|^2 at k=0.
    '''
    all_u = fluid_u(x, y)
    all_v = fluid_v(x, y)
    all_p = fluid_p(x, y)

    one = tf.transpose(0.*x + 1.)

    def cgrad_all(fgrad, xgrad):
        parts = [tf.complex(tf.gradients(tf.real(fgrad[:,:,k]), xgrad, grad_ys=one)[0],tf.gradients(tf.imag(fgrad[:,:,k]), xgrad, grad_ys=one)[0]) for k in range(Nmodes)]
        return tf.transpose(tf.convert_to_tensor(parts), perm=[2,1,0])

    def cgrad_mode0(fgrad_mode0, xgrad):
        return tf.complex(tf.gradients(tf.real(fgrad_mode0), xgrad, grad_ys=one)[0], tf.gradients(tf.imag(fgrad_mode0), xgrad, grad_ys=one)[0])

    # 1st derivatives - genuinely needed for all Nmodes (the convolution
    # sums read u_x/u_y/v_x/v_y at every mode, not just mode 0).
    all_u_x = cgrad_all(all_u, x)
    all_u_y = cgrad_all(all_u, y)
    all_v_x = cgrad_all(all_v, x)
    all_v_y = cgrad_all(all_v, y)

    # Mode-0-only: pressure gradient and 2nd derivatives.
    p0_x = cgrad_mode0(all_p[:,:,0], x)
    p0_y = cgrad_mode0(all_p[:,:,0], y)
    u0_xx = cgrad_mode0(all_u_x[:,:,0], x)
    u0_yy = cgrad_mode0(all_u_y[:,:,0], y)
    v0_xx = cgrad_mode0(all_v_x[:,:,0], x)
    v0_yy = cgrad_mode0(all_v_y[:,:,0], y)

    k = 0
    f_u = tf.complex(0., k*omega_0)*all_u[:,:,k] + p0_x - (1./Re)*(u0_xx + u0_yy)
    f_u = f_u + conv_deriv_k(all_u, all_u_x, k, Nmodes)
    f_u = f_u + conv_deriv_k(all_v, all_u_y, k, Nmodes)

    f_v = tf.complex(0., k*omega_0)*all_v[:,:,k] + p0_y - (1./Re)*(v0_xx + v0_yy)
    f_v = f_v + conv_deriv_k(all_u, all_v_x, k, Nmodes)
    f_v = f_v + conv_deriv_k(all_v, all_v_y, k, Nmodes)

    div_u = all_u_x[:,:,k] + all_v_y[:,:,k]

    return nnf.square_norm(f_u) + nnf.square_norm(f_v) + nnf.square_norm(div_u)

def grad_mode1(fgrad,xgrad):
    '''
    1st derivative of JUST the k=1 mode slice w.r.t. xgrad - used by the k=1
    control-volume loss (--CV1Loss), which only ever needs mode 1's
    derivative on box faces. Deliberately NOT a loop over all Nmodes like
    loss_int_mode_per_k's internal customgrad (which needs every mode):
    an earlier version reused that all-modes pattern here, computing and
    discarding gradients for modes 0 and 2 at every one of the 4
    derivatives (u1_x,u1_y,v1_x,v1_y) x 4 faces x 4 boxes - 3x more
    tf.gradients calls than necessary, which OOM-killed a gate smoke test
    (see PROJECT_LOG.md, R5 entry) on a 12GB colab-cli CPU session before
    training even started. Fixed by only ever differentiating mode 1.
    Input fgrad,xgrad : tf.complex64 [1,Npts,Nmodes] and tf.float32 [Npts,1] resp.
    Return tf.complex64 tensor d(mode 1)/dx of shape [1,Npts]
    '''
    one = tf.transpose(0.*xgrad + 1.)
    f1 = fgrad[:,:,1]
    return tf.complex(tf.gradients(tf.real(f1), xgrad, grad_ys=one)[0], tf.gradients(tf.imag(f1), xgrad, grad_ys=one)[0])

# Four fixed boxes, |y|<=2, clear of the cylinder (r_c=0.5) and strictly
# inside the domain (Lxmin=-4,Lxmax=8,Lymin=-4,Lymax=4): three paired
# (upstream x, downstream x) boxes sweeping into the near/mid wake, plus one
# full-wake box spanning all of them. The plan originally specified five
# paired boxes (x_up in {0.5,1,2,3,4}, x_down in {2,3,4,5,6}) plus the
# full-wake box (six total); Phase 0's R3-checkpoint audit
# (audit_r5_losses.py --Mode checkpoint) found the two furthest-downstream
# boxes (x in [3,5] and [4,6]) had INVERTED sensitivity - the dead R3 wake
# scored *better* on them than the true field (ratios 0.13x and 0.08x) - the
# same pathology the plan explicitly forbade for the k=2 harmonic (there
# measured at 0.47x). Dropped for the same reason: a box scoring the dead
# wake as more correct than the truth would train against wake revival in
# exactly the region R5 is trying to fix.
#
# Down to a single box (from the four survivors above) for an unrelated,
# later reason: --CV1Loss's memory cost turned out to be a FIXED graph-
# topology cost per box (confirmed by testing quadrature resolution from
# 64pts/face, 32x16 area down to 16pts/face, 8x8 area with ZERO change in
# peak memory - 10459MB vs 10406MB), not a data-volume cost, so cutting
# point density doesn't help but cutting box count does. Two boxes still
# OOM-killed a combined --K0Loss --CV1Loss smoke test at ~11.8GB on the
# 12GB colab-cli session (reached the multigrid/first-training-iteration
# step, which needs its own headroom on top of the ~10.9GB already used
# just building the graph and loading data) - down to one box for a real
# safety margin. Kept the full-wake box [0.5,6] (7.7x correct-sign
# discrimination in the R3-checkpoint audit) over the numerically stronger
# but narrower [0.5,2] box (16.5x): R5's primary acceptance criterion is
# far-wake E_v revival, and the full-wake box spans both near and far wake
# while [0.5,2] sits entirely in the near-cylinder region - see
# PROJECT_LOG.md for the full audit numbers and memory-debugging trace.
CV1_X_UP   = [0.5]
CV1_X_DOWN = [6. ]
CV1_YMIN, CV1_YMAX = -2., 2.
CV1_N_FACE_PTS = 64   # quadrature points per face (surface integrals) - resolution isn't the memory driver (see above), kept high for integration accuracy
CV1_N_AREA_X, CV1_N_AREA_Y = 32, 16  # quadrature grid (area/volume integral)

# Per-box normalizer for Loss_cv1 (see loss_cv1 below) - the full-wake
# box's R3-checkpoint |R_j|^2 from audit_r5_losses.py --Mode checkpoint, so
# it contributes ~O(1) to Loss_cv1 at the R3 checkpoint.
CV1_NORMALIZERS = [1.70640340664987e-2]

def _cv1_trapz_nodes_weights(a, b, n):
    '''Uniform trapezoid quadrature nodes+weights on [a,b], n points.'''
    nodes = np.linspace(a, b, n)
    w = np.full(n, (b - a) / (n - 1))
    w[0] *= 0.5
    w[-1] *= 0.5
    return nodes, w

def _build_cv1_boxes():
    '''Precompute, per box, the fixed numpy quadrature node coordinates +
    trapezoid weights for the 4 faces (surface integral) and the area grid
    (volume/storage integral). Pure numpy constants - the boxes do not move
    over training, so this only needs to run once at import time.'''
    boxes = []
    for x_up, x_down in zip(CV1_X_UP, CV1_X_DOWN):
        y_face, wy_face = _cv1_trapz_nodes_weights(CV1_YMIN, CV1_YMAX, CV1_N_FACE_PTS)
        x_face, wx_face = _cv1_trapz_nodes_weights(x_up, x_down, CV1_N_FACE_PTS)
        xa, wxa = _cv1_trapz_nodes_weights(x_up, x_down, CV1_N_AREA_X)
        ya, wya = _cv1_trapz_nodes_weights(CV1_YMIN, CV1_YMAX, CV1_N_AREA_Y)
        Xa, Ya = np.meshgrid(xa, ya, indexing='ij')
        Wa = np.outer(wxa, wya)
        boxes.append(dict(
            left_x=np.full(CV1_N_FACE_PTS, x_up, dtype=np.float32), left_y=y_face.astype(np.float32),
            left_w=wy_face.astype(np.float32), left_n=(-1., 0.),
            right_x=np.full(CV1_N_FACE_PTS, x_down, dtype=np.float32), right_y=y_face.astype(np.float32),
            right_w=wy_face.astype(np.float32), right_n=(1., 0.),
            bottom_x=x_face.astype(np.float32), bottom_y=np.full(CV1_N_FACE_PTS, CV1_YMIN, dtype=np.float32),
            bottom_w=wx_face.astype(np.float32), bottom_n=(0., -1.),
            top_x=x_face.astype(np.float32), top_y=np.full(CV1_N_FACE_PTS, CV1_YMAX, dtype=np.float32),
            top_w=wx_face.astype(np.float32), top_n=(0., 1.),
            area_x=Xa.flatten().astype(np.float32), area_y=Ya.flatten().astype(np.float32),
            area_w=Wa.flatten().astype(np.float32),
        ))
    return boxes

CV1_BOXES = _build_cv1_boxes()

def _cv1_all_quadrature_xy():
    '''All quadrature node (x,y) coordinates across all six boxes and all
    five sub-integrals (4 faces + area), concatenated and deduplicated by
    (x,y) pair. Used to append these exact points to the interior
    collocation set (Vin) so the ordinary pointwise physics loss also polices
    them every iteration - the anti-escape-hatch rule: every point the CV
    integral reads must also be physics-policed elsewhere, so the network
    cannot satisfy the box balance via a field that is only locally
    pathological exactly at the quadrature nodes.'''
    xs, ys = [], []
    for box in CV1_BOXES:
        for face in ['left', 'right', 'bottom', 'top']:
            xs.append(box[face + '_x']); ys.append(box[face + '_y'])
        xs.append(box['area_x']); ys.append(box['area_y'])
    x_all = np.concatenate(xs).astype(np.float32)
    y_all = np.concatenate(ys).astype(np.float32)
    xy = np.unique(np.stack([x_all, y_all], axis=1), axis=0)
    return xy[:, 0], xy[:, 1]

CV1_VIN_X, CV1_VIN_Y = _cv1_all_quadrature_xy()

def _cv1_box_residual(box):
    '''
    Complex 2-vector (Rx,Ry) k=1 control-volume momentum residual for one
    box (see module docstring above): the standard control-volume momentum
    balance, i*omega_0*integral(u1)dA (storage) + surface flux + pressure -
    viscous traction, matching sign conventions exactly against loss_int_time
    (pressure enters as +grad(p), viscous as -(1/Re)*Laplacian) and against
    force_cylinder_flatten's already-existing stress-tensor construction
    (fx_local = -p*nx + 2*(1/Re)*u_x*nx + (1/Re)*(u_y+v_x)*ny) for the
    viscous traction term, just evaluated on the k=1 complex mode instead of
    the real time-domain field.
    Returns (Rx, Ry) : two tf.complex64 scalar tensors.
    '''
    def col(a):
        return tf.constant(a.reshape(-1, 1), dtype=tf.float32)

    xa, ya = col(box['area_x']), col(box['area_y'])
    wa_c = tf.complex(tf.constant(box['area_w'], dtype=tf.float32), 0.)
    u1_a = fluid_u(xa, ya)[0, :, 1]
    v1_a = fluid_v(xa, ya)[0, :, 1]
    Rx = tf.complex(0., omega_0) * tf.reduce_sum(wa_c * u1_a)
    Ry = tf.complex(0., omega_0) * tf.reduce_sum(wa_c * v1_a)

    Re_c = tf.complex(Re, 0.)
    for face in ['left', 'right', 'bottom', 'top']:
        xf, yf = col(box[face + '_x']), col(box[face + '_y'])
        wf_c = tf.complex(tf.constant(box[face + '_w'], dtype=tf.float32), 0.)
        nx, ny = box[face + '_n']
        nx_c, ny_c = tf.complex(nx, 0.), tf.complex(ny, 0.)

        all_u = fluid_u(xf, yf)
        all_v = fluid_v(xf, yf)
        all_p = fluid_p(xf, yf)
        p1 = all_p[0, :, 1]
        Qxx = conv_mode_k(all_u, all_u, 1, Nmodes)[0, :]
        Qxy = conv_mode_k(all_u, all_v, 1, Nmodes)[0, :]
        Qyy = conv_mode_k(all_v, all_v, 1, Nmodes)[0, :]
        u1_x = grad_mode1(all_u, xf)[0, :]
        u1_y = grad_mode1(all_u, yf)[0, :]
        v1_x = grad_mode1(all_v, xf)[0, :]
        v1_y = grad_mode1(all_v, yf)[0, :]

        flux_x = Qxx*nx_c + Qxy*ny_c
        flux_y = Qxy*nx_c + Qyy*ny_c
        visc_x = (1./Re_c)*(2.*u1_x*nx_c + (u1_y+v1_x)*ny_c)
        visc_y = (1./Re_c)*((u1_y+v1_x)*nx_c + 2.*v1_y*ny_c)

        Rx = Rx + tf.reduce_sum(wf_c * (flux_x + p1*nx_c - visc_x))
        Ry = Ry + tf.reduce_sum(wf_c * (flux_y + p1*ny_c - visc_y))

    return Rx, Ry

def loss_cv1():
    '''
    Sum over all six k=1 control-volume boxes of |Rx|^2+|Ry|^2, each
    normalized by a fixed per-box constant (CV1_NORMALIZERS - see there;
    must be calibrated by audit_r5_losses.py before a real run) so no single
    box dominates purely from its size.
    Returns a tf.float32 scalar tensor.
    '''
    total = 0.
    for j, box in enumerate(CV1_BOXES):
        Rx, Ry = _cv1_box_residual(box)
        total = total + (nnf.square_norm(Rx) + nnf.square_norm(Ry)) / CV1_NORMALIZERS[j]
    return total


def loss_int_time(x,y,t):
    '''
    Parameters
    ----------
    x,y,t : tf.float 32 tensor [Nint,1]

    Returns
    -------
    Return [Nint,1] tensor containing squared error on NS equations
    '''
    u = fluid_u_t(x,y,t)
    v = fluid_v_t(x,y,t)
    p = fluid_p_t(x,y,t)

    u_t = tf.gradients(u,t)[0]
    v_t = tf.gradients(v,t)[0]

    u_x = tf.gradients(u, x)[0]
    u_y = tf.gradients(u, y)[0]
    u_xx = tf.gradients(u_x, x)[0]
    u_yy = tf.gradients(u_y, y)[0]

    v_x = tf.gradients(v, x)[0]
    v_y = tf.gradients(v, y)[0]
    v_xx = tf.gradients(v_x, x)[0]
    v_yy = tf.gradients(v_y, y)[0]

    p_x = tf.gradients(p, x)[0]
    p_y = tf.gradients(p, y)[0]

    f_u = u_t + (u*u_x + v*u_y) + p_x - (1./Re)*(u_xx + u_yy) 
    f_v = v_t + (u*v_x + v*v_y) + p_y - (1./Re)*(v_xx + v_yy)
    div_u = u_x + v_y

    return tf.square(f_u)+tf.square(f_v)+tf.square(div_u)


def loss_mes(xmes,ymes,tmes,umes,vmes,pmes):
    '''
    xmes,ymes,tmes,umes,vmes,pmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements 
    '''
    u_DNN = fluid_u_t(xmes,ymes,tmes)
    v_DNN = fluid_v_t(xmes,ymes,tmes)
    p_DNN = fluid_p_t(xmes,ymes,tmes)

    return tf.square(u_DNN-umes) + tf.square(v_DNN-vmes) + tf.square(p_DNN-pmes)

def loss_mes_uv(xmes,ymes,tmes,umes,vmes):
    '''
    xmes,ymes,tmes,umes,vmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements of velocity
    '''
    u_DNN = fluid_u_t(xmes,ymes,tmes)
    v_DNN = fluid_v_t(xmes,ymes,tmes)

    return tf.square(u_DNN-umes) + tf.square(v_DNN-vmes)

def loss_mes_p(xmes,ymes,tmes,pmes):
    '''
    xmes,ymes,tmes,pmes : [Nmes,1] tf.float32 tensor
    Return [Nmes,1] tf.float32 tensor containing square difference to measurements of pressure
    '''
    p_DNN = fluid_p_t(xmes,ymes,tmes)

    return tf.square(p_DNN-pmes)


def loss_bvf(x,y,t,g,residual_clip=50.):
    '''
    Lighthill wall relation (see bvf.md): at a stationary no-slip wall, all
    nonlinear/unsteady terms in the momentum equation vanish exactly, leaving
    (1/Re)*d(omega)/dn = (1/R)*dp/dtheta, omega = v_x - u_y, n = (x,y)/R
    outward. x,y must lie exactly on r = R = r_c (bvf_targets.py's analytic
    wall grid, not the ~0.4999 mesh nodes).

    Needs a third derivative through the network (omega is already a first
    derivative of u,v; w_x,w_y are second), which can occasionally produce
    very large - not NaN, just numerically extreme - values at a handful of
    points once weights move away from their small random init (observed on
    a real full-scale GPU run: a single L-BFGS step landed on a residual
    large enough to spike the loss to ~1e6 and abnormally terminate the line
    search - not reproduced in smaller-scale CPU checks, consistent with it
    being a rare event that needs many collocation points to hit). The
    residual is clipped before squaring so one such point can't dominate the
    mean or blow up the gradient - it still contributes its full unclipped
    gradient anywhere within a generous +-50 band (Phase 0 validation showed
    the true LHS/RHS are both O(1) in magnitude), only saturating for
    genuine outliers.
    Input x,y,t,g : [Nbvf,1] tf.float32 tensor
    Return [Nbvf,1] tf.float32 tensor of squared residuals
    '''
    u = fluid_u_t(x,y,t)
    v = fluid_v_t(x,y,t)
    w = tf.gradients(v,x)[0] - tf.gradients(u,y)[0]
    w_x = tf.gradients(w,x)[0]
    w_y = tf.gradients(w,y)[0]
    dwdn = (x*w_x + y*w_y) / r_c
    residual = (1./Re)*dwdn - g
    residual = tf.clip_by_value(residual, -residual_clip, residual_clip)
    return tf.square(residual)


def loss_BC(s):
    '''
    Return error on u=v=0 on cylinder border for each mode
    Input s : [Nbc,1] tf.float32 tensor of coordinates \in [0,1]
    Output : [] tf.float32 real positive number
    '''    
    x = xbc5(s)
    y = ybc5(s)
    u_k = fluid_u(x,y)
    v_k = fluid_v(x,y)

    err = tf.convert_to_tensor([nnf.square_norm(u_k[0,:,k]) + nnf.square_norm(v_k[0,:,k]) for k in range(Nmodes)])

    return tf.reduce_sum(tf.reduce_mean(err,axis=1))


# =============================================================================
# Causal weighting of the physics residual (R4, see "R4 fluctuation..." plan
# note: this addresses the PINN "causality violation" pathology (Wang et al.
# 2022) - distant collocation points can score near-zero residual for a
# near-zero (wrong) field just as easily as for a correct one, so nothing
# forces the network to solve the harder far-field/wake problem before it's
# "cheap" not to. x_front sweeps downstream over training so credit expands
# outward - same NS residual (loss_int_time, unchanged), only reweighted.
# =============================================================================

# x_front is scheduled externally (see causal_step_hook below), not learned -
# a plain tf.Variable(trainable=False) rather than a placeholder, so it never
# needs to be threaded through the multigrid tf_dict list: TF reads a
# variable's current graph-stored value on every sess.run automatically,
# and trainable=False already excludes it from tf.trainable_variables(), so
# neither declare_LBFGS's ScipyOptimizerInterface nor declare_Adam (both of
# which default to optimizing tf.trainable_variables()) will try to learn it.
x_front_var = tf.Variable(args.CausalStartX, dtype=tf.float32, trainable=False, name='x_front')
x_front_new_ph = tf.compat.v1.placeholder(dtype=tf.float32, shape=[])
x_front_assign_op = tf.compat.v1.assign(x_front_var, x_front_new_ph)

def causal_weight_fn(x, x_front, steepness):
    '''
    1 near/upstream of the frontier, smoothly -> 0 well past it.
    Input x : [Nint,1] tf.float32 tensor. x_front : scalar tf.Variable.
    Output : [Nint,1] tf.float32 tensor in (0,1)
    '''
    return tf.sigmoid(-(x - x_front) * steepness)

def causal_step_hook(it):
    '''Advance x_front linearly in iteration count from CausalStartX to
    CausalEndX over CausalWarmupIters, then hold at CausalEndX (full-domain
    weighting) for the remainder of training. No-op unless --CausalWeighting.
    Prints x_front's progress on its own line every 100 iterations (not
    appended to the "Loss: %.3e" line, which several scripts in this project
    parse via regex on that exact format) - for a loss-vs-iteration figure
    annotated with frontier position.'''
    if not args.CausalWeighting:
        return
    frac = min(1.0, it / max(1, args.CausalWarmupIters))
    new_x_front = args.CausalStartX + frac * (args.CausalEndX - args.CausalStartX)
    sess.run(x_front_assign_op, feed_dict={x_front_new_ph: new_x_front})
    if it % 100 == 0:
        print('Causal x_front @ it %d : %.4f' % (it, new_x_front))

# =============================================================================
# Training loss creation
# =============================================================================

# Wrap error on modal equations - the existing all-modes diagnostic, unchanged
# (Nint=50000 points, forward-evaluated once at the end for the "Loss eqs.
# modes" print; never part of the optimized Loss unless --LossModes, which no
# run uses, so this never gets differentiated a second time).
Loss_int_mode_wrap = tf.reduce_mean(loss_int_mode(x_tf_int, y_tf_int))

# k=0 harmonic residual (R5's --K0Loss - see the CLI arg docstring): the
# mean-flow/Reynolds-stress balance. A dead wake cannot satisfy this - it
# needs the quadratic Reynolds-stress divergence that only a live oscillating
# (k>=1) wake supplies. Uses k0_residual (see there), NOT
# loss_int_mode_per_k(x,y)[:,:,0] - mathematically the same k=0 formula, but
# built without the ~24 wasted 2nd-derivative/pressure-gradient ops
# loss_int_mode_per_k computes for modes 1,2 (never read by k=0's formula).
# That waste is harmless for a forward-only evaluation (e.g. the diagnostic
# above), but --K0Loss puts this INSIDE the optimized Loss, so
# declare_LBFGS's tf.gradients(Loss, weights) call has to differentiate
# whatever graph this builds a SECOND time - confirmed (by testing with as
# few as 10 collocation points, no change in peak memory) to be a fixed
# graph-topology cost, not a data-volume one, so K0_N_POINTS below is kept
# small mainly for per-iteration runtime, not as the OOM fix - see
# PROJECT_LOG.md's R5 entry for the full diagnosis (this OOM-killed a gate
# smoke test at ~11.7GB RSS on a 12GB colab-cli session before this fix).
K0_N_POINTS = 2000
_k0_rng = np.random.RandomState(42)
def _build_k0_points():
    x = _k0_rng.uniform(Lxmin + 0.5, Lxmax - 0.5, K0_N_POINTS * 2).astype(np.float32)
    y = _k0_rng.uniform(Lymin + 0.5, Lymax - 0.5, K0_N_POINTS * 2).astype(np.float32)
    r = np.sqrt((x - x_c) ** 2 + (y - y_c) ** 2)
    keep = r > 1.5 * r_c
    return x[keep][:K0_N_POINTS].reshape(-1, 1), y[keep][:K0_N_POINTS].reshape(-1, 1)
K0_X, K0_Y = _build_k0_points()
x_tf_k0 = tf.constant(K0_X, dtype=tf.float32)
y_tf_k0 = tf.constant(K0_Y, dtype=tf.float32)
Loss_k0_wrap = tf.reduce_mean(k0_residual(x_tf_k0, y_tf_k0))

# Wrap error on physical equations - unweighted mean by default (unchanged
# behavior); when --CausalWeighting is set, a weighted mean that upweights
# points near/upstream of x_front and downweights points still ahead of it.
Loss_int_time_raw = loss_int_time(x_tf_int, y_tf_int, t_tf_int)
if args.CausalWeighting:
    causal_w = causal_weight_fn(x_tf_int, x_front_var, args.CausalSteepness)
    Loss_int_time_wrap = tf.reduce_sum(causal_w * Loss_int_time_raw) / (tf.reduce_sum(causal_w) + 1e-8)
else:
    Loss_int_time_wrap = tf.reduce_mean(Loss_int_time_raw)

# Wrap error on (u,v,p) measurements
Loss_dense_mes = tf.reduce_mean(loss_mes(x_tf_mes,y_tf_mes,t_tf_mes,u_tf_mes,v_tf_mes,p_tf_mes))

# Wrap error on (u,v) measurements at simulated pitot probes locations
Loss_mes_pitot = tf.reduce_mean(loss_mes_uv(x_tf_mes_pitot,y_tf_mes_pitot,t_tf_mes_pitot_resync,u_tf_mes_pitot,v_tf_mes_pitot))
Loss_mes_pitot_desync = tf.reduce_mean(loss_mes_uv(x_tf_mes_pitot,y_tf_mes_pitot,t_tf_mes_pitot,u_tf_mes_pitot,v_tf_mes_pitot))

# Wrap error on pressure measurement around cylindre border
Loss_mes_cyl = tf.reduce_mean(loss_mes_p(x_tf_mes_cyl,y_tf_mes_cyl,t_tf_mes_cyl,p_tf_mes_cyl))

# Wrap error on the Lighthill boundary-vorticity-flux identity (only built when --BVF is set)
if args.BVF:
    Loss_bvf_wrap = tf.reduce_mean(loss_bvf(x_tf_bvf,y_tf_bvf,t_tf_bvf,g_tf_bvf))

# Simulated experimental losses
if args.PressureOnly:
    # Pressure-only mode: cylinder-surface pressure taps only, pitot velocity dropped entirely
    Loss_mes_exp = Loss_mes_cyl
else:
    Loss_mes_exp = Loss_mes_pitot + Loss_mes_cyl

if args.SparseData:
    Loss_mes = Loss_mes_exp
else: # Dense measurements are used for training
    Loss_mes = Loss_dense_mes

if args.LossModes:
    Loss = Loss_int_mode_wrap + Loss_mes
else: #Physical equations are used instead of modal equations
    Loss = Loss_int_time_wrap + Loss_mes

if args.BVF:
    Loss = Loss + args.LambdaBVF * Loss_bvf_wrap

if args.K0Loss:
    Loss = Loss + args.LambdaK0 * Loss_k0_wrap

if args.CV1Loss:
    Loss_cv1_wrap = loss_cv1()
    Loss = Loss + args.LambdaCV1 * Loss_cv1_wrap

print_mem('after Loss assembly (forward graph built)')

# =============================================================================
# Optimizer configuration
# =============================================================================

opt_LBFGS = nnf.declare_LBFGS(Loss, maxit=args.LBFGSMaxit, maxfun=args.LBFGSMaxfun, ftol=args.LBFGSFtol)
print_mem('after declare_LBFGS (d(Loss)/d(weights) graph built)')

opt_Adam = nnf.declare_Adam(Loss, lr=1e-5)
print_mem('after declare_Adam')

sess = nnf.declare_init_session()
print_mem('after session init')

# R14 checkpoint machinery. The ScipyOptimizerInterface packs exactly the
# trainable variables in graph order. We create an explicit unpack/assign map
# now so accepted-step xk vectors can later be converted into standard DNN
# pickle checkpoints without rerunning optimization.
_r14_ckpt_targets=[]
if args.LBFGSCheckpointIters.strip():
    _r14_ckpt_targets=sorted(set(
        int(z.strip()) for z in args.LBFGSCheckpointIters.split(',')
        if z.strip()))
    if any(z <= 0 for z in _r14_ckpt_targets):
        raise ValueError('--LBFGSCheckpointIters must contain positive integers.')
    print('R14 accepted-step checkpoint targets:',_r14_ckpt_targets)

_r14_train_vars=tf.compat.v1.trainable_variables()
_r14_var_shapes=[v.shape.as_list() for v in _r14_train_vars]
_r14_var_sizes=[int(np.prod(sh)) for sh in _r14_var_shapes]
_r14_total_size=int(np.sum(_r14_var_sizes))
_r14_assign_ph=[tf.compat.v1.placeholder(tf.float32,shape=sh)
                for sh in _r14_var_shapes]
_r14_assign_ops=[tf.compat.v1.assign(v,p)
                 for v,p in zip(_r14_train_vars,_r14_assign_ph)]

def _r14_get_flat():
    vals=sess.run(_r14_train_vars)
    return np.concatenate([np.asarray(v).reshape(-1) for v in vals]).astype(np.float64)

def _r14_assign_flat(xflat):
    xflat=np.asarray(xflat,dtype=np.float64).reshape(-1)
    if xflat.size != _r14_total_size:
        raise ValueError('R14 flat checkpoint size mismatch: %d vs %d' %
                         (xflat.size,_r14_total_size))
    feed={}
    j=0
    for ph,sh,n in zip(_r14_assign_ph,_r14_var_shapes,_r14_var_sizes):
        feed[ph]=xflat[j:j+n].reshape(sh).astype(np.float32)
        j+=n
    sess.run(_r14_assign_ops,feed_dict=feed)


# =============================================================================
# GPU use before loading data
# =============================================================================
print('GPU use before loading data')
GPUtil.showUtilization()

# =============================================================================
# Data set preparation
# =============================================================================

if args.SparseData:
    # Let's load data only at locations defined for simulated measurements
    print('Loading Sparse Data')

    x_int,y_int,t_int,s_train,xmes_pitot,ymes_pitot,tmes_pitot,umes_pitot,vmes_pitot,pmes_pitot,xmes_cyl,ymes_cyl,tmes_cyl,umes_cyl,vmes_cyl,pmes_cyl,Delta_phi_np_pitot_applied = ltd.training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmax=1e2,data_selection = 'cylinder_pitot',desync=args.DesyncSparseData, multigrid=multigrid,Ngrid=Ngrid,stdNoise=stdNoise,method_int = IntSampling, n_taps=args.NTaps)
    Ncyl = len(xmes_cyl)
    Npitot = len(xmes_pitot)
    Tmin = 400.

    if args.CV1Loss:
        # Anti-escape-hatch (see "R5 measured best candidates plan.md"):
        # append the k=1 CV integral's own fixed quadrature nodes to the
        # interior collocation set, so the ordinary pointwise physics loss
        # also polices these exact points every iteration, not just the CV
        # loss's own dedicated evaluation of them. Random times, same scale
        # as the rest of t_int (Tintmax=1e2 above) - a physically valid
        # periodic solution's NS residual should hold at any time.
        n_cv1 = len(CV1_VIN_X)
        if multigrid:
            for k in range(Ngrid):
                x_int[k] = np.concatenate([np.asarray(x_int[k]).flatten(), CV1_VIN_X])
                y_int[k] = np.concatenate([np.asarray(y_int[k]).flatten(), CV1_VIN_Y])
                t_int[k] = np.concatenate([np.asarray(t_int[k]).flatten(), np.random.uniform(0., 1e2, size=n_cv1).astype(np.float32)])
        else:
            x_int = np.concatenate([np.asarray(x_int).flatten(), CV1_VIN_X])
            y_int = np.concatenate([np.asarray(y_int).flatten(), CV1_VIN_Y])
            t_int = np.concatenate([np.asarray(t_int).flatten(), np.random.uniform(0., 1e2, size=n_cv1).astype(np.float32)])
        print('CV1: appended %d quadrature nodes to the interior collocation set (Vin)' % n_cv1)

    if multigrid:
        tf_dict = []
        for k in range(Ngrid):
            tf_dict_temp = {x_tf_int : np.reshape(x_int[k],(-1,1)),
             y_tf_int : np.reshape(y_int[k],(-1,1)),
             t_tf_int : np.reshape(t_int[k],(-1,1)),
             s_tf : np.reshape(s_train,(Nbc,1)),
             x_tf_mes_cyl : np.reshape(xmes_cyl,(Ncyl,1)),
             y_tf_mes_cyl : np.reshape(ymes_cyl,(Ncyl,1)),
             p_tf_mes_cyl : np.reshape(pmes_cyl,(Ncyl,1)),
             t_tf_mes_cyl : np.reshape(tmes_cyl,(Ncyl,1)),
             x_tf_mes_pitot : np.reshape(xmes_pitot,(Npitot,1)),
             y_tf_mes_pitot : np.reshape(ymes_pitot,(Npitot,1)),
             u_tf_mes_pitot : np.reshape(umes_pitot,(Npitot,1)),
             v_tf_mes_pitot : np.reshape(vmes_pitot,(Npitot,1)),
             p_tf_mes_pitot : np.reshape(pmes_pitot,(Npitot,1)),
             t_tf_mes_pitot : np.reshape(tmes_pitot,(Npitot,1)),
             }
            tf_dict.append(tf_dict_temp)

    else:
        tf_dict = {x_tf_int : np.reshape(x_int,(-1,1)),
             y_tf_int : np.reshape(y_int,(-1,1)),
             t_tf_int : np.reshape(t_int,(-1,1)),
             s_tf : np.reshape(s_train,(Nbc,1)),
             x_tf_mes_cyl : np.reshape(xmes_cyl,(Ncyl,1)),
             y_tf_mes_cyl : np.reshape(ymes_cyl,(Ncyl,1)),
             p_tf_mes_cyl : np.reshape(pmes_cyl,(Ncyl,1)),
             t_tf_mes_cyl : np.reshape(tmes_cyl,(Ncyl,1)),
             x_tf_mes_pitot : np.reshape(xmes_pitot,(Npitot,1)),
             y_tf_mes_pitot : np.reshape(ymes_pitot,(Npitot,1)),
             u_tf_mes_pitot : np.reshape(umes_pitot,(Npitot,1)),
             v_tf_mes_pitot : np.reshape(vmes_pitot,(Npitot,1)),
             p_tf_mes_pitot : np.reshape(pmes_pitot,(Npitot,1)),
             t_tf_mes_pitot : np.reshape(tmes_pitot,(Npitot,1))
             }

else:
    print('Loading Dense Data')
    x_int,y_int,t_int,s_train,xmes,ymes,tmes,umes,vmes,pmes = ltd.training_dict(Nmes,Nint,Nbc,filename_data,geom,Tintmax=1e2,data_selection = 'all',desync=False, multigrid=multigrid,Ngrid=Ngrid,stdNoise=stdNoise,cut=True,method_int=IntSampling)
    Nmes = len(xmes)
    Tmin = 400.

    if multigrid:
        tf_dict = []
        for k in range(Ngrid):
            tf_dict_temp = {x_tf_int : np.reshape(x_int[k],(Nint,1)),
              y_tf_int : np.reshape(y_int[k],(Nint,1)),
              t_tf_int : np.reshape(t_int[k],(Nint,1)),
              s_tf : np.reshape(s_train,(Nbc,1)),
              x_tf_mes : np.reshape(xmes,(Nmes,1)),
              y_tf_mes : np.reshape(ymes,(Nmes,1)),
              p_tf_mes : np.reshape(pmes,(Nmes,1)),
              t_tf_mes : np.reshape(tmes,(Nmes,1)),
              u_tf_mes : np.reshape(umes,(Nmes,1)),
              v_tf_mes : np.reshape(vmes,(Nmes,1))
              }
            tf_dict.append(tf_dict_temp)

    else:      
        tf_dict = {x_tf_int : np.reshape(x_int,(Nint,1)),
              y_tf_int : np.reshape(y_int,(Nint,1)),
              t_tf_int : np.reshape(t_int,(Nint,1)),
              s_tf : np.reshape(s_train,(Nbc,1)),
              x_tf_mes : np.reshape(xmes,(Nmes,1)),
              y_tf_mes : np.reshape(ymes,(Nmes,1)),
              p_tf_mes : np.reshape(pmes,(Nmes,1)),
              t_tf_mes : np.reshape(tmes,(Nmes,1)),
              u_tf_mes : np.reshape(umes,(Nmes,1)),
              v_tf_mes : np.reshape(vmes,(Nmes,1))
              }

if args.BVF:
    # Feed the same fixed wall grid + target into every entry of tf_dict
    # (a plain dict in the non-multigrid case, a list of dicts otherwise),
    # since Loss now depends on these placeholders whenever --BVF is set.
    bvf_npz = np.load(args.BVFTargets)
    x_bvf_wall = bvf_npz['x_wall'].astype(np.float32)
    y_bvf_wall = bvf_npz['y_wall'].astype(np.float32)
    t_bvf_grid = bvf_npz['t_grid'].astype(np.float32)
    G_bvf = bvf_npz['G'].astype(np.float32)  # [Ntheta, Ntime]
    Ntheta_bvf = len(x_bvf_wall)
    Ntime_bvf = len(t_bvf_grid)
    X_bvf = np.tile(x_bvf_wall.reshape(-1,1), (1,Ntime_bvf)).reshape(-1,1)
    Y_bvf = np.tile(y_bvf_wall.reshape(-1,1), (1,Ntime_bvf)).reshape(-1,1)
    T_bvf = np.tile(t_bvf_grid.reshape(1,-1), (Ntheta_bvf,1)).reshape(-1,1)
    Gflat_bvf = G_bvf.reshape(-1,1)
    bvf_feed = {x_tf_bvf: X_bvf, y_tf_bvf: Y_bvf, t_tf_bvf: T_bvf, g_tf_bvf: Gflat_bvf}

    if isinstance(tf_dict, list):
        for tf_dict_k in tf_dict:
            tf_dict_k.update(bvf_feed)
    else:
        tf_dict.update(bvf_feed)
    print('BVF targets loaded from %s: %d wall points x %d times = %d enforcement points' %
          (args.BVFTargets, Ntheta_bvf, Ntime_bvf, Ntheta_bvf*Ntime_bvf))


# Validation data set loading
# We extract 10 times more points for both dense measurements and equation penalisation
print('Loading validation data set')

x_int_valid,y_int_valid,t_int_valid,s_train,xmes_valid,ymes_valid,tmes_valid,umes_valid,vmes_valid,pmes_valid = ltd.training_dict(10*Nmes,10*Nint,Nbc,filename_data,geom,Tintmax=1e2,cut=True,method_int='uniform')
Nmesvalid = len(xmes_valid)

tf_dict_valid = {x_tf_int : np.reshape(x_int_valid,(10*Nint,1)),
     y_tf_int : np.reshape(y_int_valid,(10*Nint,1)),
     t_tf_int : np.reshape(t_int_valid,(10*Nint,1)),
     s_tf : np.reshape(s_train,(Nbc,1)),
     x_tf_mes : np.reshape(xmes_valid,(Nmesvalid,1)),
     y_tf_mes : np.reshape(ymes_valid,(Nmesvalid,1)),
     u_tf_mes : np.reshape(umes_valid,(Nmesvalid,1)),
     v_tf_mes : np.reshape(vmes_valid,(Nmesvalid,1)),
     p_tf_mes : np.reshape(pmes_valid,(Nmesvalid,1)),
     t_tf_mes : np.reshape(tmes_valid,(Nmesvalid,1))}


# =============================================================================
# GPU use after loading data
# =============================================================================
print('GPU use after loading data')
GPUtil.showUtilization()
print_mem('after data loading')


# =============================================================================
# Training
# =============================================================================

nnf.print_bar()
t1 = time.time()
print('Start training after %d s'%(t1-t0))

print('Start L-BFGS-B training')

_r14_state={'accepted':0,'last_xk':None,'saved':[]}
_r14_flat_dir=None
_r14_initial_flat=None
if _r14_ckpt_targets:
    _r14_flat_dir=repertoire+'/lbfgs_flat_checkpoints'
    if not os.path.exists(_r14_flat_dir):
        os.makedirs(_r14_flat_dir)
    _r14_initial_flat=_r14_get_flat()
    np.save(_r14_flat_dir+'/iter_00000.npy',_r14_initial_flat)
    print('R15 saved accepted-step checkpoint iter=0')

    def _r14_accepted_step_callback(xk):
        _r14_state['accepted'] += 1
        _r14_state['last_xk'] = np.asarray(xk,dtype=np.float64).copy()
        ii=_r14_state['accepted']
        if ii in _r14_ckpt_targets:
            fn=_r14_flat_dir+'/iter_%05d.npy'%ii
            np.save(fn,_r14_state['last_xk'])
            _r14_state['saved'].append(ii)
            print('R15 CHECKPOINT accepted_iter=%d saved'%ii)
            sys.stdout.flush()
else:
    _r14_accepted_step_callback=None

List_it_loss_LBFGS,List_it_loss_valid_LBFGS = nnf.model_train_scipy(
    opt_LBFGS,sess,Loss,tf_dict[0],List_loss=True,
    tf_dict_valid=tf_dict_valid,loss_valid=Loss_dense_mes,
    step_hook=causal_step_hook,
    accepted_step_callback=_r14_accepted_step_callback)

t2 = time.time()
print('L-BFGS-B training ended after %d s'%(t2-t1))

# Convert accepted-step flat vectors into ordinary DNN pickle checkpoints.
# This happens AFTER optimization, so checkpointing does not insert expensive
# sess.run/save operations into the L-BFGS line search itself.
if _r14_ckpt_targets:
    _r14_final_flat=_r14_get_flat()
    _r14_final_iter=int(_r14_state['accepted'])
    print('R15 accepted L-BFGS iterations:',_r14_final_iter)

    if _r14_state['last_xk'] is not None:
        _den=max(float(np.linalg.norm(_r14_final_flat)),1e-30)
        _rel=float(np.linalg.norm(_r14_final_flat-_r14_state['last_xk'])/_den)
        print('R15 final session-flat vs last accepted xk relative difference = %.3e'%_rel)
        if _rel > 5.e-5:
            raise RuntimeError('R15 accepted-step packed-vector mapping check failed.')

    _final_flat_path=_r14_flat_dir+'/iter_%05d.npy'%_r14_final_iter
    if not os.path.exists(_final_flat_path):
        np.save(_final_flat_path,_r14_final_flat)
        print('R15 saved additional final checkpoint iter=%d'%_r14_final_iter)

    _ckpt_root=repertoire+'/lbfgs_checkpoints'
    if not os.path.exists(_ckpt_root):
        os.makedirs(_ckpt_root)

    _flat_files=glob.glob(_r14_flat_dir+'/iter_*.npy')
    _items=[]
    for _ff in _flat_files:
        _base=os.path.basename(_ff)
        _ii=int(_base.replace('iter_','').replace('.npy',''))
        _items.append((_ii,_ff))
    _items=sorted(_items,key=lambda z:z[0])

    _feed=tf_dict[0] if isinstance(tf_dict,list) else tf_dict
    _phys_tensor=Loss_int_mode_wrap if args.LossModes else Loss_int_time_wrap
    _manifest={
        'accepted_iterations_final':_r14_final_iter,
        'requested_iterations':_r14_ckpt_targets,
        'packed_variable_count':len(_r14_train_vars),
        'packed_parameter_count':_r14_total_size,
        'checkpoints':[]
    }
    _dnn_name='DNN'+'_'.join([str(j) for j in layers])+'_tanh.pickle'

    for _ii,_ff in _items:
        _x=np.load(_ff)
        _r14_assign_flat(_x)
        _tot,_phy,_tap=sess.run([Loss,_phys_tensor,Loss_mes],feed_dict=_feed)
        _data=sess.run([w_u,b_u,w_v,b_v,w_p,b_p])
        _dir=_ckpt_root+'/iter_%05d'%_ii
        if not os.path.exists(_dir):
            os.makedirs(_dir)
        with open(_dir+'/'+_dnn_name,'wb') as _pf:
            pickle.dump(_data,_pf)
        _entry={
            'accepted_iter':int(_ii),
            'is_final':bool(_ii==_r14_final_iter),
            'model_dir':_dir,
            'model_file':_dir+'/'+_dnn_name,
            'training_total_loss':float(_tot),
            'training_physics_loss':float(_phy),
            'training_pressure_tap_loss':float(_tap),
        }
        _manifest['checkpoints'].append(_entry)
        print('R15 converted checkpoint iter=%d total=%.6e physics=%.6e taps=%.6e'%
              (_ii,_tot,_phy,_tap))

    # Restore the true optimizer endpoint before the normal safety-save path.
    _r14_assign_flat(_r14_final_flat)
    with open(repertoire+'/lbfgs_checkpoint_manifest.json','w') as _mf:
        import json as _json
        _json.dump(_manifest,_mf,indent=2)
    print('R15 checkpoint manifest saved:',
          repertoire+'/lbfgs_checkpoint_manifest.json')

if args.SkipAdam:
    print('Skipping Adam training (--SkipAdam)')
    List_it_loss_Adam = []
    List_it_loss_valid_Adam = []
    t3 = t2
else:
    print('Start Adam training')
    # Here Adam training is stopped if it reaches a time limit AdamTmax, or number of iterations Nit or if training loss goes under tolAdam
    AdamTmax = Tmax-(t2-t0)
    List_it_loss_Adam,List_it_loss_valid_Adam = nnf.model_train_Adam(opt_Adam,sess,Loss,liste_tf_dict=tf_dict,Nit=1e5,tolAdam=1e-5,it=it,itdisp=100,maxTime=AdamTmax,multigrid=multigrid,NgridTurn=NgridTurn,List_loss = True,tf_dict_valid=tf_dict_valid,loss_valid = Loss_dense_mes,step_hook=causal_step_hook)
    t3 = time.time()
    print('Adam training ended after %d s'%(t3-t2))

# =============================================================================
# GPU use after training
# =============================================================================
print('GPU use after training')
GPUtil.showUtilization()
print('End of training')

# =============================================================================
# R9: Save NN Model coefficients FIRST, before any post-training diagnostics.
# The R9 smoke test caught the modal-equation diagnostic below hanging for
# >10 min on its first evaluation (TF 1.14 builds/optimizes that large
# residual graph - now larger still with the street ops threaded in - on
# first sess.run). Saving used to happen ~70 lines further down, i.e. a hang
# or kill in the diagnostics would lose a full 9h run's weights. The save at
# the original location below is kept (it appends to the same file; loaders
# read the first pickle record) so downstream tooling is unchanged.
# =============================================================================

# R11: save a compact final loss breakdown BEFORE the clean exit.
# With pressure-only and no auxiliary loss terms:
#   total loss = physics loss + pressure-tap loss.
_eval_dict = tf_dict[0] if isinstance(tf_dict, list) else tf_dict
_physics_tensor = Loss_int_mode_wrap if args.LossModes else Loss_int_time_wrap
_r11_total, _r11_phys, _r11_taps = sess.run(
    [Loss, _physics_tensor, Loss_mes], feed_dict=_eval_dict)
_r11_summary = {
    'total_loss': float(_r11_total),
    'physics_loss': float(_r11_phys),
    'pressure_tap_loss': float(_r11_taps),
    'warm_started': bool(args.RestoreModel is not None),
    'restore_model': args.RestoreModel,
    'Nint': int(Nint),
    'Nmes': int(Nmes),
    'LBFGS_maxit': int(args.LBFGSMaxit),
    'LBFGS_maxfun': int(args.LBFGSMaxfun),
    'adam_skipped': bool(args.SkipAdam),
    'int_sampling': IntSampling,
}
with open(repertoire+'/training_loss_summary.json','w') as _fj:
    import json as _json
    _json.dump(_r11_summary,_fj,indent=2)
print('R11 FINAL LOSS SUMMARY:', _r11_summary)

print('Saving NN Model (pre-diagnostics safety save)...')
str_layers_fluid = [str(j) for j in layers]
filename_fluid = repertoire + '/DNN' + '_'.join(str_layers_fluid) + '_tanh.pickle'
Data_fluid = sess.run([w_u,b_u,w_v,b_v,w_p,b_p])
pcklfile_fluide = open(filename_fluid,'ab+')
pickle.dump(Data_fluid,pcklfile_fluide)
pcklfile_fluide.close()
print('Model exported in '+repertoire+' (safety save done)')

if args.ExitAfterSafetySave:
    print('Clean smoke exit requested (--ExitAfterSafetySave).')
    print('Weights are safely saved; skipping all legacy post-training plotting/history code.')
    sys.stdout.flush()
    sys.stderr.flush()
    sys.exit(0)


# =============================================================================
# Print residuals errors and losses
# =============================================================================

nnf.print_bar()
print('Error details')
nnf.print_bar()

if not(multigrid):
    tf_dict = [tf_dict]

print('')
if args.SkipDiagnostics:
    print('Skipping post-training diagnostics (--SkipDiagnostics); only the tap loss:')
    nnf.tf_print('Loss mesures training',Loss_mes,sess,tf_dict[0])
else:
    nnf.tf_print('Border',loss_BC(s_tf),sess,tf_dict[0])
    nnf.tf_print('Loss eqs. modes',Loss_int_mode_wrap,sess,tf_dict[0])
    nnf.tf_print('Loss eqs. int time',Loss_int_time_wrap,sess,tf_dict[0])
    nnf.tf_print('Loss mesures training',Loss_mes,sess,tf_dict[0])
    nnf.tf_print('Loss mesures validation',Loss_dense_mes,sess,tf_dict_valid)
    if args.SparseData:
        nnf.tf_print('Loss mes pitot (component)',Loss_mes_pitot,sess,tf_dict[0])
        nnf.tf_print('Loss mes cyl (component)',Loss_mes_cyl,sess,tf_dict[0])

    if args.BVF:
        nnf.tf_print('Loss BVF (component)',Loss_bvf_wrap,sess,tf_dict[0])

    if args.K0Loss:
        nnf.tf_print('Loss k0 harmonic (component)',Loss_k0_wrap,sess,tf_dict[0])

    if args.CV1Loss:
        nnf.tf_print('Loss cv1 (component)',Loss_cv1_wrap,sess,tf_dict[0])

    if args.CausalWeighting:
        print('Causal x_front final value : %.4f (target was %.4f)' % (sess.run(x_front_var), args.CausalEndX))

if args.DesyncSparseData:

    def r_div_eucli(a,b):
        '''
        a,b real numbers
        return r with a = n*b + r, n (int) and -b/2 <= r < b/2
        '''
        rtemp = a%b
        return np.where(rtemp>0.5*b,rtemp-b,rtemp)


    print('Validation Resync')
    Delta_phi_tf_pitot_found_o = sess.run(Delta_phi_tf_pitot)
    err_rms_resync = np.sqrt(np.mean(np.square((r_div_eucli(Delta_phi_tf_pitot_found_o-Delta_phi_np_pitot_applied,2*np.pi/omega_0)))))
    err_rms_resync_normalized = err_rms_resync/np.sqrt(np.mean(np.square(Delta_phi_np_pitot_applied)))
    print('Err RMS Resynchro : %.3e'%(err_rms_resync))
    print('Err RMS Resynchro normalized : %.3e'%(err_rms_resync_normalized))

    # Plot répartition des  erreurs de resyncro
    xpitot = np.reshape(xmes_pitot,[Ntimes,Nxpitot])[0,:]
    ypitot = np.reshape(ymes_pitot,[Ntimes,Nxpitot])[0,:]
    err_resync_pitot = r_div_eucli(Delta_phi_tf_pitot_found_o-Delta_phi_np_pitot_applied,2*np.pi/omega_0)

    size_resync = np.log10(err_resync_pitot)

    plt.figure()
    plt.scatter(xpitot,ypitot,c=np.log10(err_resync_pitot),marker='o',s=1.+size_resync)
    plt.colorbar()
    plt.scatter(xmes_cyl,ymes_cyl,c='black',marker='.',s=1.)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.axis('equal')
    plt.xlim((Lxmin,Lxmax))
    plt.ylim((Lymin,Lymax))
    plt.title('Synchronisation error - log')
    plt.tight_layout()
    plt.savefig(repertoire+'/resync_err.png')
    plt.close()



# =============================================================================
# Save NN Model coefficients in a pickle archive
# =============================================================================

print('Saving NN Model...')

str_layers_fluid = [str(j) for j in layers]
filename_fluid = repertoire + '/DNN' + '_'.join(str_layers_fluid) + '_tanh.pickle'

Data_fluid = sess.run([w_u,b_u,w_v,b_v,w_p,b_p])
pcklfile_fluide = open(filename_fluid,'ab+')
pickle.dump(Data_fluid,pcklfile_fluide)
pcklfile_fluide.close()
print('Model exported in '+repertoire)

# =============================================================================
# Save convergence history
# =============================================================================

print('Saving convergence history...')

filename_hist = repertoire + '/Convergence_history.pickle'

Data_loss_history = [List_it_loss_LBFGS,List_it_loss_valid_LBFGS,List_it_loss_Adam,List_it_loss_valid_Adam]
pckl_hist = open(filename_hist,'ab+')
pickle.dump(Data_loss_history,pckl_hist)
pckl_hist.close()
print('History exported in '+repertoire)

plt.figure()
plt.scatter(np.array(List_it_loss_LBFGS)[:,0],np.array(List_it_loss_LBFGS)[:,1],label='LBFGS train',marker='.',s=1.,c='red')
# plt.scatter(np.array(List_it_loss_valid_LBFGS)[:,0],np.array(List_it_loss_valid_LBFGS)[:,1],label='LBFGS valid',marker='.',s=1.,c='pink')
# Validation loss does not seem to be accessible during L-BFGS-B training. It returns constant values
plt.scatter(np.array(List_it_loss_Adam)[:,0]+np.max(np.array(List_it_loss_LBFGS)[:,0]),np.array(List_it_loss_Adam)[:,1],label='Adam train',marker='.',s=1.,c='blue')
plt.scatter(np.array(List_it_loss_valid_Adam)[:,0]+np.max(np.array(List_it_loss_LBFGS)[:,0]),np.array(List_it_loss_valid_Adam)[:,1],label='Adam valid',marker='.',s=1.,c='green')
plt.xlabel('Iterations')
plt.ylabel('Error')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.savefig(repertoire+'/Convergence_history.png')
plt.close()



# =============================================================================
# Plot of modal shapes
# =============================================================================


for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_u(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='u Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/u_mode_'+str(k)+'.png')
    plt.close()



for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_v(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='v Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/v_mode_'+str(k)+'.png')
    plt.close()

for k in range(Nmodes):
    nnf.tf_plot_scatter_complex(x_tf_int[:,0],y_tf_int[:,0],fluid_p(x_tf_int,y_tf_int)[0,:,k],
                        sess,
                        title='p Mode '+str(k),
                        xlabel='$x$',ylabel='$y$',
                        tf_dict=tf_dict_valid)
    plt.savefig(repertoire+'/p_mode_'+str(k)+'.png')
    plt.close()


# =============================================================================
# Comparison at a given timestep between modalPINN and simulations data
# =============================================================================
inst = 16

Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = ltd.read_cut_simulation_data(filename_data,geom)

tf_dict_compare = {
    x_tf_mes : np.reshape(nodes_X[0,:],(len(nodes_X[0,:]),1)),
    y_tf_mes : np.reshape(nodes_Y[0,:],(len(nodes_Y[0,:]),1)),
    t_tf_mes : np.reshape(times[inst]*np.ones(len(nodes_X[0,:])),(len(nodes_Y[0,:]),1)),
    u_tf_mes : np.reshape(Us[inst,:],(len(nodes_X[0,:]),1))
    }

suptitle='u difference at t = '+'{0:.2f}'.format(times[inst])

nnf.tf_plot_compare_3plot(x_tf_mes,y_tf_mes,u_tf_mes,fluid_u_t(x_tf_mes,y_tf_mes,t_tf_mes),sess,xlabel='$x$',ylabel='$y$',title1='Exact',title2='ModalPINN',suptitle='',tf_dict=tf_dict_compare)
plt.savefig(repertoire+'/diff_u_t_'+'{0:.2f}'.format(times[inst])+'.png')

In [ ]:
%%writefile NN_functions.py
# -*- coding: utf-8 -*-
"""
This file contains functions specific to
        o neural networks (construction, initialisation),
        o optimisers (calling from scipy or tf interfaces, initialisation, training steps),
        o plots.
@author: Gaétan Raynaud
"""

# =============================================================================
# Libraries
# =============================================================================

import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()  # required: this codebase uses tf.compat.v1 placeholders
import matplotlib.pyplot as plt
import pickle
import time

# =============================================================================
# Matplotlib parameters
# =============================================================================

plt.rc('text', usetex=False)
plt.rc('font', family='serif')
plt.rc('font', size=18)
plt.rc('axes',titlesize=20)
plt.rc('legend',fontsize=18)
plt.rc('figure',titlesize=24)


# =============================================================================
# Functions for defining, restoring and initialising neural networks
# =============================================================================

# True --> z = w1*exp(i*w2)
# False --> z = w1 + i*w2
complex_value_exp = True  

def initialize_NN(layers,name_nn=''):        
    '''
    Initialize a complex neural network which structure is defined by layers
    Input layers : list of integers defining the width of each layer. 
                   The number of elements in layers defines the depth of the NN
    Return weights : list of matrices filled with tf.complex64 variables
           biases : list of vectors filled with tf.complex64 variables
    '''
    weights = []
    biases = []
    num_layers = len(layers) 
    for l in range(0,num_layers-1):
        W_1 = xavier_init(size=[layers[l], layers[l+1]],name_w = 'weights_'+name_nn+str(l))
        W_2 = xavier_init(size=[layers[l], layers[l+1]],name_w = 'weights_'+name_nn+str(l))

        b_1 = tf.Variable(tf.zeros([1,layers[l+1]], dtype=tf.float32), dtype=tf.float32, name ='biases_'+name_nn+str(l))
        b_2 = tf.Variable(tf.zeros([1,layers[l+1]], dtype=tf.float32), dtype=tf.float32, name ='biases_'+name_nn+str(l))

        if complex_value_exp:
            W = tf.complex(W_1, 0.)*tf.exp(tf.complex(0., W_2))
            b = tf.complex(b_1, 0.)*tf.exp(tf.complex(0., b_2))
        else :
            W = tf.complex(W_1, W_2)
            b = tf.complex(b_1, b_2)

        weights.append(W)
        biases.append(b)     

    return weights, biases

def restore_one_NN(layers,w_value,b_value,tf_as_constant=False):
    '''
    input
    layers : list of integers describing the structure of the NN
    w_value,b_value : list of values of the tensors coefficients
    tf_as_constant : bool (False) : if True, construct directly tf.complex64 coefficients as tf.constant
    If False, construct tf.Variables as tf.float32 and then join them according to complex_value_exp (bool) policy
    ----
    return
    weights and biases tensors variables initialised with the given values

    R11 compatibility note:
    Saved checkpoints are NumPy complex arrays. TensorFlow 1.14's
    tf.math.angle() does not reliably accept a NumPy complex dtype directly.
    Therefore, for trainable warm-starts we decompose the complex arrays with
    NumPy first and then create float32 TensorFlow Variables.
    '''
    weights = []
    biases = []
    num_layers = len(layers)
    for l in range(0,num_layers-1):
        if tf_as_constant:
            W = tf.constant(
                np.asarray(w_value[l],dtype=np.complex64),
                dtype=tf.complex64,shape=[layers[l],layers[l+1]])
            b = tf.constant(
                np.asarray(b_value[l],dtype=np.complex64),
                dtype=tf.complex64,shape=[1,layers[l+1]])

        else:
            if complex_value_exp:
                W_mag0 = np.abs(w_value[l]).astype(np.float32)
                W_ang0 = np.angle(w_value[l]).astype(np.float32)
                b_mag0 = np.abs(b_value[l]).astype(np.float32)
                b_ang0 = np.angle(b_value[l]).astype(np.float32)

                W_1 = tf.Variable(
                    W_mag0,dtype=tf.float32,
                    shape=[layers[l],layers[l+1]])
                W_2 = tf.Variable(
                    W_ang0,dtype=tf.float32,
                    shape=[layers[l],layers[l+1]])
                W = tf.complex(W_1,0.)*tf.exp(tf.complex(0.,W_2))

                b_1 = tf.Variable(
                    b_mag0,dtype=tf.float32,
                    shape=[1,layers[l+1]])
                b_2 = tf.Variable(
                    b_ang0,dtype=tf.float32,
                    shape=[1,layers[l+1]])
                b = tf.complex(b_1,0.)*tf.exp(tf.complex(0.,b_2))

            else:
                W_re0 = np.real(w_value[l]).astype(np.float32)
                W_im0 = np.imag(w_value[l]).astype(np.float32)
                b_re0 = np.real(b_value[l]).astype(np.float32)
                b_im0 = np.imag(b_value[l]).astype(np.float32)

                W_1 = tf.Variable(
                    W_re0,dtype=tf.float32,
                    shape=[layers[l],layers[l+1]])
                W_2 = tf.Variable(
                    W_im0,dtype=tf.float32,
                    shape=[layers[l],layers[l+1]])
                W = tf.complex(W_1,W_2)

                b_1 = tf.Variable(
                    b_re0,dtype=tf.float32,
                    shape=[1,layers[l+1]])
                b_2 = tf.Variable(
                    b_im0,dtype=tf.float32,
                    shape=[1,layers[l+1]])
                b = tf.complex(b_1,b_2)

        weights.append(W)
        biases.append(b)
    return weights,biases


def restore_NN(layers,filename_restore,tf_as_constant=False):

    '''
    Restore u, v and p ModalPINN models 
    Input layers : list of each layers width
          filename_restore (str) : location of the pickle archive where the values are stored
          tf_as_constant (bool) : if True, model's parameters are initialised as tf.constant 
                                  (and are therefore fixed). Else, they are set as 
                                  tf.variable and can be trained once again.
    Return the weights and biases 
    '''    

    file = open(filename_restore,'rb')
    w_u_value,b_u_value,w_v_value,b_v_value,w_p_value,b_p_value = pickle.load(file)
    file.close()

    w_u,b_u = restore_one_NN(layers,w_u_value,b_u_value,tf_as_constant)
    w_v,b_v = restore_one_NN(layers,w_v_value,b_v_value,tf_as_constant)
    w_p,b_p = restore_one_NN(layers,w_p_value,b_p_value,tf_as_constant)

    return w_u,b_u,w_v,b_v,w_p,b_p



def xavier_init(size,name_w):
    '''
    Initialisation of weights using xavier init.
    This function comes from Raissi et al. (2019)
    '''
    in_dim = size[0]
    out_dim = size[1]        
    xavier_stddev = np.sqrt(2/(in_dim + out_dim))
    return tf.Variable(tf.random.truncated_normal([in_dim, out_dim], stddev=xavier_stddev, dtype=tf.float32), dtype=tf.float32, name = name_w)


def neural_net(X, weights, biases):
    '''
    Construct one neural network as a succession of affine transformation and 
    non-linear functions (here sigma = tanh)
    Input : tf.tensor X, weights and biases that define the model
    Output: tf.tensor Y
    '''
    H = X
    num_layers = len(weights) + 1
    for l in range(0,num_layers-2):
        W = weights[l]
        b = biases[l]
        H = tf.tanh(tf.add(tf.matmul(H, W), b))
    W = weights[-1]
    b = biases[-1]
    Y = tf.add(tf.matmul(H, W), b)
    return Y


def f_BC5(x, y, geom, fact=5.):
    '''
    return real tensor of same size than x and y
    equal to zero on the cylinder border
    '''
    Lxmin,Lxmax,Lymin,Lymax,x_c,y_c,r_c = geom
    r = tf.sqrt(tf.square(x-x_c) + tf.square(y-y_c)) - r_c
    return tf.tanh(fact*r)

def f_freestream_weight(x, x_transition=-2., gamma=3.):
    '''
    Blending weight for an inlet free-stream prior: ~1 upstream of the
    cylinder (near the inlet, where the real flow genuinely is undisturbed
    free-stream), ~0 near/after the cylinder and downstream (where the
    real flow is still inside the wake and should NOT be forced toward
    free-stream). Only depends on x, not y or the cylinder radius, since
    the transition is a streamwise one, not a distance-from-cylinder one.
    Input x : [Nint,1] tf.float32 tensor
    Output : [Nint,1] tf.float32 tensor in (0,1)
    '''
    return 0.5 * (1. - tf.tanh(gamma * (x - x_transition)))

def street_modes_k(x, y, sp, k):
    '''
    R9: closed-form von Karman street mode k (k>=1) as TF ops - the analytic
    prior of the trust ansatz (see --TrustStreet in ModalPINN_VortexShedding
    and R9_wake_rescue/REPORT.md for derivation + validation).

    Two staggered rows of vortices expanded harmonically: for each row
    (lateral position y_row, sign s_row, streamwise origin x0),
        base_k = s_row*G/(2a) * e^{i(-2 pi k (x-x0)/a - k phase)}
                 * e^{-2 pi k softabs(y-y_row)/a} * e^{-(pi k)^2 rc2(x)/a^2}
        u_k += -tanh((y-y_row)/delta) * base_k     (shear mode, flips sign)
        v_k += i * base_k
    with viscous core growth rc2(x) = r0^2 + 4 nu relu(x-xf)/Uc and a
    formation-ramp envelope. Pressure anchor: p_k = -(1-Uc)*scale_p*u_k
    (linearized Bernoulli in the advected frame, amplitude-calibrated).

    All parameters in sp come from street_prior.py (taps-only). amp_scale
    calibrates the closed form to the numeric image-vortex street; the
    trailing factor 2 converts to this codebase's one-sided convention
    (NN_time_* reconstructs Re(sum q_k e^{ikwt}) without a factor 2).

    Input  x,y : [Nint,1] tf.float32 tensors
    Output u_k, v_k, p_k : [1,Nint] tf.complex64 tensors
    '''
    G, Uc, xf, r0 = sp['Gamma'], sp['Uc'], sp['xf'], sp['r0']
    omega, phase = sp['omega'], sp['phase']
    ramp, delta = sp['ramp'], sp['delta']
    amp = sp['amp_scale'] * 2.0          # closed-form calib x convention
    nu = 1. / 100.
    a = 2. * np.pi * Uc / omega
    h = 0.281 * a
    xs, ys = x[:, 0], y[:, 0]
    env = 0.5 * (1. + tf.tanh((xs - xf) / ramp))
    rc2 = r0 ** 2 + 4. * nu * tf.nn.relu(xs - xf) / Uc
    att = tf.exp(-(np.pi * k) ** 2 * rc2 / a ** 2)
    u_re = tf.zeros_like(xs); u_im = tf.zeros_like(xs)
    v_re = tf.zeros_like(xs); v_im = tf.zeros_like(xs)
    for y_row, s_row, x0 in ((+h / 2., -1., xf), (-h / 2., +1., xf + a / 2.)):
        yp = ys - y_row
        sabs = tf.sqrt(tf.square(yp) + delta ** 2) - delta
        sgn = -tf.tanh(yp / delta)
        Dk = tf.exp(-2. * np.pi * k * sabs / a)
        ph = -2. * np.pi * k * (xs - x0) / a - k * phase
        mag = s_row * G / (2. * a) * Dk * att
        b_re = mag * tf.cos(ph)
        b_im = mag * tf.sin(ph)
        u_re += sgn * b_re; u_im += sgn * b_im
        v_re += -b_im;      v_im += b_re          # i * base
    scale = amp * env
    u_k = tf.complex(scale * u_re, scale * u_im)
    v_k = tf.complex(scale * v_re, scale * v_im)
    p_fac = -(1. - Uc) * sp['scale_p']
    p_k = tf.complex(p_fac * scale * u_re, p_fac * scale * u_im)
    return (tf.expand_dims(u_k, 0), tf.expand_dims(v_k, 0),
            tf.expand_dims(p_k, 0))

def smootherstep01(z):
    # C2 smootherstep clamped to [0,1]:
    # 6 z^5 - 15 z^4 + 10 z^3.
    # First AND second derivatives are zero at both ends, which is important
    # because the NS loss differentiates velocity twice in space.
    zc = tf.clip_by_value(z, 0., 1.)
    return zc*zc*zc*(zc*(zc*6. - 15.) + 10.)


def out_nn_modes_uv(x,y,weights,biases,geom,freestream_target=None,damp_fluctuations=False,kill_k0_imag=False,hard_sym=False,is_v=False,street_params=None,trust_rho=0.6,trust_cap=0.12,v1_radial_params=None,v1_trust_rho=0.70,v1_xstart=3.0,v1_xwidth=0.30,v1_ymax=2.0,v1_ywidth=0.20):
    '''
    Return Nmode complex modes shapes of DNN defined with weights and biases
    Prior dictionary f_BC5 is applied so that each mode shape verifies =0 on cylinder's border
    If freestream_target is not None, the mean mode (k=0) is additionally blended
    toward that known constant near the inlet (see f_freestream_weight) - a second,
    independent prior alongside f_BC5, not a replacement for it.
    If damp_fluctuations is True, the fluctuating modes (k>=1) are damped toward
    zero at the inlet using the same ramp - physically, shedding is a wake
    phenomenon, and reality kills any upstream-travelling fluctuation branch via
    the inlet condition "no incoming fluctuations"; nothing previously encoded
    that, which let a spurious oscillation appear upstream (see
    "R3 fluctuation inlet bc plan.md"). Velocity only - pressure fluctuations
    physically do reach the inlet, so out_nn_modes_p is untouched.
    If kill_k0_imag is True, the k=0 (mean) mode is hard-projected onto its real
    part. The mean of a real signal is real, and NN_time_uv/NN_time_p only ever
    take tf.real(...) of the time reconstruction, so Im(mode_0) is otherwise an
    unconstrained null direction invisible to every pointwise/measurement loss -
    it drifted to +-4 in R2 (see v_mode_0.png). Harmless by default; only matters
    once a loss that actually reads the complex k=0 mode (R5's --K0Loss) exists,
    which is the only place this flag is set True (see "R5 measured best
    candidates plan.md").
    If hard_sym is True, each mode is reflection-symmetrized in y to hard-enforce
    the Karman-street parity: u_k,p_k are even in y (parity (-1)^k, is_v=False),
    v_k is odd/even the opposite way (parity (-1)^(k+1), is_v=True). Applied
    right after the f_BC5 mask (which is itself even in y since the cylinder is
    centered at y_c=0, so fbc5(x,-y)=fbc5(x,y) - no need for a second f_BC5
    evaluation) and before the freestream/damp_fluctuations/kill_k0_imag
    adjustments below, which only depend on x and so compose safely with it.
    Roughly doubles this function's cost (a second neural_net forward pass at
    the reflected point). See "R5 measured best candidates plan.md" - optional,
    go/no-go at the smoke test.
    Input x,y : [Nint,1] tf.float32 tensor
    Output shape : [1,Nint,Nmode] tf.complex64 tensor
    '''
    xint = tf.complex(x,0.)
    yint = tf.complex(y,0.)
    out_nn = neural_net(tf.transpose(tf.stack([xint,yint])),weights,biases)
    Nmode = int(out_nn[0,0,:].shape[0])
    fbc5c = tf.complex(f_BC5(x,y,geom)[:,0],0.)
    if hard_sym:
        yint_neg = tf.complex(-y,0.)
        out_nn_neg = neural_net(tf.transpose(tf.stack([xint,yint_neg])),weights,biases)
    w = None
    if freestream_target is not None or damp_fluctuations:
        w = tf.complex(f_freestream_weight(x)[:,0], 0.)
    modes = []
    for k in range(Nmode):
        if v1_radial_params is not None and is_v and k == 1:
            # R10 smoke candidate: radial trust ONLY on v_1, only downstream.
            #
            # Outside the trusted wake the mode stays the ordinary free ModalPINN
            # output. Inside, the network is a bounded COMPLEX radial correction:
            #
            #   v1 = S_v1 + rho*|S_v1| * z/sqrt(1+|z|^2)
            #
            # so |correction| < rho|S|. With rho<1, exact v1=0 cannot be
            # reached wherever the street prior is nonzero. This fixes the R9
            # componentwise-square geometry and avoids imposing the street on
            # u, p, k=2, or the near wake.
            _, S_v_k, _ = street_modes_k(x, y, v1_radial_params, k)
            corr_raw = out_nn[:,:,k]
            if hard_sym:
                parity = -1. if ((k % 2 == 0) == is_v) else 1.
                corr_raw = 0.5*(corr_raw + tf.complex(parity,0.)*out_nn_neg[:,:,k])

            corr_mag2 = tf.square(tf.real(corr_raw)) + tf.square(tf.imag(corr_raw))
            corr_den = tf.sqrt(1. + corr_mag2)
            corr_radial = corr_raw / tf.complex(corr_den, 0.*corr_den)

            # C2 gate with an EXACT trusted core:
            # x gate is free for x <= xstart-xwidth, transitions over
            # [xstart-xwidth, xstart], and is exactly 1 for x >= xstart.
            # y gate is exactly 1 for |y| <= ymax and transitions to zero
            # over [ymax, ymax+ywidth] on each side.
            tx = (x[:,0] - (v1_xstart - v1_xwidth))/v1_xwidth
            wx = smootherstep01(tx)
            wy_hi = 1. - smootherstep01(( y[:,0] - v1_ymax)/v1_ywidth)
            wy_lo = 1. - smootherstep01((-y[:,0] - v1_ymax)/v1_ywidth)
            W = wx*wy_hi*wy_lo
            Wc = tf.complex(W, 0.*W)

            # Smooth |S| surrogate, exactly zero at S=0 and slightly smaller
            # than |S| otherwise. This avoids a non-smooth complex abs inside
            # a field differentiated twice by the PDE loss.
            s_re = tf.real(S_v_k)
            s_im = tf.imag(S_v_k)
            mag_eps = tf.constant(1.e-6, dtype=tf.float32)
            S_mag = tf.sqrt(tf.square(s_re) + tf.square(s_im) +
                            tf.square(mag_eps)) - mag_eps
            A = v1_trust_rho*S_mag

            trusted_inner = S_v_k + tf.complex(A, 0.*A)*corr_radial
            free_inner = corr_raw
            mode_k = fbc5c*((1.-Wc)*free_inner + Wc*trusted_inner)
        elif street_params is not None and k >= 1:
            # R9 trust ansatz: q_k = fbc5*(S_k + (rho|S_k| + cap)*bounded_corr)
            # S_k = closed-form street mode (taps-only prior, see
            # street_modes_k). tanh-bounded correction => q_k = 0 is OUTSIDE
            # the search space wherever the street is alive (|S_k| > cap/rho)
            # - the dead-wake spurious minimum of R1-R8 is excluded by
            # construction. fbc5 keeps the hard no-slip prior. The k=0 mode
            # (else-branch below) stays a free network: the street has no
            # boundary layer, the mean flow is the network's job.
            S_u_k, S_v_k, S_p_k = street_modes_k(x, y, street_params, k)
            S_k = S_v_k if is_v else S_u_k
            corr_raw = out_nn[:,:,k]
            if hard_sym:
                parity = -1. if ((k % 2 == 0) == is_v) else 1.
                corr_raw = 0.5*(corr_raw + tf.complex(parity,0.)*out_nn_neg[:,:,k])
            corr = tf.complex(tf.tanh(tf.real(corr_raw)), tf.tanh(tf.imag(corr_raw)))
            A = trust_rho*tf.abs(S_k) + trust_cap
            mode_k = fbc5c*(S_k + tf.complex(A, 0.*A)*corr)
        else:
            mode_k = fbc5c*out_nn[:,:,k]
            if hard_sym:
                parity = -1. if ((k % 2 == 0) == is_v) else 1.  # is_v: (-1)^(k+1); else (-1)^k
                mode_k = 0.5*(mode_k + tf.complex(parity,0.)*fbc5c*out_nn_neg[:,:,k])
        if k == 0 and freestream_target is not None:
            mode_k = w*tf.complex(freestream_target, 0.) + (1.-w)*mode_k
        elif k >= 1 and damp_fluctuations:
            mode_k = (1.-w)*mode_k          # fluctuations -> 0 at the inlet
        if k == 0 and kill_k0_imag:
            mode_k = tf.complex(tf.real(mode_k), 0.*tf.real(mode_k))
        modes.append(mode_k)
    t_parts = tf.convert_to_tensor(modes)
    return tf.transpose(t_parts,perm=[1,2,0])

def out_nn_modes_p(x,y,weights,biases,kill_k0_imag=False,hard_sym=False,street_params=None,trust_rho=0.6,trust_cap=0.12):
    '''
    Return Nm complex modes of dnn defined with weights and biases
    If kill_k0_imag is True, the k=0 (mean pressure) mode is hard-projected onto
    its real part - see out_nn_modes_uv's docstring for why.
    If hard_sym is True, each mode is reflection-symmetrized in y with parity
    (-1)^k (p_k is even/odd the same way as u_k) - see out_nn_modes_uv's
    docstring for why/cost.
    Input x,y : [Nint,1] real tf.float32 tensor
    Output shape : [1,Nint,Nmode] tf.complex64 tensor
    '''
    xint = tf.complex(x,0.)
    yint = tf.complex(y,0.)
    out_nn = neural_net(tf.transpose(tf.stack([xint,yint])),weights,biases)
    Nmode = int(out_nn[0,0,:].shape[0])
    if hard_sym:
        yint_neg = tf.complex(-y,0.)
        out_nn_neg = neural_net(tf.transpose(tf.stack([xint,yint_neg])),weights,biases)
    modes = []
    for k in range(Nmode):
        if street_params is not None and k >= 1:
            # R9 trust ansatz for pressure - same structure as
            # out_nn_modes_uv (see there), with the street's linearized-
            # Bernoulli pressure anchor as S_k. No fbc5 mask (pressure is
            # not zero on the cylinder - the taps live there).
            _, _, S_p_k = street_modes_k(x, y, street_params, k)
            corr_raw = out_nn[:,:,k]
            if hard_sym:
                parity = 1. if k % 2 == 0 else -1.
                corr_raw = 0.5*(corr_raw + tf.complex(parity,0.)*out_nn_neg[:,:,k])
            corr = tf.complex(tf.tanh(tf.real(corr_raw)), tf.tanh(tf.imag(corr_raw)))
            A = trust_rho*tf.abs(S_p_k) + trust_cap
            mode_k = S_p_k + tf.complex(A, 0.*A)*corr
        else:
            mode_k = out_nn[:,:,k]
            if hard_sym:
                parity = 1. if k % 2 == 0 else -1.  # (-1)^k
                mode_k = 0.5*(mode_k + tf.complex(parity,0.)*out_nn_neg[:,:,k])
        if k == 0 and kill_k0_imag:
            mode_k = tf.complex(tf.real(mode_k), 0.*tf.real(mode_k))
        modes.append(mode_k)
    t_parts = tf.convert_to_tensor(modes)
    return tf.transpose(t_parts,perm=[1,2,0])


def NN_time_uv(x,y,t,weights,biases,geom,omega_0,trunc_mode=None,freestream_target=None,damp_fluctuations=False,kill_k0_imag=False,hard_sym=False,is_v=False,street_params=None,trust_rho=0.6,trust_cap=0.12,v1_radial_params=None,v1_trust_rho=0.70,v1_xstart=3.0,v1_xwidth=0.30,v1_ymax=2.0,v1_ywidth=0.20):
    '''
    x,y,t : [Nint,1] tf.float32 tensors, list of coordinates (x,t) where to compute u or v(x,y,t)
    omega_0 : fondamental frequency
    Output [Nint,1] tf.float32 tensor
    trunc_mode : int (or None) : if an integer value is provided, select only
                the trunc_mode first mode given. Else if trunc_mode=None, use all modes
    freestream_target : passed through to out_nn_modes_uv (see there)
    damp_fluctuations : passed through to out_nn_modes_uv (see there)
    kill_k0_imag : passed through to out_nn_modes_uv (see there)
    hard_sym, is_v : passed through to out_nn_modes_uv (see there)
    street_params, trust_rho, trust_cap : R9 trust ansatz, passed through to
                out_nn_modes_uv (see there)
    '''
    out_NN = out_nn_modes_uv(x,y,weights,biases,geom,freestream_target=freestream_target,damp_fluctuations=damp_fluctuations,kill_k0_imag=kill_k0_imag,hard_sym=hard_sym,is_v=is_v,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap,v1_radial_params=v1_radial_params,v1_trust_rho=v1_trust_rho,v1_xstart=v1_xstart,v1_xwidth=v1_xwidth,v1_ymax=v1_ymax,v1_ywidth=v1_ywidth)
    Nmode = int(out_NN[0,0,:].shape[0])
    if trunc_mode!=None and trunc_mode <= Nmode:
        Nmode = trunc_mode
    parts = [out_NN[0,:,k]*tf.exp(k*omega_0*tf.complex(0.,t[:,0])) for k in range(Nmode)]
    t_parts = tf.convert_to_tensor(parts)
    t_real = tf.real(tf.reduce_sum(t_parts,axis=0))
    return tf.transpose(tf.convert_to_tensor([t_real])) # retrait de perm=[1,0]

def NN_time_p(x,y,t,weights,biases,omega_0,trunc_mode=None,kill_k0_imag=False,hard_sym=False,street_params=None,trust_rho=0.6,trust_cap=0.12):
    '''
    x,y,t : [Nint,1] tf.float32 tensors, list of coordinates (x,t) where to compute p(x,y,t)
    omega_0 : fondamental frequency
    Output [Nint,1] tf.float32 tensor
    kill_k0_imag : passed through to out_nn_modes_p (see there)
    hard_sym : passed through to out_nn_modes_p (see there)
    street_params, trust_rho, trust_cap : R9 trust ansatz, passed through to
                out_nn_modes_p (see there)
    '''
    out_NN = out_nn_modes_p(x,y,weights,biases,kill_k0_imag=kill_k0_imag,hard_sym=hard_sym,street_params=street_params,trust_rho=trust_rho,trust_cap=trust_cap)
    Nmode = int(out_NN[0,0,:].shape[0])
    if trunc_mode!=None and trunc_mode <= Nmode:
        Nmode = trunc_mode
    parts = [out_NN[0,:,k]*tf.exp(k*omega_0*tf.complex(0.,t[:,0])) for k in range(Nmode)]
    t_parts = tf.convert_to_tensor(parts)
    t_real = tf.real(tf.reduce_sum(t_parts,axis=0))
    return tf.transpose(tf.convert_to_tensor([t_real]))


# =============================================================================
# Declaration of the optimisers
# =============================================================================

def declare_LBFGS(loss,maxit=50000,maxfun=50000,ftol=1.0 * np.finfo(float).eps):
    # 'disp': True (ported from R6/R8): makes scipy's L-BFGS-B wrapper print
    # its actual termination message (CONVERGENCE vs
    # ABNORMAL_TERMINATION_IN_LNSRCH) - needed to tell genuine convergence
    # apart from a line-search failure (see R8/src for the full history).
    optimizer = tf.contrib.opt.ScipyOptimizerInterface(loss, method = 'L-BFGS-B', 
                                                                options = {'maxiter': maxit, #50000
                                                                           'maxfun': maxfun, #50000
                                                                           'maxcor': 50,
                                                                           'maxls': 50,
                                                                           'disp': True,
                                                                           'ftol' : ftol}) 
    print('L-BFGS-B optimizer declared with maxit = %d, maxfun = %d, ftol = %.2e'%(maxit,maxfun,ftol))
    return optimizer


def declare_Adam(loss,lr=1e-3,*args): #list_var=tf.trainable_variables()
    '''
    *args :
        var_list : trainable variables
    '''
    optimizer_Adam = tf.compat.v1.train.AdamOptimizer(learning_rate=lr)
    print('Adam optimize declared with learning rate = %.2e'%(lr))
    if len(args) == 1:
        list_var = args[0]
        return optimizer_Adam.minimize(loss,var_list=list_var)
    else:
        return optimizer_Adam.minimize(loss)


def declare_init_session():
    sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(allow_soft_placement=True, log_device_placement=False))
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)
    return sess

def square_norm(z):
    '''
    z : tf.complex64 tensor
    return tf tensor of square norm value of each complex
    '''
    return tf.square(tf.math.real(z)) + tf.square(tf.math.imag(z))

def square_norm_np(z):
    '''
    z : numpy array of complex values
    return np array of same dimension with square norm value of each complex number
    '''
    return np.square(np.real(z)) + np.square(np.imag(z))


# =============================================================================
# Entrainement
# =============================================================================


def simple_callback(loss):
    print('Loss: %.3e' % (loss))



def model_train_scipy(optimizer,sess,loss,tf_dict=None,fn_callback=simple_callback,List_loss = False, loss_valid = None, tf_dict_valid=None, step_hook=None, accepted_step_callback=None):
    '''
    step_hook : historical callback driven by loss_callback evaluations.
    accepted_step_callback : optional callable(xk) passed to TensorFlow's
    ScipyOptimizerInterface as step_callback. Unlike loss_callback, this is
    invoked once per accepted SciPy L-BFGS iteration and receives the packed
    optimizer parameter vector xk. R14 uses it for exact trajectory checkpoints.
    '''
    global it
    it = 0
    global List_it_loss_LBFGS
    List_it_loss_LBFGS = []
    global List_it_loss_valid_LBFGS
    List_it_loss_valid_LBFGS = []

    if List_loss == True and tf_dict_valid != None:

        def callback(loss,List_loss=List_loss):
            global it
            global List_it_loss_LBFGS
            global List_it_loss_valid_LBFGS
            it += 1
            List_it_loss_LBFGS.append([it,loss])
            if List_loss==True and it%100 == 0:
                loss_valid_value = sess.run(loss_valid,feed_dict=tf_dict_valid)
                List_it_loss_valid_LBFGS.append([it,loss_valid_value])
            print('Loss: %.3e' % (loss))
            if step_hook is not None:
                step_hook(it)
            # global it
            # it += 1

        fn_callback=callback

    if tf_dict != None:
        optimizer.minimize(sess,
                feed_dict = tf_dict,
                fetches = [loss],
                loss_callback = fn_callback,
                step_callback = accepted_step_callback)
    else:
        optimizer.minimize(sess,
                fetches = [loss],
                loss_callback = fn_callback,
                step_callback = accepted_step_callback) 
    return List_it_loss_LBFGS,List_it_loss_valid_LBFGS

def model_train_Adam(optimizer,sess,loss,liste_tf_dict=None,Nit=1e4,tolAdam=1e-4,it=0,itdisp=1000,maxTime=None,multigrid=False,NgridTurn=1000,List_loss = False, loss_valid = None, tf_dict_valid=None, step_hook=None):
    '''
    step_hook : optional callable(it) invoked once per Adam iteration - see
    model_train_scipy's docstring. None (default) = no-op.
    '''
    List_it_loss_Adam = []
    List_it_loss_valid_Adam = []
    if not(multigrid):
        tf_dict=[liste_tf_dict]
        NgridTurn=1
        Ngrid = 1
    else:
        tf_dict = liste_tf_dict
        Ngrid = len(tf_dict)

    t0 = time.time()
    loss_value = sess.run(loss, tf_dict[0])
    it0 = it
    conditionTime = True
    while(it-it0<Nit and loss_value>tolAdam and conditionTime):

        k_dict = int(it/NgridTurn)%Ngrid

        sess.run(optimizer, tf_dict[k_dict])
        loss_value = sess.run(loss, tf_dict[k_dict])

        if it%itdisp ==0:
            print('Post Adam it %d - Loss value :  %.3e' % (it, loss_value))

        if List_loss==True and it%100 == 0:
            loss_valid_value = sess.run(loss_valid,feed_dict=tf_dict_valid)
            List_it_loss_valid_Adam.append([it,loss_valid_value])
        List_it_loss_Adam.append([it,loss_value])

        if step_hook is not None:
            step_hook(it)

        it += 1
        conditionTime = (maxTime==None) or ((time.time()-t0)<maxTime)

    return List_it_loss_Adam,List_it_loss_valid_Adam




# =============================================================================
# Print and plot
# =============================================================================

def print_bar():
    print('--------------------------------------------')

def tf_print(string,tensor,sess,tf_dict=None):
    '''
    Parameters
    ----------
    string : String of character to display before the result of tf output
    tensor : tensor to compute and print
    sess : Current session object to compute given tensor
    tf_dict : dictionnary to feed, in cas it is necessary
    '''
    print(string + " " + str(sess.run(tensor,feed_dict=tf_dict)))



def tf_plot_scatter(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)

    # Step 2 : plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    plt.scatter(x_np,y_np,c=c_np,marker='.',s=1.)
    ax.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.colorbar()
    fig.tight_layout()

    return fig,ax


def tf_plot_scatter_complex_4fig(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    x and y are float32 tensors
    c is complex64 tensor
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)

    # Step 2 : plot
    fig = plt.figure(figsize=(8,6))
    plt.subplot(221)
    plt.scatter(x_np,y_np,c=np.real(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - real part')
    plt.colorbar()
    plt.subplot(222)
    plt.scatter(x_np,y_np,c=np.imag(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - imaginary part')
    plt.colorbar()
    plt.subplot(223)
    plt.scatter(x_np,y_np,c=np.sqrt(np.square(np.real(c_np))+np.square(np.imag(c_np))),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - norm')
    plt.colorbar()
    plt.subplot(224)
    plt.scatter(x_np,y_np,c=np.angle(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - angle')
    plt.colorbar()
    plt.tight_layout()

    return fig


def tf_plot_scatter_complex(x_tf,y_tf,c_tf,sess,xlabel='x',ylabel='y',title='',tf_dict=None):
    '''
    Parameters
    ----------
    x_tf, y_tf, c_tf : 1D tensor to compute and plot
    x and y are float32 tensors
    c is complex64 tensor
    sess : sess object to compute given tensors
    xlabel, ylabel, title : string to display on figure
    tf_dict = None : dictionnary to provide for computation of given tensors
    -------
    Returns pyplot fig object and ax
    '''
    # Step 1 : copputation
    x_np,y_np,c_np = sess.run([x_tf,y_tf,c_tf],feed_dict=tf_dict)

    # Step 2 : plot
    fig = plt.figure(figsize=(8,4))
    plt.subplot(121)
    plt.scatter(x_np,y_np,c=np.real(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+' - real part')
    plt.colorbar()
    plt.subplot(122)
    plt.scatter(x_np,y_np,c=np.imag(c_np),marker='.',s=1.)
    plt.axis('equal')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title+ ' - imaginary part')
    plt.colorbar()
    plt.tight_layout()

    return fig


def tf_plot_compare_3plot(x_tf,y_tf,c_tf_1,c_tf_2,sess,xlabel='x',ylabel='y',title1='',title2='',suptitle='',tf_dict=None):
    '''
    x_tf, y_tf, c_tf_1, c_tf_2 : 1D float32 tensor to compute and plot
    Return a figure with 3 subplots : c_tf_1, c_tf_2, log10((c_tf_1-c_tf_2)^2)
    '''

    x_np,y_np,c_np_1,c_np_2 = sess.run([x_tf,y_tf,c_tf_1,c_tf_2],feed_dict=tf_dict)

    fig = plt.figure(figsize=(15,4))
    plt.subplot(131)
    plt.scatter(x_np,y_np,c=c_np_1,marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title(title1)
    plt.subplot(132)
    plt.scatter(x_np,y_np,c=c_np_2,marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title(title2)
    plt.subplot(133)
    plt.scatter(x_np,y_np,c=np.log10(np.square(c_np_1-c_np_2)),marker='.',s=1.)
    plt.axis('equal')
    plt.colorbar()
    plt.title('Square difference - log10')
    plt.suptitle(suptitle)
    plt.tight_layout()

    return fig

In [ ]:
%%writefile evaluate_regions.py
"""
Standalone regional-error evaluation for a trained (pressure-only or dense)
ModalPINN run.

Loads a saved model checkpoint (no retraining) and the real CFD dataset,
reconstructs u, v, p over the real mesh nodes, and reports relative L2
error split by region (near-cylinder / near-wake / far-wake / whole domain)
so we can see whether reconstruction quality degrades away from the
sensors, per the project plan's regional-error metric (Section 8.1).

Usage:
    python evaluate_regions.py --RunDir <path to the run's output folder> \
        --WidthLayer 25 --Nmodes 3
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import matplotlib
matplotlib.use('Agg')  # must be set before NN_functions imports pyplot
import tensorflow as tf
tf.compat.v1.disable_eager_execution()

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
import NN_functions as nnf  # noqa: E402
from text_flow import read_flow  # noqa: E402

# Geometry / physics constants, matching ModalPINN_VortexShedding.py
X_C, Y_C, R_C = 0., 0., 0.5
LXMIN, LXMAX, LYMIN, LYMAX = -4., 8., -4., 4.
GEOM = [LXMIN, LXMAX, LYMIN, LYMAX, X_C, Y_C, R_C]
OMEGA_0 = 1.036
D = 2 * R_C  # cylinder diameter

# Relative to the current working directory, matching the plain relative path
# ModalPINN_VortexShedding.py itself uses (filename_data = 'Data/fixed_cylinder_atRe100').
# Don't compute this from __file__ / repo structure - in the Colab notebook this
# script and Data/ both sit flat in /content, not nested under src/pressure_only/.
DEFAULT_DATA_FILE = 'Data/fixed_cylinder_atRe100'


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--RunDir', required=True, help="Path to the run's output folder (contains DNN..._tanh.pickle)")
    parser.add_argument('--WidthLayer', type=int, required=True)
    parser.add_argument('--Nmodes', type=int, required=True)
    parser.add_argument('--Out', default=None, help='Optional JSON output path.')
    parser.add_argument('--DataFile', default=DEFAULT_DATA_FILE)
    parser.add_argument('--FreestreamBC', action='store_true', default=None,
                         help="Force freestream_target on restore. Default: auto-detect from 'FSBC' in --RunDir.")
    parser.add_argument('--FluctuationInletBC', action='store_true', default=None,
                         help="Force damp_fluctuations on restore. Default: auto-detect from 'FIBC' in --RunDir.")
    parser.add_argument('--HardSym', action='store_true', default=None,
                         help="Force hard mode-parity symmetrization on restore. Default: auto-detect from 'SYM' in --RunDir. Third occurrence of this bug class (see R3/R4) - a run trained with --HardSym computes a genuinely different field, not just a different training-time regularizer, so restoring without it evaluates the wrong function.")
    parser.add_argument('--StreetPrior', type=str, default=None,
                         help="Path to street_prior_Ntap<N>.npz. REQUIRED when evaluating a run trained "
                              "with --TrustStreet (auto-detected from 'TRUST' in --RunDir, or pass this "
                              "explicitly): the trust ansatz computes q_k = S_k + bounded correction, so "
                              "restoring without the street prior evaluates the wrong function - same bug "
                              "class as --HardSym (see there). Also copied into the run dir as "
                              "street_prior_used.npz by the R9 notebook for exactly this reason.")
    parser.add_argument('--TrustRho', type=float, default=0.6, help="Must match training (see ModalPINN_VortexShedding.py --TrustRho).")
    parser.add_argument('--TrustCap', type=float, default=0.12, help="Must match training (see ModalPINN_VortexShedding.py --TrustCap).")
    parser.add_argument('--V1RadialTrust', action='store_true', default=False)
    parser.add_argument('--V1TrustRho', type=float, default=0.70)
    parser.add_argument('--V1TrustXStart', type=float, default=3.0)
    parser.add_argument('--V1TrustXWidth', type=float, default=0.30)
    parser.add_argument('--V1TrustYMax', type=float, default=2.0)
    parser.add_argument('--V1TrustYWidth', type=float, default=0.20)
    args = parser.parse_args()

    run_dir_name = os.path.basename(os.path.normpath(args.RunDir))
    use_freestream = True if args.FreestreamBC else ('FSBC' in run_dir_name)
    use_fluct_damp = True if args.FluctuationInletBC else ('FIBC' in run_dir_name)
    use_hard_sym = True if args.HardSym else ('_SYM' in run_dir_name)
    street_params = None
    v1_radial_params = None
    if args.StreetPrior is not None or 'TRUST' in run_dir_name or args.V1RadialTrust or 'V1RAD' in run_dir_name:
        sp_path = args.StreetPrior
        if sp_path is None:
            cand = glob.glob(os.path.join(args.RunDir, 'street_prior*.npz'))
            assert cand, ("Trust/radial run needs street_prior*.npz in run dir or --StreetPrior.")
            sp_path = cand[0]
        _sp = np.load(sp_path)
        _loaded = {k: float(_sp[k]) for k in
                   ('Gamma', 'Uc', 'xf', 'r0', 'omega', 'phase',
                    'amp_scale', 'scale_p', 'ramp', 'delta')}
        if 'TRUST' in run_dir_name and 'V1RAD' not in run_dir_name and not args.V1RadialTrust:
            street_params = _loaded
            print('Restoring with legacy TrustStreet prior from', sp_path)
        if args.V1RadialTrust or 'V1RAD' in run_dir_name:
            v1_radial_params = _loaded
            print('Restoring with v1-only radial prior from', sp_path)
    freestream_target_u = 1.0 if use_freestream else None
    freestream_target_v = 0.0 if use_freestream else None
    print('Restoring with freestream_target=%s, damp_fluctuations=%s, hard_sym=%s (run dir: %s)' %
          (use_freestream, use_fluct_damp, use_hard_sym, run_dir_name))

    layers = [2, args.WidthLayer * args.Nmodes, args.WidthLayer * args.Nmodes, args.Nmodes]

    pickle_candidates = glob.glob(os.path.join(args.RunDir, 'DNN*_tanh.pickle'))
    assert pickle_candidates, f'No model pickle found in {args.RunDir}'
    model_file = pickle_candidates[0]
    print('Loading model:', model_file)

    w_u, b_u, w_v, b_v, w_p, b_p = nnf.restore_NN(layers, model_file, tf_as_constant=True)

    print('Reading real dataset...')
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)

    # Crop to the training domain box, same condition as read_cut_simulation_data
    nodes_x0, nodes_y0 = nodes_X[0, :], nodes_Y[0, :]
    in_box = ((nodes_x0 < LXMAX) & (nodes_x0 > LXMIN) &
              (nodes_y0 > LYMIN) & (nodes_y0 < LYMAX))
    idx = np.argwhere(in_box)[:, 0]
    nodes_x0, nodes_y0 = nodes_x0[idx], nodes_y0[idx]
    Us_c, Vs_c, Ps_c = Us[:, idx], Vs[:, idx], Ps[:, idx]

    r = np.sqrt((nodes_x0 - X_C) ** 2 + (nodes_y0 - Y_C) ** 2)
    region_near_cyl = r < 1.5 * R_C
    region_near_wake = (~region_near_cyl) & (nodes_x0 >= X_C) & (nodes_x0 < X_C + 3 * D)
    region_far_wake = (~region_near_cyl) & (~region_near_wake) & (nodes_x0 >= X_C + 3 * D)
    region_other = ~(region_near_cyl | region_near_wake | region_far_wake)

    regions = {
        'near-cylinder': region_near_cyl,
        'near-wake': region_near_wake,
        'far-wake': region_far_wake,
        'other (upstream/off-axis)': region_other,
        'whole domain': np.ones_like(region_near_cyl, dtype=bool),
    }

    x_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])
    y_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])
    t_tf = tf.compat.v1.placeholder(tf.float32, shape=[None, 1])

    u_pred_tf = nnf.NN_time_uv(x_tf, y_tf, t_tf, w_u, b_u, GEOM, OMEGA_0,
                                freestream_target=freestream_target_u, damp_fluctuations=use_fluct_damp,
                                hard_sym=use_hard_sym, is_v=False,
                                street_params=street_params, trust_rho=args.TrustRho, trust_cap=args.TrustCap)
    v_pred_tf = nnf.NN_time_uv(x_tf, y_tf, t_tf, w_v, b_v, GEOM, OMEGA_0,
                                freestream_target=freestream_target_v, damp_fluctuations=use_fluct_damp,
                                hard_sym=use_hard_sym, is_v=True,
                                street_params=street_params, trust_rho=args.TrustRho, trust_cap=args.TrustCap,
                                v1_radial_params=v1_radial_params, v1_trust_rho=args.V1TrustRho,
                                v1_xstart=args.V1TrustXStart, v1_xwidth=args.V1TrustXWidth,
                                v1_ymax=args.V1TrustYMax, v1_ywidth=args.V1TrustYWidth)
    p_pred_tf = nnf.NN_time_p(x_tf, y_tf, t_tf, w_p, b_p, OMEGA_0, hard_sym=use_hard_sym,
                               street_params=street_params, trust_rho=args.TrustRho, trust_cap=args.TrustCap)

    sess = tf.compat.v1.Session()
    sess.run(tf.compat.v1.global_variables_initializer())

    Nt, Nnode = Us_c.shape
    print(f'Reconstructing {Nt} timesteps x {Nnode} nodes (one timestep at a time, to keep memory bounded)...')
    x_col = nodes_x0.reshape(-1, 1).astype(np.float32)
    y_col = nodes_y0.reshape(-1, 1).astype(np.float32)

    u_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    v_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    p_pred = np.zeros((Nt, Nnode), dtype=np.float32)
    for k in range(Nt):
        t_col = np.full((Nnode, 1), times[k], dtype=np.float32)
        feed = {x_tf: x_col, y_tf: y_col, t_tf: t_col}
        u_pred[k, :] = sess.run(u_pred_tf, feed_dict=feed)[:, 0]
        v_pred[k, :] = sess.run(v_pred_tf, feed_dict=feed)[:, 0]
        p_pred[k, :] = sess.run(p_pred_tf, feed_dict=feed)[:, 0]
        if (k + 1) % 50 == 0 or k == Nt - 1:
            print(f'  {k + 1}/{Nt} timesteps done')

    def rel_l2(pred, true, mask):
        if mask.sum() == 0:
            return float('nan')
        diff = pred[:, mask] - true[:, mask]
        return np.linalg.norm(diff) / np.linalg.norm(true[:, mask])

    print()
    header = f"{'Region':<26}{'n_nodes':>9}{'E_u':>10}{'E_v':>10}{'E_p':>10}"
    print(header)
    print('-' * len(header))
    result = {'run_dir': args.RunDir, 'regions': {}}
    for name, mask in regions.items():
        eu = rel_l2(u_pred, Us_c, mask)
        ev = rel_l2(v_pred, Vs_c, mask)
        ep = rel_l2(p_pred, Ps_c, mask)
        result['regions'][name] = {
            'n_nodes': int(mask.sum()),
            'E_u': float(eu), 'E_v': float(ev), 'E_p': float(ep)}
        print(f"{name:<26}{mask.sum():>9}{eu:>10.4f}{ev:>10.4f}{ep:>10.4f}")

    if args.Out:
        with open(args.Out,'w') as fj:
            json.dump(result,fj,indent=2)
        print('Saved',args.Out)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile evaluate_v1_smoke.py
# -*- coding: utf-8 -*-
"""Short, mode-level evaluator for the R10 smoke comparison.

Restores a trained model, extracts the real CFD k=1 Fourier coefficient at the
same shedding frequency, and reports v1 relative error / complex correlation /
amplitude ratio by region. This is deliberately much cheaper than rebuilding
the full 201-timestep field.
"""
from __future__ import print_function
import argparse, os, glob, json, pickle
import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()

import NN_functions as nnf
from text_flow import read_flow

LXMIN, LXMAX = -4., 8.
LYMIN, LYMAX = -4., 4.
R_C = 0.5
D = 1.0
OMEGA_0 = 1.036
GEOM = [LXMIN,LXMAX,LYMIN,LYMAX,0.,0.,R_C]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--RunDir', required=True)
    ap.add_argument('--DataFile', default='Data/fixed_cylinder_atRe100')
    ap.add_argument('--WidthLayer', type=int, default=25)
    ap.add_argument('--Nmodes', type=int, default=3)
    ap.add_argument('--StreetPrior', default=None)
    ap.add_argument('--V1RadialTrust', action='store_true')
    ap.add_argument('--V1TrustRho', type=float, default=0.70)
    ap.add_argument('--V1TrustXStart', type=float, default=3.0)
    ap.add_argument('--V1TrustXWidth', type=float, default=0.30)
    ap.add_argument('--V1TrustYMax', type=float, default=2.0)
    ap.add_argument('--V1TrustYWidth', type=float, default=0.20)
    ap.add_argument('--Out', default=None)
    args = ap.parse_args()

    layers = [2, args.WidthLayer*args.Nmodes, args.WidthLayer*args.Nmodes, args.Nmodes]
    picks = glob.glob(os.path.join(args.RunDir, 'DNN*_tanh.pickle'))
    assert picks, 'No DNN pickle in %s' % args.RunDir
    model_file = picks[0]
    print('Loading model:', model_file)
    w_u,b_u,w_v,b_v,w_p,b_p = nnf.restore_NN(layers, model_file, tf_as_constant=True)

    v1_params = None
    if args.V1RadialTrust:
        assert args.StreetPrior is not None, '--V1RadialTrust requires --StreetPrior'
        z = np.load(args.StreetPrior)
        v1_params = {k: float(z[k]) for k in
                     ('Gamma','Uc','xf','r0','omega','phase',
                      'amp_scale','scale_p','ramp','delta')}

    print('Reading CFD...')
    Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)
    times = np.asarray(times)
    xa, ya = nodes_X[0], nodes_Y[0]
    box = ((xa > LXMIN)&(xa < LXMAX)&(ya > LYMIN)&(ya < LYMAX))
    idx = np.where(box)[0]
    x = xa[idx].astype(np.float32)
    y = ya[idx].astype(np.float32)
    V = Vs[:,idx]

    # Simultaneous k=0..3 fit at the ModalPINN omega.
    t = times-times[0]
    cols=[np.ones_like(t)]
    for k in range(1,4):
        cols += [np.cos(k*OMEGA_0*t), np.sin(k*OMEGA_0*t)]
    B=np.stack(cols,axis=1)
    coef,*_ = np.linalg.lstsq(B,V,rcond=None)
    v1_true = 0.5*(coef[1]-1j*coef[2])  # conventional q1, q=q0+q1e^iwt+c.c.
    del V, Vs, Us, Ps, nodes_X, nodes_Y

    x_tf=tf.compat.v1.placeholder(tf.float32,[None,1])
    y_tf=tf.compat.v1.placeholder(tf.float32,[None,1])
    modes_tf = nnf.out_nn_modes_uv(
        x_tf,y_tf,w_v,b_v,GEOM,
        freestream_target=0.0,damp_fluctuations=True,is_v=True,
        v1_radial_params=v1_params,
        v1_trust_rho=args.V1TrustRho,
        v1_xstart=args.V1TrustXStart,
        v1_xwidth=args.V1TrustXWidth,
        v1_ymax=args.V1TrustYMax,
        v1_ywidth=args.V1TrustYWidth)

    # Also evaluate the raw street mode, if provided, for reference.
    S_tf = None
    if v1_params is not None:
        _, S_tf, _ = nnf.street_modes_k(x_tf,y_tf,v1_params,1)

    sess=tf.compat.v1.Session()
    sess.run(tf.compat.v1.global_variables_initializer())

    chunk=20000
    v1_pred=np.zeros(len(x),dtype=np.complex64)
    s1=np.zeros(len(x),dtype=np.complex64) if S_tf is not None else None
    for i in range(0,len(x),chunk):
        sl=slice(i,min(i+chunk,len(x)))
        fd={x_tf:x[sl,None],y_tf:y[sl,None]}
        mm=sess.run(modes_tf,feed_dict=fd)
        v1_pred[sl]=mm[0,:,1]/2.0   # network is one-sided; convert to conventional
        if S_tf is not None:
            ss=sess.run(S_tf,feed_dict=fd)
            s1[sl]=ss[0,:]/2.0

    r=np.sqrt(x*x+y*y)
    regions={
        'near-cylinder': r < 0.75,
        'near-wake': (r>=0.75)&(x>=0)&(x<3),
        'far-wake': (r>=0.75)&(x>=3),
        'far-core': (r>=0.75)&(x>=3)&(np.abs(y)<=2.0),
        'whole-domain': np.ones(len(x),dtype=bool),
    }

    def metrics(pred,true,m):
        p=pred[m]; q=true[m]
        nt=np.linalg.norm(q)+1e-30
        npred=np.linalg.norm(p)
        return {
            'n': int(m.sum()),
            'rel_L2': float(np.linalg.norm(p-q)/nt),
            'corr': float(abs(np.vdot(p,q))/(npred*nt+1e-30)),
            'amp_ratio': float(npred/nt),
        }

    out={'run_dir':args.RunDir,'v1_radial':bool(args.V1RadialTrust),
         'rho':args.V1TrustRho,'xstart':args.V1TrustXStart,
         'regions':{}}
    for name,m in regions.items():
        out['regions'][name]=metrics(v1_pred,v1_true,m)

    if s1 is not None:
        out['raw_prior_far_core']=metrics(s1,v1_true,regions['far-core'])

    print('\nV1 MODE SMOKE METRICS')
    print('%-16s %8s %10s %10s %10s' % ('region','n','rel_L2','corr','amp_ratio'))
    for name in ['near-cylinder','near-wake','far-wake','far-core','whole-domain']:
        z=out['regions'][name]
        print('%-16s %8d %10.4f %10.4f %10.4f' %
              (name,z['n'],z['rel_L2'],z['corr'],z['amp_ratio']))
    if 'raw_prior_far_core' in out:
        z=out['raw_prior_far_core']
        print('\nraw street far-core: rel_L2=%.4f corr=%.4f amp_ratio=%.4f' %
              (z['rel_L2'],z['corr'],z['amp_ratio']))

    out_path=args.Out or os.path.join(args.RunDir,'v1_smoke_metrics.json')
    with open(out_path,'w') as f:
        json.dump(out,f,indent=2)
    print('Saved',out_path)

if __name__=='__main__':
    main()

In [ ]:
%%writefile evaluate_physics_uniform.py
# -*- coding: utf-8 -*-
from __future__ import print_function
import argparse, os, glob, json
import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()
import NN_functions as nnf

LXMIN,LXMAX=-4.,8.
LYMIN,LYMAX=-4.,4.
RC=0.5
RE=100.
OMEGA=1.036
GEOM=[LXMIN,LXMAX,LYMIN,LYMAX,0.,0.,RC]

def sample_uniform_fluid(n,seed):
    rng=np.random.RandomState(seed)
    xs=[]; ys=[]; got=0
    while got<n:
        m=max(2*(n-got),256)
        x=LXMIN+(LXMAX-LXMIN)*rng.rand(m)
        y=LYMIN+(LYMAX-LYMIN)*rng.rand(m)
        ok=x*x+y*y>RC*RC
        x=x[ok]; y=y[ok]
        take=min(n-got,len(x))
        xs.append(x[:take]); ys.append(y[:take]); got+=take
    x=np.concatenate(xs).astype(np.float32)
    y=np.concatenate(ys).astype(np.float32)
    t=(100.*rng.rand(n)).astype(np.float32)
    return x,y,t

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--RunDir',required=True)
    ap.add_argument('--WidthLayer',type=int,default=25)
    ap.add_argument('--Nmodes',type=int,default=3)
    ap.add_argument('--N',type=int,default=20000)
    ap.add_argument('--Seed',type=int,default=99173)
    ap.add_argument('--StreetPrior',required=True)
    ap.add_argument('--V1TrustRho',type=float,default=.70)
    ap.add_argument('--V1TrustXStart',type=float,default=3.)
    ap.add_argument('--V1TrustXWidth',type=float,default=.30)
    ap.add_argument('--V1TrustYMax',type=float,default=2.)
    ap.add_argument('--V1TrustYWidth',type=float,default=.20)
    ap.add_argument('--Out',required=True)
    args=ap.parse_args()

    layers=[2,args.WidthLayer*args.Nmodes,args.WidthLayer*args.Nmodes,args.Nmodes]
    picks=glob.glob(os.path.join(args.RunDir,'DNN*_tanh.pickle'))
    assert picks,'No DNN pickle in '+args.RunDir
    wu,bu,wv,bv,wp,bp=nnf.restore_NN(layers,picks[0],tf_as_constant=True)

    z=np.load(args.StreetPrior)
    sp={k:float(z[k]) for k in
        ('Gamma','Uc','xf','r0','omega','phase','amp_scale','scale_p','ramp','delta')}

    xph=tf.compat.v1.placeholder(tf.float32,[None,1])
    yph=tf.compat.v1.placeholder(tf.float32,[None,1])
    tph=tf.compat.v1.placeholder(tf.float32,[None,1])

    u=nnf.NN_time_uv(
        xph,yph,tph,wu,bu,GEOM,OMEGA,
        freestream_target=1.0,damp_fluctuations=True,is_v=False)
    v=nnf.NN_time_uv(
        xph,yph,tph,wv,bv,GEOM,OMEGA,
        freestream_target=0.0,damp_fluctuations=True,is_v=True,
        v1_radial_params=sp,
        v1_trust_rho=args.V1TrustRho,
        v1_xstart=args.V1TrustXStart,
        v1_xwidth=args.V1TrustXWidth,
        v1_ymax=args.V1TrustYMax,
        v1_ywidth=args.V1TrustYWidth)
    p=nnf.NN_time_p(xph,yph,tph,wp,bp,OMEGA)

    ut=tf.gradients(u,tph)[0]
    vt=tf.gradients(v,tph)[0]
    ux=tf.gradients(u,xph)[0]; uy=tf.gradients(u,yph)[0]
    vx=tf.gradients(v,xph)[0]; vy=tf.gradients(v,yph)[0]
    uxx=tf.gradients(ux,xph)[0]; uyy=tf.gradients(uy,yph)[0]
    vxx=tf.gradients(vx,xph)[0]; vyy=tf.gradients(vy,yph)[0]
    px=tf.gradients(p,xph)[0]; py=tf.gradients(p,yph)[0]

    fu=ut+(u*ux+v*uy)+px-(1./RE)*(uxx+uyy)
    fv=vt+(u*vx+v*vy)+py-(1./RE)*(vxx+vyy)
    div=ux+vy
    terms=[tf.square(fu),tf.square(fv),tf.square(div)]

    x,y,t=sample_uniform_fluid(args.N,args.Seed)
    r=np.sqrt(x*x+y*y)
    masks={
        'whole_uniform':np.ones(args.N,dtype=bool),
        'formation_wake':(x>=0.)&(x<3.)&(np.abs(y)<=2.),
        'far_core':(x>=3.)&(np.abs(y)<=2.),
        'near_cylinder':(r>0.5)&(r<=1.2),
        'outside_wake':~((x>=0.)&(np.abs(y)<=2.)),
    }

    sess=tf.compat.v1.Session()
    sess.run(tf.compat.v1.global_variables_initializer())

    vals=[np.empty(args.N,dtype=np.float64) for _ in range(3)]
    chunk=5000
    for i in range(0,args.N,chunk):
        sl=slice(i,min(i+chunk,args.N))
        got=sess.run(terms,feed_dict={
            xph:x[sl,None],yph:y[sl,None],tph:t[sl,None]})
        for j in range(3):
            vals[j][sl]=got[j][:,0]

    out={'N':args.N,'seed':args.Seed,'regions':{}}
    for name,m in masks.items():
        fu2=float(np.mean(vals[0][m]))
        fv2=float(np.mean(vals[1][m]))
        div2=float(np.mean(vals[2][m]))
        out['regions'][name]={
            'n':int(np.sum(m)),
            'fu2':fu2,'fv2':fv2,'div2':div2,
            'total_ns_mse':fu2+fv2+div2,
        }

    with open(args.Out,'w') as f:
        json.dump(out,f,indent=2)
    print(json.dumps(out,indent=2))

if __name__=='__main__':
    main()


In [ ]:
%%writefile street_prior.py
"""R9: derive the closed-form vortex-street prior from the 32 wall taps.

Standalone, numpy-only (no TF). Run BEFORE training:

    python street_prior.py --DataFile Data/fixed_cylinder_atRe100 --NTaps 32

Writes street_prior_Ntap<N>.npz with the closed-form street parameters that
ModalPINN_VortexShedding.py --TrustStreet consumes.

EVERY number here derives from the tap pressures + classical physics:
- omega0: nonlinear sinusoid fit to the tap-integrated lift series
- Gamma:  von Karman drag relation == tap-measured pressure drag / 0.75
          (0.75 = pressure share of total drag at Re~100, textbook value)
- Uc, a:  self-consistent street kinematics (Uc = 1 - Gamma/(sqrt8 a),
          a = 2 pi Uc / omega0)
- xf, r0, phase: matching the IMAGE-SYSTEM street's induced k=1 surface
          pressure pattern to the measured tap k=1 harmonics (the
          Milne-Thomson images make the surface pattern orientation-aware)
- closed-form calibration: the TF-portable single-harmonic expansion is
          aligned (phase offset + amplitude scale) against the numeric
          Lamb-Oseen street ON WAKE PROBE POINTS - a street-to-street
          calibration, no reference data involved.

The reference CFD file is read ONLY to extract the tap pressures - the
exact signals the training script itself trains on.

Method developed and validated in R9_wake_rescue/ (see REPORT.md there).
"""
import argparse
import os

import numpy as np
from scipy.optimize import least_squares

from text_flow import read_flow

# geometry, matching ModalPINN_VortexShedding.py
X_C, Y_C, R_C = 0.0, 0.0, 0.5
LXMIN, LXMAX, LYMIN, LYMAX = -4.0, 8.0, -4.0, 4.0
GEOM = [LXMIN, LXMAX, LYMIN, LYMAX, X_C, Y_C, R_C]
D = 2 * R_C
RE = 100.0
NU = 1.0 / RE
HA_RATIO = 0.281


# ===========================================================================
# numeric street (Lamb-Oseen rows + Milne-Thomson images + dipole)
# - reference implementation for the fit; identical math to
#   R9_wake_rescue/src/analytic_street.py
# ===========================================================================
class Street:
    def __init__(self, Gamma, U_c, x_f=1.0, r0=0.3, phase=0.0, omega=1.036,
                 ramp=0.75):
        self.G, self.Uc, self.omega = Gamma, U_c, omega
        self.a = 2 * np.pi * U_c / omega
        self.h = HA_RATIO * self.a
        self.xf, self.r0, self.phase, self.ramp = x_f, r0, phase, ramp

    def _vortex_positions(self, t, nwin=30):
        ks = np.arange(-nwin, nwin + 1)
        shift = (self.Uc * t + self.phase / self.omega * self.Uc) % self.a
        xu = self.xf + self.a * ks + shift
        xl = self.xf + self.a * (ks + 0.5) + shift
        return (np.stack([xu, np.full_like(xu, +self.h / 2)], 1),
                np.stack([xl, np.full_like(xl, -self.h / 2)], 1))

    def _induced(self, pts, vort_xy, gamma, core_from_x=None):
        dx = pts[:, None, 0] - vort_xy[None, :, 0]
        dy = pts[:, None, 1] - vort_xy[None, :, 1]
        r2 = dx ** 2 + dy ** 2 + 1e-12
        xv = vort_xy[:, 0] if core_from_x is None else core_from_x
        rc2 = self.r0 ** 2 + 4 * NU * np.clip(xv - self.xf, 0, None) / self.Uc
        fac = (1 - np.exp(-r2 / rc2[None, :])) / (2 * np.pi * r2)
        return (-gamma * dy * fac).sum(1), (gamma * dx * fac).sum(1)

    def velocity(self, pts, t):
        up, lo = self._vortex_positions(t)
        uu, vu = self._induced(pts, up, -self.G)
        ul, vl = self._induced(pts, lo, +self.G)
        u, v = uu + ul, vu + vl
        a2 = R_C ** 2
        for row, g in ((up, -self.G), (lo, +self.G)):
            r2v = row[:, 0] ** 2 + row[:, 1] ** 2
            img = row * (a2 / r2v)[:, None]
            ui, vi = self._induced(pts, img, -g, core_from_x=row[:, 0])
            u, v = u + ui, v + vi
        env = 0.5 * (1 + np.tanh((pts[:, 0] - self.xf) / self.ramp))
        x, y = pts[:, 0], pts[:, 1]
        r2 = x ** 2 + y ** 2 + 1e-12
        u_mean = 1.0 - a2 * (x ** 2 - y ** 2) / r2 ** 2
        v_mean = -a2 * 2 * x * y / r2 ** 2
        return u_mean + u * env, v_mean + v * env

    def pressure(self, pts, t):
        u, v = self.velocity(pts, t)
        return -0.5 * ((u - self.Uc) ** 2 + v ** 2)

    def modes(self, pts, nk=3, nt=16):
        T = 2 * np.pi / self.omega
        ts = np.arange(nt) * T / nt
        U = np.empty((nt, len(pts))); V = np.empty_like(U); P = np.empty_like(U)
        for i, t in enumerate(ts):
            U[i], V[i] = self.velocity(pts, t)
            P[i] = self.pressure(pts, t)
        out = {}
        for name, F in (('u', U), ('v', V), ('p', P)):
            c = np.fft.fft(F, axis=0) / nt
            out[name] = [c[0].real] + [c[k] for k in range(1, nk + 1)]
        return out


def uc_of_gamma(G, omega):
    Uc = 0.85
    for _ in range(50):
        a = 2 * np.pi * Uc / omega
        Uc_new = 1.0 - G / (np.sqrt(8.0) * a)
        if abs(Uc_new - Uc) < 1e-12:
            break
        Uc = Uc_new
    return Uc, 2 * np.pi * Uc / omega


def karman_drag_CD(G, Uc, a, h):
    u_ind = G / (np.sqrt(8.0) * a)
    return ((G * h / a) * (1.0 - 2.0 * u_ind) + G ** 2 / (2 * np.pi * a)) \
        / (0.5 * D)


# ===========================================================================
# closed-form street (TF-portable) - identical math to
# R9_wake_rescue/src/closed_form_street.py, numpy backend
# ===========================================================================
def cf_modes_uv(x, y, prm, nk=3):
    """One-sided modes k=1..nk of the closed-form street. Returns us, vs."""
    G, Uc, xf, r0, phase, omega, ramp, delta = (
        prm['Gamma'], prm['Uc'], prm['xf'], prm['r0'], prm['phase'],
        prm['omega'], prm.get('ramp', 0.75), prm.get('delta', 0.35))
    a = 2 * np.pi * Uc / omega
    h = HA_RATIO * a
    env = 0.5 * (1 + np.tanh((x - xf) / ramp))
    rc2 = r0 ** 2 + 4 * NU * np.clip(x - xf, 0, None) / Uc
    us, vs = [], []
    for k in range(1, nk + 1):
        att = np.exp(-(np.pi * k) ** 2 * rc2 / a ** 2)
        tot_u = np.zeros_like(x, dtype=complex)
        tot_v = np.zeros_like(x, dtype=complex)
        for y_row, sgn_row, x0 in ((+h / 2, -1.0, xf), (-h / 2, +1.0, xf + a / 2)):
            yp = y - y_row
            sabs = np.sqrt(yp ** 2 + delta ** 2) - delta
            sgn = -np.tanh(yp / delta)
            Dk = np.exp(-2 * np.pi * k * sabs / a)
            ph = -2 * np.pi * k * (x - x0) / a - k * phase
            Ek = np.cos(ph) + 1j * np.sin(ph)
            base = sgn_row * G / (2 * a) * Ek * Dk * att
            tot_u = tot_u + sgn * base
            tot_v = tot_v + 1j * base
        us.append(tot_u * env)
        vs.append(tot_v * env)
    return us, vs


# ===========================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--DataFile', default='Data/fixed_cylinder_atRe100')
    ap.add_argument('--NTaps', type=int, default=32)
    ap.add_argument('--Out', default=None)
    args = ap.parse_args()
    out_path = args.Out or f'street_prior_Ntap{args.NTaps}.npz'

    # ---- 1. tap pressures - same selection logic as Load_train_data_desync
    # cut_simu_cylinder_only (transcribed, not imported: that module imports
    # tensorflow, which this numpy-only script must not depend on).
    Re_, Ur_, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)
    eps = 1e-5
    r_all = np.sqrt((nodes_X[0, :] - X_C) ** 2 + (nodes_Y[0, :] - Y_C) ** 2)
    idx_cyl = np.argwhere((r_all - R_C) ** 2 < eps)[:, 0]
    xc_all, yc_all = nodes_X[0, idx_cyl], nodes_Y[0, idx_cyl]
    s_lin = np.linspace(0., 1., args.NTaps, endpoint=False)
    x_t = X_C + R_C * np.cos(2 * np.pi * s_lin)
    y_t = Y_C + R_C * np.sin(2 * np.pi * s_lin)
    pick = np.array([np.argmin((xc_all - x_t[k]) ** 2 + (yc_all - y_t[k]) ** 2)
                     for k in range(args.NTaps)])
    print('Cylinder taps requested: %d, distinct mesh nodes found: %d'
          % (args.NTaps, len(np.unique(pick))))
    x_cyl, y_cyl = xc_all[pick], yc_all[pick]
    p_cyl = Ps[:, idx_cyl[pick]]             # (Nt, NTaps)
    t = np.asarray(times) - times[0]
    print(f'taps: {p_cyl.shape}, t in [0, {t[-1]:.1f}]')

    theta = np.arctan2(y_cyl - Y_C, x_cyl - X_C)
    order = np.argsort(theta)
    th_s, p_s = theta[order], p_cyl[:, order]
    dth = np.diff(np.concatenate([th_s, [th_s[0] + 2 * np.pi]]))
    w = 0.5 * (dth + np.roll(dth, 1))
    CD = -(p_s * np.cos(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)
    CL = -(p_s * np.sin(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)

    # ---- 2. omega0 from a nonlinear sinusoid fit to CL
    z = CL - CL.mean()
    crossings = np.where(np.diff(np.sign(z)) != 0)[0]
    w_init = np.pi / np.mean(np.diff(t[crossings]))
    fit = least_squares(
        lambda prm: prm[0] * np.sin(prm[2] * t + prm[1]) + prm[3] - CL,
        [0.5 * (CL.max() - CL.min()), 0.0, w_init, CL.mean()], method='lm')
    omega = abs(float(fit.x[2]))
    CD0 = float(CD.mean())
    print(f'omega0_hat = {omega:.5f}  CD_pressure = {CD0:.4f}')

    # ---- 3. per-tap k=1 harmonics
    cols = [np.ones_like(t), np.cos(omega * t), np.sin(omega * t)]
    A = np.stack(cols, 1)
    cf_, *_ = np.linalg.lstsq(A, p_s, rcond=None)
    p1_meas = 0.5 * (cf_[1] - 1j * cf_[2])

    # ---- 4. Gamma from the Karman drag relation (bisection)
    CD_target = CD0 / 0.75
    lo, hi = 0.5, 6.0
    for _ in range(60):
        G = 0.5 * (lo + hi)
        Uc, a = uc_of_gamma(G, omega)
        if karman_drag_CD(G, Uc, a, HA_RATIO * a) < CD_target:
            lo = G
        else:
            hi = G
    G = 0.5 * (lo + hi)
    Uc, a = uc_of_gamma(G, omega)
    print(f'Gamma = {G:.3f}  Uc = {Uc:.3f}  a = {a:.3f}')

    # ---- 5. xf, r0, phase from the tap k=1 pattern (image street)
    tap_pts = np.stack([R_C * np.cos(th_s), R_C * np.sin(th_s)], 1)
    best = None
    for xf in (0.6, 0.8, 1.0, 1.2):
        for r0 in (0.2, 0.3, 0.4):
            st = Street(G, Uc, x_f=xf, r0=r0, omega=omega)
            sm = st.modes(tap_pts, nk=1, nt=16)
            p1s = sm['p'][1]
            corr = np.abs(np.vdot(p1s, p1_meas)) / (
                np.linalg.norm(p1s) * np.linalg.norm(p1_meas))
            phi = np.angle(np.vdot(p1s, p1_meas))
            if best is None or corr > best[0]:
                best = (corr, xf, r0, phi)
    corr_tap, xf, r0, phi = best
    print(f'xf = {xf}  r0 = {r0}  phase = {phi:+.3f}  tap-p1 corr = {corr_tap:.3f}')

    # ---- 6. calibrate the closed form against the numeric street
    num = Street(G, Uc, x_f=xf, r0=r0, phase=phi, omega=omega)
    rng = np.random.default_rng(3)
    pts = rng.uniform([1.0, -2.0], [8.0, 2.0], size=(1500, 2))
    sm = num.modes(pts, nk=3, nt=16)
    prm = dict(Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega, phase=phi)
    best = None
    for extra in np.linspace(-np.pi, np.pi, 48, endpoint=False):
        prm['phase'] = phi + extra
        us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
        inner = np.vdot(vs[0], sm['v'][1])
        corr = abs(inner) / (np.linalg.norm(vs[0])
                             * np.linalg.norm(sm['v'][1]) + 1e-30)
        score = corr - abs(np.angle(inner)) * 0.05
        if best is None or score > best[0]:
            best = (score, corr, extra)
    _, corr_cf, extra = best
    prm['phase'] = phi + extra
    us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
    amp_scale = float(np.linalg.norm(sm['v'][1]) / np.linalg.norm(vs[0]))
    # pressure anchor: p_k ~ -(1-Uc) u_k, amplitude-matched at k=1
    p1_approx = -(1.0 - Uc) * us[0] * amp_scale
    scale_p = float(np.linalg.norm(sm['p'][1]) / np.linalg.norm(p1_approx))
    print(f'closed-form calibration: corr vs numeric = {corr_cf:.3f}, '
          f'amp_scale = {amp_scale:.3f}, scale_p = {scale_p:.3f}')
    assert corr_cf > 0.95, 'closed-form street failed to match numeric street'

    np.savez(out_path,
             Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega,
             phase=prm['phase'], amp_scale=amp_scale, scale_p=scale_p,
             ramp=0.75, delta=0.35,
             CD_pressure=CD0, tap_p1_corr=corr_tap,
             cf_corr_vs_numeric=corr_cf)
    print('saved', out_path)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile text_flow.py
"""

Author: Mouad Boudina
From: https://zenodo.org/record/5039610


The flow file structure is the following:

Re Ur
(blank line)
Nt N_nodes (Nt = length of the timeline of the flow simulation)
(blank line)
t0
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...
t1
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...

"""
import time
import numpy as np
#==============================================================================

def floatIt(l):
    return np.array([float(e) for e in l])

def intIt(l):
    return np.array([int(e) for e in l])

def read_flow(infile):
    f = open(infile, 'r')

    t1 = time.process_time()

    print('Reading flow...')

    Re, Ur = floatIt(f.readline().strip().split())

    f.readline() # blank line

    Nt, N_nodes = intIt(f.readline().strip().split())

    f.readline()

    times = []

    nodes_X, nodes_Y = [], []
    Us, Vs, ps = [], [], []

    for n in range(Nt):
        tn = float(f.readline().strip())
        times.append(tn)

        print('%.3f' % tn)

        tmp_nodes_X, tmp_nodes_Y = [], []
        tmp_Us, tmp_Vs, tmp_ps = [], [], []

        for k in range(N_nodes):
            x, y, U, V, p = floatIt(f.readline().strip().split())

            tmp_nodes_X.append(x)
            tmp_nodes_Y.append(y)

            tmp_Us.append(U)
            tmp_Vs.append(V)
            tmp_ps.append(p)

        nodes_X.append(tmp_nodes_X)
        nodes_Y.append(tmp_nodes_Y)

        Us.append(tmp_Us)
        Vs.append(tmp_Vs)
        ps.append(tmp_ps)

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

    return Re, Ur, np.array(times), \
           np.array(nodes_X), np.array(nodes_Y), \
           np.array(Us), np.array(Vs), np.array(ps)

def write_flow(flow, outfile):
    f = open(outfile, 'w')

    t1 = time.process_time()

    print('Writing flow...')

    f.write('%.0f %.1f\n' % (flow.Re, flow.Ur))
    f.write('\n') # blank line

    Nt, N_nodes = len(flow.times), len(flow.nodes_X[0])

    f.write('%d %d\n' % (Nt, N_nodes))
    f.write('\n')

    for n in range(Nt):
        tn = flow.times[n]

        print('%.6f' % tn)

        f.write('%.6f\n' % tn)

        for k in range(N_nodes):
            f.write('%13.9f %13.9f %13.9f %13.9f %13.9f\n' %\
                    (flow.nodes_X[n, k],
                     flow.nodes_Y[n, k],
                     flow.Us[n, k],
                     flow.Vs[n, k],
                     flow.ps[n, k]))

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

In [ ]:
%%writefile reactions_process.py
"""
Author: Mouad Boudina
From: https://zenodo.org/record/5039610
"""
import numpy as np

from scipy.optimize  import curve_fit
from scipy.integrate import trapz
#==============================================================================
labels_solid = {1:r'$X/D$',
                2:r'$Y/D$',
                3:r'$\theta$',
                4:r'$U/U_{0}$',
                5:r'$V/U_{0}$',
                6:r'$\dot{\theta}$',
                7:r'$F_{X}$',
                8:r'$F_{Y}$',
                9:r'$M_{z}$'}

labels_entity = {1:r'$F_{X}$',
                 2:r'$F_{Y}$',
                 3:r'$M_{z}$'}

labels = {'solid':labels_solid, 'entity':labels_entity}

#fontsize = 12
fontsize = 10

tol = 1e-2

def floatIt(l):
    return np.array([float(e) for e in l])

def extract_reactions(infile):
    f = open(infile, 'r')

    # A dummy test to know whether we are reading reactions of an entity or a
    # solid (in 2D).
    line = f.readline()
    if len(line.strip().split()) == 5:
        flag = 'entity'
    else:
        flag = 'solid'
    f.seek(0)

    times = []

    if flag == 'entity':
        Fx, Fy, Mz = [], [], []

        for line in f:
            # The ':-1' is used to avoid floating the name of the entity in the
            # last column.
            fragmented = floatIt(line.strip().split()[:-1])

            times.append(fragmented[0])

            Fx.append(fragmented[1])
            Fy.append(fragmented[2])
            Mz.append(fragmented[3])

        return np.array(times), np.array(Fx), np.array(Fy), np.array(Mz), flag

    else:
        X, Y, theta = [], [], []
        U, V, theta_dot = [], [], []
        Fx, Fy, Mz = [], [], []

        for line in f:
            # The ':-1' is used to avoid floating the name of the solid in the
            # last column.
            fragmented = floatIt(line.strip().split()[:-1])

            times.append(fragmented[0])

            X.append(fragmented[1])
            Y.append(fragmented[2])
            theta.append(fragmented[3])

            U.append(fragmented[4])
            V.append(fragmented[5])
            theta_dot.append(fragmented[6])

            Fx.append(fragmented[7])
            Fy.append(fragmented[8])
            Mz.append(fragmented[9])

        return np.array(times), \
               np.array(X), np.array(Y), np.array(theta), \
               np.array(U), np.array(V), np.array(theta_dot), \
               np.array(Fx), np.array(Fy), np.array(Mz), flag

def plot_reactions(reactions, variable_index, ax, color):
    plot_params = {'linestyle':'-',
                   'color'    :color,
                   'marker'   :'.',
                   'markerfacecolor':'cyan',
                   'label'    :labels[reactions[-1]][variable_index]}

    ax.plot(reactions[0], reactions[variable_index], **plot_params)

    ax.set_xlabel(r'$\bar{t}=tU_{0}/D$', fontsize=12)
    ax.set_ylabel(r'$F_{y}$', fontsize=12)

#    ax.legend(loc='best', numpoints=1,
#              fontsize=fontsize,
#              frameon=False,
#              ncol=1,
#              labelspacing=0.2,
#              handlelength=0.2)

def get_Fd_and_Fl(reactions, alpha):
    if reactions[-1] == 'solid':
        exception_msg = 'We are sorry, but this function is useful only for' \
                      + 'fixed objects.'
        raise Exception(exception_msg)

    # WARNING:
    # alpha IS IN DEGREES. MUST BE CONVERTED TO RADIANS TO USE IT IN PYTHON FUNCTIONS.
    a = (np.pi/180.)*alpha

    times, Fx, Fy = reactions[:3]

    Fd =  Fx*np.cos(a) + Fy*np.sin(a)
    Fl = -Fx*np.sin(a) + Fy*np.cos(a)

    return times, Fd, Fl

def first_maximum(reactions, variable_index, i0):
    """
    Find the first occurrence of the maximum, starting from index i0, and
    returns that index with the value of the maximum.
    """
    times = reactions[0]

    psi = reactions[variable_index]

    Nt = len(times)
    for i in range(i0, Nt-1):
        if psi[i] > max(psi[i-1], psi[i+1]):
            return psi[i], i

    raise Exception('The function is probably constant, or hasn\'t the same peak.')

def find_t400_t410(reactions):
    times = reactions[0]
    Nt = len(times)

    i400, i410 = 0, 0

    for i in range(Nt):
        if times[i] > 400.:
            i400 = i
            break

    for i in range(i400+1, Nt):
        if times[i] > 410.:
            i410 = i
            break

    return i400, i410

def find_period(reactions, variable_index, ax):
#    if reactions[-1] == 'entity':
#        exception_msg = 'We are sorry, but this function is useful only for' \
#                      + ' moving objects.'
#        raise Exception(exception_msg)

    times = reactions[0]

    i400, i410 = find_t400_t410(reactions)

    psi_max, imax = first_maximum(reactions, variable_index, i400)

    psi_max2, imax2 = first_maximum(reactions, variable_index, imax+1)
    # Sometimes there is local maximums, so we keep searching until we find
    # the real maximum that has the same value as psi_max.
    while abs(psi_max - psi_max2) > tol:
        psi_max2, imax2 = first_maximum(reactions, variable_index, imax2+1)

    psi = reactions[variable_index]

    if ax != None:
        ax.plot(times[i400:i410], psi[i400:i410], color='blue', linestyle='-')

        ax.plot(times[[imax, imax2]], psi[[imax, imax2]],
                color='red', linestyle='-', marker='o')

        ax.set_xlabel('Time', size='xx-large')
        ax.set_ylabel(labels[reactions[-1]][variable_index], size='xx-large')

    period = times[imax2] - times[imax]
    print('FLOW PERIOD = %0.6f' % period)

    mean = np.mean(psi[imax:imax2])
    print('MEAN = %0.6f' % mean)

    maxi = max(psi[imax:imax2])
    print('MAX  = %0.6f' % maxi)

    amp = maxi - mean
    print('AMP  = %0.6f' % amp)

    return period, mean, maxi, amp

def fit_a_sine(reactions, variable_index, ax):
    period, mean, maxi, amp = find_period(reactions, variable_index, ax)

    times = reactions[0]
    psi = reactions[variable_index]
    cent_norm = (psi - mean)/amp

    i400, i410 = find_t400_t410(reactions)

    def f(t, phi):
        return np.sin(2*np.pi*t/period + phi)

    popt, pcov = curve_fit(f, times[i400:i410], cent_norm[i400:i410])
    phi = popt

    print('PHASE LAG = %0.6f = %0.3f PI' % (phi, phi/np.pi))

    if ax != None:
        ax.plot(times[i400:i410], mean + amp*f(times[i400:i410], phi),
                color='r',
                linestyle='--')

    return phi

def eight_figure(reactions, frac, ax, c, equal, maxi_norm=False):
    if reactions[-1] == 'entity':
        exception_msg = 'The eight is a trajectory of a moving solid, but your' \
                      + ' entry is an entity (i.e. fixed body).'
        raise Exception(exception_msg)

    times, X, Y = reactions[:3]

    Nt = len(times)

    # Same notice as in the previous function find_period.
    i0 = int(frac*Nt)

    x, y = X[i0:]/0.075, Y[i0:]
#    x, y = X[i0:], Y[i0:]

    width  = max(x) - min(x)
    height = max(y) - min(y)
    ratio  = height/width

    print('width/2 = %0.6f' % (width/2.))
    print('ratio   = %0.6f' % ratio)

    m = np.mean(x)

    if maxi_norm:
        maxi_x = np.max(x-m)
        maxi_y = np.max(y)
    else:
        maxi_x = 1
        maxi_y = 1

    ax.plot((x - m)/maxi_x + c, y/maxi_y, linestyle='-', color='black')

#    ax.set_xlabel(r'$X/D$', fontsize=fontsize)
    ax.set_xlabel(r'$U_{\mathrm{r}}$', fontsize=fontsize)
    ax.set_ylabel(r'$Y/D$', fontsize=fontsize)

    ax.tick_params(axis='both', which='major', labelsize=fontsize)

#    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

    ax.set_xticks(range(3,11))
#    ax.set_yticks([-5e-1,0,5e-1])

    if equal:
        ax.axis('equal')

    annotate = False
    if annotate:
        x1 = min(x) - m
        x2 = max(x) - m

        y1_arrow = 1.1*min(y)
        y1_text = 1.1*y1_arrow

        ax.annotate(s='', xy=(x1,y1_arrow), xytext=(x2,y1_arrow),
                    arrowprops=dict(arrowstyle='<->'))

        ax.text(x= (x1 + x2)/2.,
                y=y1_text,
                s=r'$2\bar{X}_{\mathrm{max}}$',
                color='black',
                horizontalalignment='center',
                verticalalignment='top',
                fontsize=fontsize)

#    ax.set_ylim([-.9,.9])
#    ax.set_ylim([-.7,.7])
#    ax.set_xlim([-.09,.09])

def find_upper_shell(reactions, frac, ax, equal):
    if reactions[-1] == 'entity':
        exception_msg = 'The upper shell is the upper trajectory of a moving '\
                      + 'solid, but your entry is an entity.'
        raise Exception(exception_msg)

    times, X, Y = reactions[:3]
    U, V = reactions[4:6]

    speed = np.sqrt(U**2 + V**2)

    Nt = len(times)

    i0 = int(frac*Nt)
#    x, y = X[i0:], Y[i0:]

    imin, imax = 0, 0

    # Countercurrent at outer shell
#    for i in range(i0, Nt-1):
#        if X[i] > max(X[i-1], X[i+1]):
#            imax = i
#            break
#
#    for i in range(imax + 1, Nt-1):
#        if X[i] < min(X[i-1], X[i+1]):
#            imin = i
#            break
#
#    ax.plot(X[imax:imin+1], Y[imax:imin+1], 'oc', linewidth=2)

    # Countercurrent at inner shell
    for i in range(i0, Nt-1):
        if X[i] < min(X[i-1], X[i+1]):
            imin = i
            break

    for i in range(imin + 1, Nt-1):
        if X[i] > max(X[i-1], X[i+1]):
            imax = i
            break

    ax.plot(X[imin:imax+1], Y[imin:imax+1], color='cyan', marker='o', linewidth=2)

#    imax_speed = imin
#    for i in range(imax + 1, Nt-1):
#    for i in range(imax + 1, imin-1):
#        if speed[i] > max(speed[i-1], speed[i+1]):
#            imax_speed = i
#            break

#    ax.plot(X[imax:], Y[imax:], color='black', linewidth=0.5)
    ax.plot(X[i0:], Y[i0:], color='black', linewidth=0.5)

#    ax.plot([X[imax_speed]], [Y[imax_speed]], color='red', marker='o',
#            label='Position of maximum speed')

    ax.plot([X[i0]], [Y[i0]], color='black', marker='o',
            label='Starting point')
    ax.plot([X[i0+5]], [Y[i0+5]], color='gray', marker='o')

#    ax.legend(loc='best', fontsize=12, numpoints=1)

    if equal:
        ax.axis('equal')

    tg = abs(Y[imax]/Y[imin])
    p = (2./np.pi)*np.arctan(tg)
    print('p = %.6f' % p)

#    max_velocity = max(speed[imax:imin+1])
#    print('||Umax|| = ' + str(max_velocity))

def travel_length(Xmax, Ymax, p):
    AR = Ymax/Xmax

    zeta = np.linspace(-.999, .999, 100)

#    tmp = np.sin(p*np.pi/2.)/np.sqrt(1 + zeta) - np.cos(p*np.pi/2.)/np.sqrt(1 - zeta)
    tmp = np.sin(p*np.pi/2.)/np.sqrt(1 + zeta) + np.cos(p*np.pi/2.)/np.sqrt(1 - zeta)

    tmp *= (1/8.)*AR

    integrand = np.sqrt(1 + tmp**2)

    integral = trapz(integrand, zeta, axis=0)

    return Xmax*integral

In [ ]:
%%writefile bvf_targets.py
# -*- coding: utf-8 -*-
"""
Phase 1 of the Lighthill boundary-vorticity-flux (BVF) plan (see bvf.md).

Builds the measured target g(theta, t) = (1/R) d p/d theta at the cylinder
wall, from the same pressure taps used by --PressureOnly training. Offline,
numpy-only: no TensorFlow training or GPU involved.

Pipeline: per-tap temporal harmonic fit (k=0,1,2) -> per-mode azimuthal
Fourier fit across taps -> analytic d/dtheta -> reassembled time-domain
target on a dense wall grid.
"""

import argparse
import os
import sys

import numpy as np

import matplotlib
matplotlib.use('Agg')  # must happen before Load_train_data_desync's unconditional `import matplotlib.pyplot`

_HERE = os.path.dirname(os.path.abspath(__file__))
# Insert src/ first, then _HERE (src/pressure_only) on top of it, so that
# for any module name present in both (e.g. Load_train_data_desync.py, which
# exists in both as different files), the pressure_only copy wins - src/ is
# only there to supply text_flow.py / reactions_process.py, which pressure_only
# doesn't duplicate.
sys.path.insert(0, os.path.join(_HERE, '..'))
sys.path.insert(0, _HERE)
try:
    import Load_train_data_desync as ltd
except ImportError:
    # Local machine missing TensorFlow and/or running a scipy new enough to
    # have dropped scipy.integrate.trapz - fall back to harmless dev stubs
    # (see _local_dev_stubs/) so the real tap-extraction logic still runs.
    sys.path.insert(0, os.path.join(_HERE, '_local_dev_stubs'))
    import Load_train_data_desync as ltd

RE = 100.
R_C = 0.5
OMEGA_0 = 1.036
KMAX = 2  # default only; overridden by --KMAX


def geom_default():
    Lxmin, Lxmax = -4., 8.
    Lymin, Lymax = -4., 4.
    x_c, y_c, r_c = 0., 0., R_C
    return (Lxmin, Lxmax, Lymin, Lymax, x_c, y_c, r_c)


def temporal_harmonic_fit(times, p_t, omega_0, kmax=KMAX):
    """
    Least-squares fit of p(t) ~ a0 + sum_k [ak cos(k w0 t) + bk sin(k w0 t)].
    Returns {k: complex p_hat_k} matching NN_functions.NN_time_p's convention
    (p(t) = Re{sum_k p_hat_k * exp(+i k w0 t)}), plus fit R^2.
    """
    cols = [np.ones_like(times)]
    for k in range(1, kmax + 1):
        cols.append(np.cos(k * omega_0 * times))
        cols.append(np.sin(k * omega_0 * times))
    A = np.stack(cols, axis=1)
    coeffs, _, _, _ = np.linalg.lstsq(A, p_t, rcond=None)
    pred = A @ coeffs
    ss_res = np.sum((p_t - pred) ** 2)
    ss_tot = np.sum((p_t - np.mean(p_t)) ** 2)
    r2 = 1. - ss_res / ss_tot if ss_tot > 0 else 1.0

    p_hat = {0: complex(coeffs[0], 0.)}
    for k in range(1, kmax + 1):
        a_k = coeffs[1 + 2 * (k - 1)]
        b_k = coeffs[2 + 2 * (k - 1)]
        # Re{(a - i b) e^{i k w0 t}} = a cos(k w0 t) + b sin(k w0 t)
        p_hat[k] = complex(a_k, -b_k)
    return p_hat, r2


def azimuthal_fourier_fit(thetas, values, M=8):
    """Least-squares fit of complex `values(thetas)` ~ sum_{m=-M}^{M} c_m e^{i m theta}."""
    ms = np.arange(-M, M + 1)
    A = np.exp(1j * np.outer(thetas, ms))
    c, _, _, _ = np.linalg.lstsq(A, values, rcond=None)
    return ms, c


def eval_azimuthal(ms, c, theta):
    return np.sum(c[None, :] * np.exp(1j * np.outer(theta, ms)), axis=1)


def eval_azimuthal_dtheta(ms, c, theta):
    return np.sum((1j * ms)[None, :] * c[None, :] * np.exp(1j * np.outer(theta, ms)), axis=1)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--DataFile', type=str, default='Data/fixed_cylinder_atRe100')
    parser.add_argument('--NTaps', type=int, default=32)
    parser.add_argument('--Seed', type=int, default=0)
    parser.add_argument('--stdNoise', type=float, default=0.0)
    parser.add_argument('--M', type=int, default=8)
    parser.add_argument('--Ntheta', type=int, default=64)
    parser.add_argument('--Ntime', type=int, default=24)
    parser.add_argument('--Out', type=str, default=None)
    parser.add_argument('--KMAX', type=int, default=KMAX,
                        help='Highest temporal harmonic to fit and reassemble. Must equal '
                             'the trained mode index max (--Nmodes minus 1). Default 2 '
                             'reproduces the original k=0,1,2 targets.')
    parser.add_argument('--NoPlot', action='store_true', default=False)
    args = parser.parse_args()
    KMAX_RUN = args.KMAX
    assert KMAX_RUN >= 1, '--KMAX must be at least 1'

    np.random.seed(args.Seed)
    geom = geom_default()

    print('Reading real dataset and extracting %d cylinder taps...' % args.NTaps)
    times, data_cyl, _ = ltd.read_cut_simulation_data_exp_point_and_cylinder(
        args.DataFile, geom, n_taps=args.NTaps)
    x_cyl, y_cyl, _, _, p_cyl = data_cyl  # each [Nt, NTaps]

    x_tap = x_cyl[0, :]
    y_tap = y_cyl[0, :]
    theta_tap = np.arctan2(y_tap, x_tap)

    if args.stdNoise > 0:
        for j in range(p_cyl.shape[1]):
            p_cyl[:, j] = ltd.addNoise(p_cyl[:, j], args.stdNoise)
        print('Added Gaussian noise, std=%.4f' % args.stdNoise)

    p_hat_taps = {k: np.zeros(args.NTaps, dtype=complex) for k in range(KMAX_RUN + 1)}
    r2_taps = np.zeros(args.NTaps)
    for j in range(args.NTaps):
        p_hat, r2 = temporal_harmonic_fit(times, p_cyl[:, j], OMEGA_0, kmax=KMAX_RUN)
        for k in range(KMAX_RUN + 1):
            p_hat_taps[k][j] = p_hat[k]
        r2_taps[j] = r2

    print('Temporal harmonic fit R^2 per tap: min=%.5f mean=%.5f max=%.5f'
          % (r2_taps.min(), r2_taps.mean(), r2_taps.max()))

    azimuthal_ms = {}
    azimuthal_c = {}
    for k in range(KMAX_RUN + 1):
        ms, c = azimuthal_fourier_fit(theta_tap, p_hat_taps[k], M=args.M)
        azimuthal_ms[k] = ms
        azimuthal_c[k] = c
        dominant = ms[np.argsort(-np.abs(c))[:3]]
        print('Mode k=%d azimuthal fit: dominant |c_m| at m=%s' % (k, dominant.tolist()))

    theta_grid = np.linspace(0., 2 * np.pi, args.Ntheta, endpoint=False)
    T_period = 2 * np.pi / OMEGA_0
    t_grid = 400. + np.linspace(0., T_period, args.Ntime, endpoint=False)

    g_hat_k = {}
    for k in range(KMAX_RUN + 1):
        dphat_dtheta = eval_azimuthal_dtheta(azimuthal_ms[k], azimuthal_c[k], theta_grid)
        g_hat_k[k] = dphat_dtheta / R_C

    G = np.zeros((args.Ntheta, args.Ntime))
    for it, t in enumerate(t_grid):
        val = np.zeros(args.Ntheta, dtype=complex)
        for k in range(KMAX_RUN + 1):
            val = val + g_hat_k[k] * np.exp(1j * k * OMEGA_0 * t)
        G[:, it] = np.real(val)

    x_wall = R_C * np.cos(theta_grid)
    y_wall = R_C * np.sin(theta_grid)

    out_path = args.Out or ('bvf_targets_Ntap%d_seed%d.npz' % (args.NTaps, args.Seed))
    save_kw = dict(
        theta_grid=theta_grid, t_grid=t_grid, x_wall=x_wall, y_wall=y_wall, G=G,
        theta_tap=theta_tap, r2_taps=r2_taps, ms=azimuthal_ms[0],
        kmax=np.int64(KMAX_RUN),
    )
    for k in range(KMAX_RUN + 1):
        save_kw['g_hat_k%d' % k] = g_hat_k[k]
        save_kw['p_hat_tap_k%d' % k] = p_hat_taps[k]
        save_kw['c_k%d' % k] = azimuthal_c[k]
    np.savez(out_path, **save_kw)
    print('Saved targets to', out_path)

    if not args.NoPlot:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, KMAX_RUN + 1, figsize=(15, 4))
        # Match arctan2's (-pi, pi] branch so the fitted curve and the tap
        # scatter points visually line up (the fit itself is periodic and
        # correct on either branch - this is purely for a legible plot).
        theta_dense = np.linspace(-np.pi, np.pi, 400)
        for k in range(KMAX_RUN + 1):
            phat_dense = eval_azimuthal(azimuthal_ms[k], azimuthal_c[k], theta_dense)
            axes[k].plot(theta_dense, phat_dense.real, label='Re fit')
            axes[k].plot(theta_dense, phat_dense.imag, label='Im fit')
            axes[k].scatter(theta_tap, p_hat_taps[k].real, marker='o', s=15, label='Re taps')
            axes[k].scatter(theta_tap, p_hat_taps[k].imag, marker='x', s=15, label='Im taps')
            axes[k].set_title('p_hat mode k=%d' % k)
            axes[k].set_xlabel('theta')
            axes[k].legend(fontsize=7)
        plt.tight_layout()
        plot_path = 'bvf_targets_diagnostic_Ntap%d_seed%d.png' % (args.NTaps, args.Seed)
        plt.savefig(plot_path)
        print('Saved diagnostic plot to', plot_path)


if __name__ == '__main__':
    main()


## 5. CFD dataset

In [ ]:
DRIVE_DATA='/content/drive/MyDrive/ModalPINN_data/fixed_cylinder_atRe100'
LOCAL_DATA='/content/Data/fixed_cylinder_atRe100'
os.makedirs('/content/Data',exist_ok=True)
def drive_copy(src,dst,tries=5,chunk=8*1024*1024):
    """Copy from the Drive FUSE mount, surviving 'Transport endpoint is not
    connected' (Errno 107). The mount drops on its own; remount and retry
    rather than failing the notebook. Verifies size, so a truncated partial
    copy is never accepted."""
    import time
    for k in range(tries):
        try:
            if not os.path.exists(src):
                raise OSError('source not visible on the mount: %s'%src)
            n=os.path.getsize(src)
            if os.path.exists(dst) and os.path.getsize(dst)==n:
                print('  already present, %.1f MB'%(n/1024**2)); return dst
            if os.path.exists(dst): os.remove(dst)
            with open(src,'rb') as fs, open(dst,'wb') as fd:
                while True:
                    b=fs.read(chunk)
                    if not b: break
                    fd.write(b)
            got=os.path.getsize(dst)
            assert got==n,'size mismatch: %d vs %d'%(got,n)
            print('  copied %.1f MB on attempt %d'%(got/1024**2,k+1)); return dst
        except (OSError,AssertionError) as e:
            print('  attempt %d failed: %s'%(k+1,e))
            if os.path.exists(dst):
                try: os.remove(dst)
                except OSError: pass
            if k==tries-1: raise
            try:
                from google.colab import drive
                drive.flush_and_unmount(); drive.mount('/content/drive',force_remount=True)
            except Exception as me: print('  remount failed: %s'%me)
            time.sleep(10)

drive_copy(DRIVE_DATA,LOCAL_DATA)
print('CFD data ready: %.1f MB'%(os.path.getsize(LOCAL_DATA)/1024**2))

## 6. Verify the sources

In [ ]:
compile_files=['NN_functions.py','ModalPINN_VortexShedding.py',
    'Load_train_data_desync.py','evaluate_regions.py','evaluate_v1_smoke.py',
    'evaluate_physics_uniform.py','street_prior.py','bvf_targets.py',
    'text_flow.py','reactions_process.py']
rr=subprocess.run([LEGACY_PY,'-m','py_compile']+compile_files,
                  env=env,capture_output=True,text=True)
print(rr.stderr[-1500:])
assert rr.returncode==0,'compilation failed'

main_text=open('ModalPINN_VortexShedding.py').read()
nnf_text=open('NN_functions.py').read()
bvf_text=open('bvf_targets.py').read()

# the mode count must be fully parameterised, with no literal-3 assumption
assert 'layers = [2,args.WidthLayer*Nmodes,args.WidthLayer*Nmodes,Nmodes]' in main_text
assert not re.search(r'Nmodes\s*==\s*3',main_text)
assert 'def street_modes_k(x, y, sp, k)' in nnf_text

# the ORIGINAL Adam schedule must still be present and budgeted by Tmax
assert 'AdamTmax = Tmax-(t2-t0)' in main_text,'the original Adam schedule is missing'
assert 'Start Adam training' in main_text

# bvf_targets.py must be the KMAX-generalised copy
assert "'--KMAX'" in bvf_text,'bvf_targets.py is the un-patched original'

for flag in ['--Tmax','--Nmodes','--WidthLayer','--BVF','--BVFTargets',
             '--V1RadialTrust','--StreetPrior','--FreestreamBC','--FluctuationInletBC']:
    assert "'"+flag+"'" in main_text,flag
print('Sources verified: mode count parameterised, Adam schedule intact, BVF patched.')

## 8. Train — the original 9-hour schedule

In [ ]:
ROOT_FILES=['ModalPINN_VortexShedding.py','NN_functions.py','Load_train_data_desync.py',
            'evaluate_regions.py','evaluate_v1_smoke.py','evaluate_physics_uniform.py',
            'street_prior.py','bvf_targets.py','text_flow.py','reactions_process.py']

WD='/content/run_'+ARM_TAG
if os.path.exists(WD): shutil.rmtree(WD)
os.makedirs(os.path.join(WD,'Data'),exist_ok=True)
os.symlink(LOCAL_DATA,os.path.join(WD,'Data','fixed_cylinder_atRe100'))
for fn in ROOT_FILES: shutil.copyfile('/content/'+fn,os.path.join(WD,fn))
NMODES=4
WIDTH=25

cmd=[LEGACY_PY,'ModalPINN_VortexShedding.py',
     '--Tmax','9',
     '--Seed','0',
     '--Nmes','5000','--Nint','50000',
     '--multigrid','--Ngrid','5','--NgridTurn','200',
     '--WidthLayer',str(WIDTH),'--Nmodes',str(NMODES),
     '--FreestreamBC',
     # periodic weight snapshots: if the wall-clock guard or a Colab drop kills
     # this run mid-L-BFGS, the checkpoints survive. Arm 14's first attempt was
     # killed at 40,800 evaluations and lost everything for want of these.
     '--LBFGSMaxit','40000','--LBFGSMaxfun','40000',
     '--LBFGSCheckpointIters','5000,10000,20000,30000,40000',
     '--SkipAdam',
     '--SkipDiagnostics','--ExitAfterSafetySave']
assert '--FluctuationInletBC' not in cmd, 'this arm must run with the fluctuation inlet BC OFF'
assert '--FreestreamBC' in cmd, 'the freestream inlet prior is kept on every arm'
assert '--LBFGSMaxit' in cmd, 'this retry must cap L-BFGS so the save path runs'
assert '--SkipAdam' in cmd, 'this retry stops at the L-BFGS cap; Adam is skipped by design'
print('COMMAND:')
print('  '+' '.join(cmd))
print()

def stream(cmd,cwd,hard_limit=13.0*3600,stall_limit=2400):
    p=subprocess.Popen(cmd,cwd=cwd,env=env,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    q=queue.Queue()
    def rd():
        for line in p.stdout: q.put(line)
        q.put(None)
    threading.Thread(target=rd,daemon=True).start()
    out=[]; t0=time.time(); last=t0; shown=0
    while True:
        try:
            it=q.get(timeout=5)
            if it is None: break
            out.append(it); last=time.time()
            if it.startswith('Loss: '):
                shown+=1
                if shown%100==0:
                    print('  [%6d evals, %6.0f s] %s'%(shown,time.time()-t0,it.strip()))
            else:
                print(it,end='')
        except queue.Empty:
            pass
        if time.time()-t0>hard_limit:
            p.kill(); raise RuntimeError('hard guard: exceeded %.1f h'%(hard_limit/3600))
        if time.time()-last>stall_limit:
            p.kill(); raise RuntimeError('no output for %d s'%stall_limit)
    return p.wait(),''.join(out),time.time()-t0

print('START',datetime.datetime.now())
rc,log,WALL=stream(cmd,WD)
print()
print('return code %d after %.2f h'%(rc,WALL/3600))
assert rc==0, ('training exited %d - do NOT treat this arm as a '
                'result; inspect train_log.txt'%rc)

# the log must confirm the configuration actually used
assert 'Fluctuation Inlet BC : True' not in log, 'log shows FIBC active - wrong arm'
assert 'Freestream BC : True' in log, 'log shows the freestream prior inactive - wrong arm'
print('log confirms: fluctuation inlet BC off, freestream prior on')

runs=sorted(glob.glob(os.path.join(WD,'OutputPythonScript','ModalPINN_*')),
            key=os.path.getmtime)
assert runs,'no output folder produced'
RUN_DIR=runs[-1]
picks=glob.glob(os.path.join(RUN_DIR,'DNN*_tanh.pickle'))
assert picks,'no weights saved'
print('run dir:',RUN_DIR)
print('weights:',os.path.basename(picks[0]))

# the checkpoint filename encodes the layer shape: independent proof of the mode count
expect='DNN2_%d_%d_%d_tanh.pickle'%(WIDTH*NMODES,WIDTH*NMODES,NMODES)
assert os.path.basename(picks[0])==expect,'expected %s, got %s'%(expect,os.path.basename(picks[0]))
print('mode count confirmed by checkpoint shape: k = 0,1,2,3')

# SAVE BEFORE EVALUATING: the 9 hours are already spent
dtr=os.path.join(DEST,'training_run')
if os.path.exists(dtr): shutil.rmtree(dtr)
shutil.copytree(RUN_DIR,dtr)
with open(os.path.join(DEST,'train_log.txt'),'w') as f: f.write(log)
json.dump(dict(arm=ARM_TAG,nmodes=NMODES,width=WIDTH,rc=rc,wall_s=round(WALL,1),
               weights=os.path.basename(picks[0]),command=cmd,
               lbfgs_evals=log.count('Loss: '),
               adam_ran=('Adam training ended' in log),
               cold_start=('--RestoreModel' not in cmd)),
          open(os.path.join(DEST,'run_record.json'),'w'),indent=2)
print('saved ->',dtr)

## 9. Evaluate

In [ ]:
pf=[]
NM=str(NMODES)

v1out='/content/%s_v1.json'%ARM_TAG
rr=subprocess.run([LEGACY_PY,'evaluate_v1_smoke.py','--RunDir',RUN_DIR,
    '--DataFile','Data/fixed_cylinder_atRe100','--WidthLayer','25','--Nmodes',NM,
    '--Out',v1out]+pf,cwd=WD,env=env,capture_output=True,text=True)
print(rr.stdout[-1600:])
print(rr.stderr[-600:])
V1=json.load(open(v1out)) if rr.returncode==0 and os.path.exists(v1out) else None

# evaluate_physics_uniform.py rebuilds the velocity field and applies the v1 trust
# wrap unconditionally. For a prior-OFF arm that would rebuild a function the network
# never trained, so the residual would be meaningless. It is skipped there.
PHYS=None
if pf:
    physout='/content/%s_physics.json'%ARM_TAG
    rr=subprocess.run([LEGACY_PY,'evaluate_physics_uniform.py','--RunDir',RUN_DIR,
        '--WidthLayer','25','--Nmodes',NM,'--N','20000','--Seed','99173',
        '--StreetPrior','street_prior_Ntap32.npz','--V1TrustRho','0.60',
        '--Out',physout],cwd=WD,env=env,capture_output=True,text=True)
    print(rr.stdout[-900:])
    if rr.returncode==0 and os.path.exists(physout): PHYS=json.load(open(physout))
else:
    print('evaluate_physics_uniform.py SKIPPED for this arm: it applies the v1 trust')
    print('wrap with no opt-out, so it cannot faithfully evaluate a prior-off')
    print('checkpoint. The regional metrics below are the comparable numbers here.')

regout='/content/%s_regions.json'%ARM_TAG
rr=subprocess.run([LEGACY_PY,'evaluate_regions.py','--RunDir',RUN_DIR,
    '--DataFile',LOCAL_DATA,'--WidthLayer','25','--Nmodes',NM,
    '--Out',regout]+pf,cwd=WD,env=env,capture_output=True,text=True)
print(rr.stdout[-1600:])
print(rr.stderr[-600:])
REG=json.load(open(regout)) if rr.returncode==0 and os.path.exists(regout) else None

for src,name in ((v1out,'v1.json'),(regout,'regions.json')):
    if os.path.exists(src): shutil.copyfile(src,os.path.join(DEST,name))
if PHYS: shutil.copyfile(physout,os.path.join(DEST,'physics.json'))

print()
print('='*66)
print('ARM %s   k = 0,1,2,3'%ARM_TAG)
print('='*66)
if V1:
    fc=V1['regions'].get('far-core',{})
    print('far-core k=1 amplitude ratio : %s'%fc.get('amp_ratio'))
    print('far-core k=1 relative L2     : %s'%fc.get('rel_L2'))
    print('far-core k=1 correlation     : %s'%fc.get('corr'))
if PHYS:
    print('uniform NS residual (whole)  : %s'
          %PHYS['regions']['whole_uniform']['total_ns_mse'])
print()
print('K3 SERIES references, same schedule, cold, Nmodes 4:')
print('  Arm 1 baseline physics-only  amp 0.01895  corr 0.16036')
print('  Arm 2 wall vorticity flux    amp 0.08945  corr 0.10809')
print('  Arm 3 Karman prior           amp 0.80878  corr 0.97159')
print('  analytical prior alone, no network       : 0.8082')

# physics is SKIPPED BY DESIGN on prior-off arms (pf empty): evaluate_physics_uniform.py
# applies the v1 trust wrap with no opt-out, so it cannot faithfully evaluate them.
# Only count it as a failure when the arm is prior-active and it still produced nothing.
_expected=[('v1',V1),('regions',REG)]+([('physics',PHYS)] if pf else [])
EVAL_FAILURES=[k for k,v in _expected if v is None]
if not pf:
    print('physics evaluator skipped by design (prior-off arm) - not a failure')
if EVAL_FAILURES:
    print('EVALUATION FAILED for: '+', '.join(EVAL_FAILURES))
    print('the checkpoint is saved and usable, but arm_summary.json is INCOMPLETE.')

json.dump(dict(arm=ARM_TAG,nmodes=NMODES,v1=V1,physics=PHYS,regions=REG),
          open(os.path.join(DEST,'arm_summary.json'),'w'),indent=2)
print()
print('saved arm_summary.json')
assert not EVAL_FAILURES, ('evaluation failed for %s - the summary is incomplete; '
                           'do NOT report this arm until re-evaluated'%EVAL_FAILURES)

## What to send back

The whole `dense_reference` folder from Drive.